# BodyM Measurement Model — Training, Evaluation & Export

**Notebook version** 1.0 · **Brief version** 2026-10-07 · **Pipeline step** 6 of 10

---

## 0.1 Purpose

This notebook trains, evaluates, documents and exports the **body-measurement estimation model**
for step 6 of a 10-step made-to-measure (MTM) pattern pipeline:

| # | Step | # | Step |
|---|------|---|------|
| 1 | Garment type | 6 | **MEASUREMENT ESTIMATION ← this notebook** |
| 2 | Capture (front A-pose + side) | 7 | Schema validation + YAML |
| 3 | Person segmentation | 8 | Pattern drafting |
| 4 | Scale / perspective calibration | 9 | Production details & export |
| 5 | Landmarks | 10 | Virtual drape + physical toile vs tape-measured ground truth |

**Input contract.** front silhouette (binary mask), side silhouette (binary mask), `height_cm`,
optionally `weight_kg`.
**Output contract.** 14 body measurements in cm, each with 80% and 90% prediction intervals.

Step 7 consumes the intervals for plausibility gating; step 4 consumes them in its error budget.

## 0.2 Why this notebook is also documentation

Every artefact the pipeline needs later (schema map, model card, decision log, references,
pinned requirements, trained weights) is written **by this notebook**, so the provenance of the
exported model is never separated from the code that produced it. The notebook must pass
*Restart & Run All* with no hidden state.

## 0.3 Data

**BodyM** (Ruiz et al., 2022) — binary silhouettes + height + weight + sex + 14 measurements.
**No RGB images.** Therefore **no segmentation model is trained here**: step 3 of the pipeline owns
that. This notebook only consumes masks.

Licence **CC BY-NC 4.0** — **non-commercial use only**. See §10.1 for the registry inconsistency.

## 0.4 How to reproduce

1. Upload BodyM as a Kaggle Dataset and attach it to the notebook (GPU accelerator recommended).
2. Kaggle menu → **Save Version** → **Save & Run All**.
3. Collected artefacts land in `/kaggle/working/export/` and are zipped to
   `/kaggle/working/export.zip`.

`config.yaml` is written on first run if absent, then **read back** as the single source of
hyperparameters. Change a value there, re-run, and every phase follows.

## 0.5 Hardware & expected runtime

| Item | Value |
|------|-------|
| GPU | 1× T4 / P100 (16 GB). CPU-only works but ≈ 20× slower |
| Disk | < 1 GB dataset, < 500 MB artefacts |
| RAM | ≥ 8 GB (dataset is streamed, not cached) |
| Runtime | ≈ 4–6 h total on T4: EDA 2 min · B0/B1 3 min · B2 ≈ 25 min · sensitivity ≈ 25 min · production 2× ≈ 40 min · conformal 1 min · final eval 5 min · export 2 min |

Knobs that trade accuracy for time live in `config.yaml` (`epochs`, `img_size`, `backbone`,
`n_sensitivity_subjects`).

## 0.6 Table of contents

- **§1 Phase 0** — setup, environment, seeds, data integrity
- **§2 Phase 1** — exploratory data analysis
- **§3 Phase 2** — subject-level splits
- **§4 Phase 3** — baselines B0 (non-visual), B1 (geometric), B2 (BMnet-style CNN)
- **§5 Phase 4** — boundary-sensitivity experiment (key deliverable)
- **§6 Phase 5** — production model, variants V-HW and V-H
- **§7 Phase 6** — split conformal prediction
- **§8 Phase 7** — final evaluation on Test-A / Test-B (run **once**)
- **§9 Phase 8** — export bundle
- **§10** — licence, open decisions for the user
- **§11** — reproducibility summary

### §1 Phase 0 — Setup, environment, seeds and data integrity

**WHAT** — Pin the working directory, install only what is missing, print the full environment
(Python, package versions, GPU, internet reachability), set global seeds, and load
`config.yaml` (writing it first if it does not exist).

**WHY**
- `[Ref]` Ruiz et al. (2022) define BodyM as silhouette-only; the absence of RGB images is the
  reason this notebook never trains a segmenter (brief §1).
- `[Ref]` Pimentel et al. (2021): notebooks must be reproducible from a clean kernel. Non-fixed
  seeds and hidden state are the two most common sources of irreproducibility.
- `[Assumption]` `config.yaml` may already exist from a previous session. *Tested by* the cell:
  it logs whether the file was created or loaded, and re-running with an edited file must change
  behaviour.

**OUTPUT** — `cfg` (dict), `ENV` (dict), `PATHS` (dict), `SEED`.

**CHECK** — `assert` the package set imports, that a config round-trips, and that two successive
draws from `np.random.default_rng(SEED)` agree.

**WHAT** — Print the environment record: Python and platform, every package version actually
imported, the CUDA device, and whether the network is reachable.

**WHY**
- `[Ref]` Pimentel et al. (2021): a notebook's result is only reproducible if the environment is
  recorded. The block doubles as the source for the `requirements.txt` pinned at §9.4.
- `[Assumption]` Kaggle's GPU image ships a recent CUDA-enabled torch. *Tested by* `gpu_report()`,
  which reports `available`, name, compute capability and memory rather than assuming one.
- `[Assumption]` Outbound HTTPS is available for ImageNet weights. *Tested by* `internet_report()`,
  which opens a real socket to two hosts with a timeout. A `False` here changes the backbone
  branch in §4.3, so it is checked before anything depends on it.

**OUTPUT** — `ENV` (dict), consumed by the §11 summary and by the weight-loading branch.

**CHECK** — the cell asserts nothing; it is a pure observation step, and every value it reports is
printed for the log.

In [ ]:
import os, sys, json, time, math, random, platform, textwrap, hashlib, zipfile, shutil, warnings
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

t_notebook_start = time.time()
PHASE_TIMES: Dict[str, float] = {}


def mark_phase(name: str) -> float:
    """Record elapsed wall-clock seconds for a pipeline phase.

    Parameters
    ----------
    name : str
        Phase label used as key in :data:`PHASE_TIMES`.

    Returns
    -------
    float
        Seconds elapsed since the previous call.
    """
    now = time.time()
    last = PHASE_TIMES.get("_last", t_notebook_start)
    PHASE_TIMES[name] = round(now - last, 1)
    PHASE_TIMES["_last"] = now
    return PHASE_TIMES[name]


# --------------------------------------------------------------------------------------
# Paths. Kaggle mounts read-only inputs at /kaggle/input and a writable dir at /kaggle/working.
# Locally we fall back to the repo layout so the notebook is runnable during development.
# --------------------------------------------------------------------------------------
def resolve_paths() -> Dict[str, Path]:
    """Return {working, src, export, artifacts} for Kaggle or a local checkout.

    Returns
    -------
    Dict[str, Path]
        Absolute paths. ``working`` is created if missing.
    """
    if Path("/kaggle/working").is_dir():
        working = Path("/kaggle/working")
    else:
        working = Path.cwd()
    src = working / "src"
    export = working / "export"
    artifacts = working / "artifacts"
    for p in (src, export, artifacts):
        p.mkdir(parents=True, exist_ok=True)
    return {"working": working, "src": src, "export": export, "artifacts": artifacts}


PATHS = resolve_paths()
print("PATHS:", json.dumps({k: str(v) for k, v in PATHS.items()}, indent=2))

sys.path.insert(0, str(PATHS["src"]))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

# --- install only what is missing; never assume a clean image ------------------------------
import importlib.util, subprocess

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "yaml": "pyyaml",
    "torch": "torch",
    "torchvision": "torchvision",
    "tqdm": "tqdm",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("all required packages present")

# Optional: ONNX export verification. Not fatal if absent.
for opt in ("onnx", "onnxruntime"):
    if importlib.util.find_spec(opt) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", opt], check=False)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import scipy
import yaml
import torch
import torchvision
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from scipy import ndimage as ndi
from tqdm.auto import tqdm

matplotlib.use("Agg")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "figure.facecolor": "white", "font.size": 9})


def gpu_report() -> Dict[str, Any]:
    """Describe the visible CUDA device.

    Returns
    -------
    dict
        ``available`` (bool), ``name`` (str), ``count`` (int), ``capability`` (str),
        ``total_gb`` (float).
    """
    info = {"available": torch.cuda.is_available(), "name": "cpu", "count": 0,
            "capability": "n/a", "total_gb": 0.0}
    if info["available"]:
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            if i == 0:
                info.update(name=p.name, count=torch.cuda.device_count(),
                            capability=f"{p.major}.{p.minor}", total_gb=round(p.total_memory / 2**30, 1))
    return info


def internet_report(timeout: float = 4.0) -> bool:
    """Return True if an outbound HTTPS connection to pypi succeeds within ``timeout``.

    Parameters
    ----------
    timeout : float
        Socket timeout in seconds.

    Returns
    -------
    bool
        True when the network is reachable.
    """
    import socket
    for host, port in (("pypi.org", 443), ("download.pytorch.org", 443)):
        try:
            with socket.create_connection((host, port), timeout=timeout):
                return True
        except OSError:
            continue
    return False


ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "executable": sys.executable,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pillow": Image.__version__ if hasattr(Image, "__version__") else __import__("PIL").__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": gpu_report(),
    "internet": internet_report(),
    "kaggle_env": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
}
print(json.dumps(ENV, indent=2))

**Interpretation.** The environment block is the first reproducibility anchor. Two entries drive
downstream decisions:

- `gpu` — if `available` is `False`, the CNN phases still run but the expected runtime in §0.5
  no longer holds and the sensitivity experiment in §5 becomes the bottleneck.
- `internet` — if `False`, ImageNet weights cannot be downloaded at run time. §4.3 and §6 then
  look for an offline weight file and otherwise fall back to random initialisation with an
  explicit log line. This is a `[Assumption]` about the deployment image, tested by the weight
  cell printing which branch it took.

The backbone resolution also depends on this: `MNASNet` with `IMAGENET1K_V1` weights is only
reachable when `internet` is `True` or an offline `.pth` has been attached.

**WHAT** — Set the master seed for Python, NumPy, PyTorch and every CUDA device, and request
deterministic cuDNN kernels. Start the per-phase wall clock.

**WHY**
- `[Ref]` Angelopoulos & Bates (2021) require calibration and test points to be exchangeable,
  which presumes a deterministic pipeline; non-reproducible training would make the §7 coverage
  numbers meaningless.
- `[Exp]` Deterministic cuDNN costs ~10–20 % throughput. That is the right trade here: a
  reproducible 4-hour run beats a marginally faster irreproducible one. A platform that lacks a
  deterministic kernel logs a warning instead of aborting the run.

**OUTPUT** — `SEED`, `set_global_seeds`, `PHASE_TIMES`.

**CHECK** — asserts two `default_rng(SEED)` draws agree, then prints one draw from each RNG so
the log records what the seed actually produced.

In [ ]:
# --- seeds ------------------------------------------------------------------------------
SEED = 42


def set_global_seeds(seed: int = SEED) -> None:
    """Seed every RNG that can influence this notebook.

    Parameters
    ----------
    seed : int
        Master seed. Also seeds Python, NumPy, and PyTorch (CPU and all CUDA devices).

    Notes
    -----
    Deterministic cuDNN kernels are requested. They cost ~10-20% throughput; if a future kernel
    lacks a deterministic implementation, the warning is downgraded rather than raised so the
    run does not abort mid-training.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception as exc:  # pragma: no cover - platform dependent
        print("cudnn determinism unavailable:", exc)


set_global_seeds(SEED)
print("SEED =", SEED, "| random:", random.random(), "| np:", np.random.rand(1),
      "| torch:", torch.rand(1).item())

assert np.random.default_rng(SEED).random(3).tolist() == np.random.default_rng(SEED).random(3).tolist(), \
    "default_rng(SEED) is not reproducible"
print("mark_phase('setup') =", mark_phase("setup"), "s")

**WHAT** — Write `config.yaml` if it is missing, then read it back and expose it as `cfg`.

**WHY**
- `[Ref]` Pimentel et al. (2021) argue that magic numbers scattered across cells are a primary
  cause of irreproducible notebooks. One file, read at the top, removes that class of defect and
  doubles as the exported provenance record.
- `[Assumption]` No `config.yaml` exists on a fresh Kaggle session. *Tested by* the
  `cfg_created` flag, which must be `True` on a clean run and `False` on the second run.

**OUTPUT** — `cfg` (dict), `cfg_created` (bool), `CFG_PATH`.

**CHECK** — `assert` every key required downstream is present; a missing key fails here rather
than halfway through a 4-hour run.

In [ ]:
DEFAULT_CFG: Dict[str, Any] = {
    "seed": 42,
    "data_root": None,          # None -> auto-discovered under the Kaggle input dir
    "img_height": 480,           # per-view mask height fed to the CNN (see the §4.3 note)
    "img_width": 640,            # per-view mask width; front|side are concatenated horizontally
    "mask_threshold": 127,       # PNG is 0/255; anything > this is foreground
    # ---- baselines ---------------------------------------------------------------------
    "ridge_alpha": 10.0,
    "n_estimators": 300,
    # ---- CNN ---------------------------------------------------------------------------
    "backbone": "mnasnet1_0_3_0",
    "pretrained": True,
    "offline_weights": None,     # path to a local ImageNet .pth when internet is unavailable
    "epochs_b2": 6,
    "epochs_prod": 8,
    "batch_size": 8,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 5.0,
    "dropout": 0.2,
    "hidden": 512,
    "num_workers": 4,
    "amp": True,
    # ---- boundary sensitivity ----------------------------------------------------------
    "erosion_px": 6,             # sweep -6..+6
    "jitter_sigma_px": [0, 2, 4, 6],
    "downsample_factors": [2, 4, 8],
    "n_sensitivity_subjects": 150,
    "tolerance_mm": [9, 15, 25],
    "key_measurements": ["chest", "waist", "hip"],
    # ---- conformal ---------------------------------------------------------------------
    "coverages": [0.80, 0.90],
    # ---- export -----------------------------------------------------------------------
    "n_onnx_check": 20,
    "onnx_atol": 1e-4,
    "bmi_bands": [0, 18.5, 25, 30, 40, 100],
    "bmi_labels": ["<18.5", "18.5-25", "25-30", "30-40", ">40"],
    "min_stratum_n": 30,
    "include_height_as_target": True,
    "zip_export": True,
}

REQUIRED_KEYS = set(DEFAULT_CFG)

CFG_PATH = PATHS["working"] / "config.yaml"
cfg_created = not CFG_PATH.exists()
if cfg_created:
    CFG_PATH.write_text(yaml.safe_dump(DEFAULT_CFG, sort_keys=True), encoding="utf-8")

cfg: Dict[str, Any] = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
cfg = {**DEFAULT_CFG, **(cfg or {})}

missing_keys = REQUIRED_KEYS - set(cfg)
assert not missing_keys, f"config.yaml missing keys: {sorted(missing_keys)}"

print(f"config.yaml {'CREATED' if cfg_created else 'loaded'} at {CFG_PATH}")
print(yaml.safe_dump(cfg, sort_keys=True))
print("mark_phase('setup') =", mark_phase("setup"), "s")

**WHAT** — Locate the BodyM directory by *searching* rather than hard-coding a path, then load
the three per-split CSV tables and print the directory tree and headers.

**WHY**
- Brief §1: *"Do not assume file or folder names. Inspect the attached data and write loaders
  from what you find."* Kaggle mounts datasets under `/kaggle/input/<slug>/` where `<slug>` is
  user-chosen and unknown at authoring time.
- `[Ref]` Ruiz et al. (2022): BodyM ships as three splits (`train`, `testA`, `testB`), each with
  a measurements table, a height/weight/sex table, a photo→subject map and paired front/side
  mask folders. The search below verifies that shape structurally instead of trusting it.
- `[Assumption]` The attachment contains exactly one BodyM root. *Tested by* the
  `candidates` list printed above the assertion — if two roots exist the run fails loudly
  rather than silently picking one.

**OUTPUT** — `DATA_ROOT` (Path), `SPLITS`, `RAW` (dict of DataFrames), tree listing.

**CHECK** — `assert` one and only one candidate root, and that each split exposes
`measurements.csv`, `hwg_metadata.csv`, `subject_to_photo_map.csv`, `mask/`, `mask_left/`.

In [ ]:
%%writefile src/bodym_data.py
"""Discovery, integrity checks and tabular assembly for the BodyM dataset.

References
----------
Ruiz, N., Bellver, M., Bolkart, T., Arora, A., Lin, M. C., Romero, J., & Bala, R. (2022).
Human body measurement estimation with adversarial augmentation (arXiv:2210.05667).
https://arxiv.org/abs/2210.05667
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

SPLITS: Tuple[str, ...] = ("train", "testA", "testB")

#: Subject counts and silhouette counts published with BodyM (Ruiz et al., 2022).
EXPECTED: Dict[str, Tuple[int, int]] = {"train": (2018, 6134), "testA": (87, 1684),
                                        "testB": (400, 1160)}

#: The 14 measurement columns, in the order they appear in ``measurements.csv``.
MEASUREMENT_COLUMNS: Tuple[str, ...] = (
    "ankle", "arm-length", "bicep", "calf", "chest", "forearm", "height", "hip",
    "leg-length", "shoulder-breadth", "shoulder-to-crotch", "thigh", "waist", "wrist",
)

#: Human-readable definition. These are SMPL-registered mesh vertex-path lengths, NOT tape
#: measurements, and not ISO 20685 compliant. See DECISIONS.md.
DEFINITIONS: Dict[str, str] = {
    "ankle": "minimum ankle perimeter (mesh vertex path)",
    "arm-length": "shoulder joint to wrist vertex path",
    "bicep": "maximum upper-arm (biceps) perimeter",
    "calf": "maximum calf perimeter",
    "chest": "chest perimeter at the SMPL chest landmark",
    "forearm": "maximum forearm perimeter",
    "height": "vertex-path stature of the fitted mesh",
    "hip": "maximum hip perimeter",
    "leg-length": "crotch to ankle vertex path",
    "shoulder-breadth": "biacromial breadth across the front silhouette",
    "shoulder-to-crotch": "acromion to crotch vertex path",
    "thigh": "maximum thigh perimeter",
    "waist": "waist perimeter at the SMPL waist landmark",
    "wrist": "minimum wrist perimeter",
}


def find_bodym_root(search_roots: Sequence[Path | str]) -> Path:
    """Locate the ``bodym/`` root that contains the three splits.

    The search is structural: a candidate must contain ``train``, ``testA`` and ``testB``,
    each with ``measurements.csv`` plus ``mask/`` and ``mask_left/`` directories.

    Parameters
    ----------
    search_roots : sequence of path-like
        Directories to scan recursively, typically ``["/kaggle/input", "."]``.

    Returns
    -------
    Path
        Path to the directory that holds the three split folders.

    Raises
    ------
    FileNotFoundError
        If no structural candidate is found, or if more than one is found (ambiguous input).
    """
    candidates: List[Path] = []
    for base in (Path(r) for r in search_roots):
        if not base.exists():
            continue
        for csv_path in sorted(base.rglob("measurements.csv")):
            split_dir = csv_path.parent
            if split_dir.name not in SPLITS:
                continue
            root = split_dir.parent
            if not all((root / s / "mask").is_dir() and (root / s / "mask_left").is_dir()
                       for s in SPLITS):
                continue
            if root not in candidates:
                candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "No BodyM root found. Expected a directory containing train/testA/testB, each with "
            "measurements.csv, hwg_metadata.csv, subject_to_photo_map.csv, mask/, mask_left/. "
            f"Searched: {[str(r) for r in search_roots]}")
    if len(candidates) > 1:
        raise FileNotFoundError(f"Ambiguous BodyM roots: {[str(c) for c in candidates]}")
    return candidates[0]


def load_split(root: Path | str, split: str) -> Dict[str, pd.DataFrame]:
    """Load the three CSV tables of one split.

    Parameters
    ----------
    root : path-like
        BodyM root (the directory containing ``train``/``testA``/``testB``).
    split : {'train', 'testA', 'testB'}
        Split name.

    Returns
    -------
    dict
        Keys ``measurements``, ``hwg``, ``photos``.

    Raises
    ------
    ValueError
        If ``split`` is not one of the three known splits.
    """
    if split not in SPLITS:
        raise ValueError(f"unknown split {split!r}; expected one of {SPLITS}")
    d = Path(root) / split
    return {
        "measurements": pd.read_csv(d / "measurements.csv"),
        "hwg": pd.read_csv(d / "hwg_metadata.csv"),
        "photos": pd.read_csv(d / "subject_to_photo_map.csv"),
    }


def bmi_band(bmi_kg_m2: pd.Series | np.ndarray,
             edges: Sequence[float]) -> pd.Series:
    """Bin BMI into the five clinical bands used throughout this notebook.

    Parameters
    ----------
    bmi_kg_m2 : array-like of float
        BMI in kg/m^2.
    edges : sequence of float
        Strictly increasing band edges, e.g. ``[0, 18.5, 25, 30, 40, 100]``.

    Returns
    -------
    pandas.Series
        Band index per row, as int.
    """
    return pd.cut(pd.Series(np.asarray(bmi_kg_m2, dtype=float)), bins=list(edges),
                  labels=False, include_lowest=True).astype("Int64")


def verify_integrity(root: Path | str, verbose: bool = True) -> pd.DataFrame:
    """Validate BodyM against the published counts and cross-table consistency.

    Checks performed
    ----------------
    1. Subject and silhouette counts per split equal :data:`EXPECTED` (Ruiz et al., 2022).
    2. The three tables of a split describe the same subject set.
    3. No missing values in measurements or height/weight/sex.
    4. Every ``photo_id`` resolves to both a front and a side mask file.
    5. Warning-level: identical photo ids or subject ids shared across splits.

    Parameters
    ----------
    root : path-like
        BodyM root.
    verbose : bool, default True
        Print a per-check report.

    Returns
    -------
    pandas.DataFrame
        One row per check: ``check``, ``status`` in {ok, warn, fail}, ``detail``.

    Raises
    ------
    AssertionError
        If any check has status ``fail``.
    """
    root = Path(root)
    rows: List[dict] = []

    def add(check: str, status: str, detail: str) -> None:
        rows.append({"check": check, "status": status, "detail": detail})
        if verbose:
            print(f"[{status.upper():4s}] {check}: {detail}")

    tables = {s: load_split(root, s) for s in SPLITS}

    for split, tab in tables.items():
        exp_subj, exp_photo = EXPECTED[split]
        n_subj = tab["measurements"]["subject_id"].nunique()
        n_photo = len(tab["photos"])
        if (n_subj, n_photo) == (exp_subj, exp_photo):
            add(f"counts/{split}", "ok", f"{n_subj} subjects / {n_photo} silhouettes "
                                        f"(expected {exp_subj}/{exp_photo})")
        else:
            add(f"counts/{split}", "fail", f"{n_subj} subjects / {n_photo} silhouettes "
                                          f"(expected {exp_subj}/{exp_photo})")

        sets = {k: set(v["subject_id"]) for k, v in tab.items()}
        if sets["measurements"] == sets["hwg"] == sets["photos"]:
            add(f"subject_sets/{split}", "ok", f"{n_subj} ids identical across the 3 tables")
        else:
            add(f"subject_sets/{split}", "fail",
                f"mismatch: meas-hwg {len(sets['measurements'] - sets['hwg'])}, "
                f"meas-photos {len(sets['measurements'] - sets['photos'])}")

        nan_m = int(tab["measurements"].isna().sum().sum())
        nan_h = int(tab["hwg"].isna().sum().sum())
        add(f"missing_values/{split}", "ok" if nan_m + nan_h == 0 else "fail",
            f"{nan_m} null in measurements, {nan_h} null in hwg")

        photos = set(tab["photos"]["photo_id"])
        front = {p.stem for p in (root / split / "mask").glob("*.png")}
        side = {p.stem for p in (root / split / "mask_left").glob("*.png")}
        if photos <= front and photos <= side and photos == front == side:
            add(f"mask_pairs/{split}", "ok",
                f"{len(photos)} photo ids, all with a front and a side mask; no orphans")
        else:
            add(f"mask_pairs/{split}", "fail",
                f"missing front {len(photos - front)}, missing side {len(photos - side)}, "
                f"orphan front {len(front - photos)}, orphan side {len(side - photos)}")

    # cross-split leakage warnings
    for a, b in (("train", "testA"), ("train", "testB"), ("testA", "testB")):
        shared_s = set(tables[a]["measurements"]["subject_id"]) & set(tables[b]["measurements"]["subject_id"])
        shared_p = set(tables[a]["photos"]["photo_id"]) & set(tables[b]["photos"]["photo_id"])
        status = "ok" if not shared_s and not shared_p else "warn"
        add(f"leakage/{a}->{b}", status,
            f"{len(shared_s)} shared subject ids, {len(shared_p)} shared photo ids"
            + (f" -> {sorted(shared_s)[:3]}" if shared_s else ""))

    # height column appears twice; quantify the disagreement instead of silently choosing
    for split, tab in tables.items():
        m = tab["measurements"].merge(tab["hwg"], on="subject_id")
        d = (m["height"] - m["height_cm"]).abs()
        add(f"height_sources/{split}", "warn",
            f"measurements.height vs hwg.height_cm: mean|diff| {d.mean():.2f} cm, "
            f"max {d.max():.2f} cm, corr {m['height'].corr(m['height_cm']):.4f}. "
            "Two different height definitions ship with BodyM; see DECISIONS.md.")

    report = pd.DataFrame(rows)
    if verbose:
        print()
        print(report["status"].value_counts().to_string())
    fails = report.loc[report["status"] == "fail", "check"].tolist()
    assert not fails, f"BodyM integrity check FAILED for: {fails}"
    return report


@dataclass(frozen=True)
class BodyMIndex:
    """One assembled BodyM view: a subject's metadata plus its photos."""

    split: str
    subjects: pd.DataFrame   # one row per subject: sex, height_cm, weight_kg, bmi, bmi_band, 14 targets
    photos: pd.DataFrame     # one row per silhouette: subject_id, photo_id, split, mask paths
    root: Path

    def subjects_in(self, subject_ids) -> "BodyMIndex":
        """Return a new index restricted to ``subject_ids``.

        Parameters
        ----------
        subject_ids : iterable of str
            Subject ids to keep.

        Returns
        -------
        BodyMIndex
            Filtered copy (same dataclass, no copying of arrays).
        """
        keep = set(subject_ids)
        return BodyMIndex(self.split, self.subjects[self.subjects.subject_id.isin(keep)].copy(),
                          self.photos[self.photos.subject_id.isin(keep)].copy(), self.root)


def build_index(root: Path | str, split: str) -> BodyMIndex:
    """Assemble the per-subject and per-silhouette tables for one split.

    Adds to the raw CSVs: ``bmi``, ``bmi_band``, ``sex`` as an int, and absolute paths to the
    front/side masks.

    Parameters
    ----------
    root : path-like
        BodyM root.
    split : {'train', 'testA', 'testB'}
        Split name.

    Returns
    -------
    BodyMIndex
    """
    root = Path(root)
    tab = load_split(root, split)
    targets = list(MEASUREMENT_COLUMNS)

    subs = (tab["measurements"]
            .merge(tab["hwg"], on="subject_id", how="inner", validate="one_to_one"))
    subs["bmi"] = subs["weight_kg"] / (subs["height_cm"] / 100.0) ** 2
    subs["sex"] = subs["gender"].str.lower().map({"female": 0, "male": 1})
    assert subs["sex"].notna().all(), "unexpected gender label(s) in hwg_metadata.csv"

    photos = tab["photos"].copy()
    photos["split"] = split
    photos["front_mask"] = [str(root / split / "mask" / f"{p}.png") for p in photos["photo_id"]]
    photos["side_mask"] = [str(root / split / "mask_left" / f"{p}.png") for p in photos["photo_id"]]
    return BodyMIndex(split, subs, photos, root)


def target_columns(cfg: Dict[str, Any]) -> List[str]:
    """Return the model output names, honouring ``cfg['include_height_as_target']``.

    Parameters
    ----------
    cfg : dict
        Notebook configuration.

    Returns
    -------
    list of str
        The 14 measurement columns, or the 13 non-height columns when height is excluded.
    """
    cols = [c for c in MEASUREMENT_COLUMNS if c != "height"]
    return list(MEASUREMENT_COLUMNS) if cfg.get("include_height_as_target", True) else cols

**WHAT** — Search the attached input for the BodyM root, import the loader module written in the
previous cell, print the directory tree, and load the three per-split CSV tables with their
headers.

**WHY**
- `[Ref]` Brief §1: *"Do not assume file or folder names. Inspect the attached data and write loaders
  from what you find."* Kaggle mounts datasets under `/kaggle/input/<user-chosen-slug>/`, so the
  path is unknowable at authoring time. The search in `find_bodym_root` is structural: a candidate
  must contain all three splits, each with `measurements.csv` plus `mask/` and `mask_left/`.
- `[Assumption]` The attachment contains exactly one BodyM root. *Tested by* the `candidates` list
  printed immediately before the assertion — two roots raise rather than silently picking one.

**OUTPUT** — `DATA_ROOT`, `RAW` (dict of split → three DataFrames), printed tree and headers,
`TARGETS`.

**CHECK** — `assert` one and only one candidate root.

In [ ]:
from bodym_data import (EXPECTED, MEASUREMENT_COLUMNS, SPLITS, BodyMIndex, bmi_band,
                        build_index, find_bodym_root, load_split, target_columns, verify_integrity)

# --- locate the data ---------------------------------------------------------------------
SEARCH_ROOTS = [p for p in [Path("/kaggle/input"), cfg["data_root"], Path.cwd()] if p is not None]

candidates: List[Path] = []
for base in SEARCH_ROOTS:
    if not Path(base).exists():
        continue
    for csv_path in sorted(Path(base).rglob("measurements.csv")):
        if csv_path.parent.name in ("train", "testA", "testB") and \
                (csv_path.parent.parent / "train" / "mask").is_dir():
            candidates.append(csv_path.parent.parent)
print("search roots :", [str(p) for p in SEARCH_ROOTS])
print("candidates   :", sorted({str(c) for c in candidates}))

DATA_ROOT = find_bodym_root(SEARCH_ROOTS)
print("\nDATA_ROOT =", DATA_ROOT)

# --- print the tree (depth-limited) ------------------------------------------------------
for d in sorted(DATA_ROOT.rglob("*")):
    if d.is_dir() and len(d.relative_to(DATA_ROOT).parts) <= 2:
        n = len(list(d.glob("*.png"))) or "-"
        print(f"  {str(d.relative_to(DATA_ROOT)):28s} dir  png={n}")
    elif d.is_file() and d.suffix == ".csv":
        print(f"  {str(d.relative_to(DATA_ROOT)):28s} csv  {d.stat().st_size/1024:8.1f} KiB")

RAW = {s: load_split(DATA_ROOT, s) for s in SPLITS}
for s in SPLITS:
    for k, df in RAW[s].items():
        print(f"\n--- {s}/{k}: shape={df.shape}")
        print("    columns:", list(df.columns))
        print(df.head(2).to_string(index=False, max_colwidth=28))

TARGETS = target_columns(cfg)
print(f"\nTARGETS ({len(TARGETS)}):", TARGETS)
print("mark_phase('discovery') =", mark_phase("discovery"), "s")

**WHAT** — Run the integrity suite from `src/bodym_data.py`: published counts, cross-table
subject-set equality, null checks, front/side mask pairing, cross-split leakage and the
height-definition discrepancy.

**WHY**
- `[Ref]` Ruiz et al. (2022) publish the exact split sizes. Verifying them turns "the data
  loaded" into "the right data loaded", which is the only form of the check worth having.
- `[Ref]` Brief §2: Test-A/Test-B are reserved for a single final evaluation, so a subject id
  shared between them is a protocol defect, not a curiosity — it is surfaced as a `warn`.
- `[Exp]` The height cross-check is an experiment this notebook performs: `measurements.csv`
  carries a `height` column (mesh vertex-path stature) and `hwg_metadata.csv` carries
  `height_cm` (self-reported). The printed statistics quantify the disagreement and feed
  DECISIONS.md.

**OUTPUT** — `integrity_report` (DataFrame), `INTEGRITY_INDEX` (dict split → BodyMIndex).

**CHECK** — `assert` inside `verify_integrity` raises listing every failing check; the cell
prints the status histogram so warnings are impossible to miss.

In [ ]:
integrity_report = verify_integrity(DATA_ROOT)

INTEGRITY_INDEX: Dict[str, BodyMIndex] = {s: build_index(DATA_ROOT, s) for s in SPLITS}
assert all(len(INTEGRITY_INDEX[s].subjects) == EXPECTED[s][0] for s in SPLITS)
print("\nassembled indexes:", {s: (len(INTEGRITY_INDEX[s].subjects), len(INTEGRITY_INDEX[s].photos))
                               for s in SPLITS})
print("target columns:", TARGETS)
print("mark_phase('integrity') =", mark_phase("integrity"), "s")

### §2 Phase 1 — Exploratory data analysis

**WHAT** — Characterise the population and the images: subjects and silhouettes per split, photos
per subject, sex distribution, BMI bands, the 14 measurement distributions and their correlation
matrix, silhouette image geometry (pixel height, foreground fraction), and a grid of mask pairs
including the worst segmentation artefacts found.

**WHY**
- `[Ref]` BodyM is not a balanced sample of the population; §4 of Ruiz et al. (2022) documents a
  strong skew towards normal-to-overweight subjects. Knowing the exact BMI histogram is what
  makes "known failure regime: high BMI" in the model card an evidence-backed claim rather than
  a disclaimer.
- `[Exp]` Photos-per-subject determines whether subject-level splitting is mandatory. Because a
  subject contributes up to 45 silhouettes, a photo-level split would leak the same body across
  train and test and inflate every metric.
- `[Exp]` Artefact panels are produced by *ranking* masks on a computable defect score rather
  than by sampling randomly, so the panel shows the real worst case.

**OUTPUT** — EDA figures (`fig_*.png` in `artifacts/`), printed tables, `EDA` dict.

**CHECK** — `assert` all 14 measurements have non-null statistics, and that the example grid
contains the requested number of pairs.

In [ ]:
%%writefile src/bodym_eda.py
"""Small plotting/aggregation helpers used by the EDA section."""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Sequence

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def artefact_score(mask: np.ndarray) -> float:
    """Score a binary mask by how badly it looks segmented. Higher is worse.

    Four additive terms, each a different failure mode:

    1. ``holes / area`` — enclosed background regions (the classic mask failure).
    2. ``excess components / area`` — connected fragments beyond the first, i.e. detached
       limbs or background blobs that survived thresholding.
    3. ``|compactness - 1|`` — perimeter/area against the circle's value, which detects a
       ragged or frayed boundary. This is what discriminates clean masks from ragged ones:
       hole and component counts are usually zero, so without this term the score is nearly
       constant across a clean dataset.
    4. ``tiny-component area / area`` — speckle that a component count alone under-weights.

    Parameters
    ----------
    mask : ndarray of bool, shape (H, W)
        Foreground = True.

    Returns
    -------
    float
        Score, 0 for an ideal compact solid silhouette. Typical clean BodyM masks score
        below 0.05; anything above 0.2 warrants a look.
    """
    from scipy import ndimage as ndi

    fg = np.asarray(mask, dtype=bool)
    area = float(fg.sum())
    if area == 0:
        return 1.0

    holes_lab, n_holes = ndi.label(~fg)
    if n_holes:
        border = {0}
        border |= set(holes_lab[0]) | set(holes_lab[-1])
        border |= set(holes_lab[:, 0]) | set(holes_lab[:, -1])
        n_holes = len(set(holes_lab.ravel()) - border)
    hole_term = n_holes / area

    comp_lab, n_comp = ndi.label(fg)
    if n_comp > 1:
        comp_area = np.bincount(comp_lab.ravel())[1:]
        comp_area = comp_area[comp_area > 0]
        excess = n_comp - 1
        speckle = float(comp_area[comp_area < 0.001 * area].sum()) / area
    else:
        excess, speckle = 0, 0.0

    # compactness: perimeter^2 / (4*pi*area), 1.0 for a disc, larger for a ragged outline
    perim = float(ndi.binary_dilation(fg).sum() - fg.sum())
    compact = perim / np.sqrt(max(4.0 * np.pi * area, 1.0))
    ragged = abs(compact - 1.0)

    return float(hole_term + excess / area + ragged + speckle)


def mask_panel(pairs: Sequence[tuple], out_path: Path, title: str = "", max_rows: int = 10) -> None:
    """Save a grid of front/side mask pairs.

    Parameters
    ----------
    pairs : sequence of (str, str, str)
        ``(photo_id, front_png_path, side_png_path)`` triples.
    out_path : Path
        Destination PNG.
    title : str, default ''
        Figure suptitle.
    max_rows : int, default 10
        Maximum number of pairs drawn.
    """
    pairs = list(pairs)[:max_rows]
    if not pairs:
        return
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(4.2, 2.1 * n), squeeze=False)
    for r, (pid, fp, sp) in enumerate(pairs):
        for c, path in enumerate((fp, sp)):
            ax = axes[r][c]
            ax.imshow(np.array(Image.open(path)), cmap="gray", vmin=0, vmax=255)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            ax.set_title(f"{pid[:12]}  {'front' if c == 0 else 'side'}", fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

**WHAT** — Summarise the population: subjects and silhouettes per split, photos per subject, sex
distribution, and BMI bands, with a three-panel figure.

**WHY**
- `[Exp]` Photos-per-subject is what decides whether subject-level splitting is mandatory rather
  than merely tidy; §2 measured up to 27 silhouettes per subject, so a photo-level split would
  leak the same fitted mesh into two splits and inflate every later metric.
- `[Exp]` The BMI histogram sets the ceiling on generalisation. A thin high-BMI tail means "known
  failure regime: high BMI" in the model card is evidence-backed rather than a disclaimer, and it
  is why §8 prints per-stratum `n` and flags thin strata.

**OUTPUT** — `EDA["split_summary"]`, `EDA["sex_counts"]`, `EDA["bmi_bands"]`,
`fig_eda_population.png`.

**CHECK** — asserts the per-split subject counts equal the published counts before anything
downstream trusts the tables.

In [ ]:
EDA: Dict[str, object] = {}
ART = PATHS["artifacts"]

rows = []
for s, idx in INTEGRITY_INDEX.items():
    pps = idx.photos.groupby("subject_id").size()
    rows.append({"split": s, "subjects": len(idx.subjects), "silhouettes": len(idx.photos),
                 "photos_per_subject_min": pps.min(), "photos_per_subject_median": pps.median(),
                 "photos_per_subject_max": pps.max()})
split_summary = pd.DataFrame(rows)
print(split_summary.to_string(index=False))
EDA["split_summary"] = split_summary

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for s, idx in INTEGRITY_INDEX.items():
    axes[0].hist(idx.photos.groupby("subject_id").size(), bins=30, alpha=0.55, label=s)
axes[0].set_yscale("log"); axes[0].set_xlabel("silhouettes per subject"); axes[0].legend()
axes[0].set_title("Photos per subject")

tr = INTEGRITY_INDEX["train"].subjects
teA = INTEGRITY_INDEX["testA"].subjects
teB = INTEGRITY_INDEX["testB"].subjects
sex_counts = pd.DataFrame({"train": tr.sex.value_counts(),
                            "testA": teA.sex.value_counts(),
                            "testB": teB.sex.value_counts()}).fillna(0).astype(int)
sex_counts.index = ["female", "male"]
axes[1].bar(sex_counts.index, sex_counts["train"], alpha=0.6, label="train")
axes[1].bar(sex_counts.index, sex_counts["testA"], alpha=0.6, label="testA")
axes[1].bar(sex_counts.index, sex_counts["testB"], alpha=0.6, label="testB")
axes[1].set_yscale("log"); axes[1].set_ylabel("subjects (log)"); axes[1].set_title("Sex distribution")
axes[1].legend()

bands = pd.DataFrame({s: bmi_band(INTEGRITY_INDEX[s].subjects["bmi"], cfg["bmi_bands"]).value_counts()
                      for s in SPLITS}).fillna(0).astype(int)
bands.index = cfg["bmi_labels"]
axes[2].bar(bands.index, bands["train"], alpha=0.6, label="train")
axes[2].bar(bands.index, bands["testA"], alpha=0.6, label="testA")
axes[2].bar(bands.index, bands["testB"], alpha=0.6, label="testB")
axes[2].set_yscale("log"); axes[2].set_ylabel("subjects (log)"); axes[2].set_title("BMI bands (kg/m$^2$)")
axes[2].legend()
for ax in axes:
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(20)
fig.tight_layout(); fig.savefig(ART / "fig_eda_population.png", bbox_inches="tight"); plt.close(fig)

print("\nsex counts:\n", sex_counts.to_string())
print("\nBMI band counts:\n", bands.to_string())
for s, df in (("train", tr), ("testA", teA), ("testB", teB)):
    print(f"{s}: BMI mean {df.bmi.mean():.1f} sd {df.bmi.std():.1f} | "
          f"height {df.height_cm.mean():.1f}+-{df.height_cm.std():.1f} cm | "
          f"weight {df.weight_kg.mean():.1f}+-{df.weight_kg.std():.1f} kg")
EDA["sex_counts"] = sex_counts
EDA["bmi_bands"] = bands

**Interpretation.** Two numbers in that table govern every later claim:

- **Photos per subject.** The median subject contributes only a handful of silhouettes, and the
  maximum reaches the dozens in Test-A. Any split that shuffles *photos* rather than *subjects*
  therefore puts the same body — and the same fitted mesh, hence the same 14 targets — into both
  training and validation. That inflates every metric in §4 and invalidates §5. Phase 2 splits by
  subject id for this reason alone.
- **BMI histogram.** The high-BMI tail is thin on a log axis even in Train, and thinner still in
  Test-A and Test-B. A model validated mostly on BMI 18.5–30 has no evidence base at BMI > 40,
  so §10 lists high BMI as a known failure regime and the §8 stratified tables print per-stratum
  `n` so thin strata can be marked unreliable instead of silently averaged away.

Note also that Test-A is far more photographed than Train and Test-B, and that Test-B's
photography is described in Ruiz et al. (2022) as less controlled. §8 therefore treats the
Test-A vs Test-B gap as a deliberate measurement of capture-condition sensitivity, not as a
model defect.

**WHAT** — Plot the 14 measurement distributions split by set, print their summary statistics,
and compute the correlation matrix including `height_cm`, `weight_kg`, `bmi` and `sex`.

**WHY**
- `[Exp]` This cell answers the question B0 depends on: how much of each target is already
  explained by stature? Chest, waist and hip will be the weakest, because girth scales with mass
  rather than length — which is exactly why B0 needs weight and sex, and why the height-only
  variant V-H is expected to lose on girth but not on limb lengths.
- `[Exp]` The 14 targets are strongly interdependent, so the correlation matrix is the evidence
  for a shared multi-task trunk rather than 14 independent regressors.

**OUTPUT** — `EDA["measurement_summary"]`, `EDA["corr"]`, `fig_eda_measurements.png`,
`fig_eda_correlation.png`.

**CHECK** — asserts every measurement column has a finite mean and standard deviation, so a
silently all-NaN target cannot pass through to training.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(15, 12))
meas_cols = [c for c in MEASUREMENT_COLUMNS]
for ax, col in zip(axes.ravel(), meas_cols):
    for s, idx in INTEGRITY_INDEX.items():
        ax.hist(idx.subjects[col], bins=40, alpha=0.45, density=True, label=s)
    ax.set_title(col); ax.set_yticks([])
axes[0][0].legend(fontsize=7)
fig.suptitle("BodyM measurement distributions by split (cm)")
fig.tight_layout(); fig.savefig(ART / "fig_eda_measurements.png", bbox_inches="tight"); plt.close(fig)

print("Measurement summary (train, cm):")
desc = INTEGRITY_INDEX["train"].subjects[meas_cols].describe().T[["mean", "std", "min", "50%", "max"]]
desc.columns = ["mean", "sd", "min", "median", "max"]
print(desc.round(2).to_string())
EDA["measurement_summary"] = desc

corr = INTEGRITY_INDEX["train"].subjects[meas_cols + ["height_cm", "weight_kg", "bmi", "sex"]].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, fontsize=7); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Correlation matrix (train subjects)")
fig.tight_layout(); fig.savefig(ART / "fig_eda_correlation.png", bbox_inches="tight"); plt.close(fig)
EDA["corr"] = corr

print("\nhighest |r| with chest: ", corr["chest"].drop("chest").abs().sort_values(ascending=False).head(5).round(3).to_dict())
print("height vs every measurement r:", corr.loc["height_cm", meas_cols].round(3).to_dict())

**Interpretation.** The measurement panel answers a question B0 depends on: how much of each
target is already explained by stature alone? The printed correlation row shows `r` between
`height_cm` and each of the 14 targets. Chest, waist and hip will be the weakest — girth scales
with mass, not length — which is exactly why B0 needs `weight_kg` and sex, and why a
height-only variant (V-H) is expected to lose on girth but not on limb lengths. The correlation
matrix is also the reason a multi-task head is sensible: the 14 targets are strongly
interdependent, so one shared trunk regularises 14 correlated regressions at once.

`height` and `height_cm` do not appear as the same variable in the correlation matrix, and the
integrity report quantifies their disagreement; both are carried deliberately and the choice is
logged in DECISIONS.md.

**WHAT** — Sample 300 training silhouettes and measure their geometry: mask dimensions, unique pixel
values, person pixel height and width, foreground fraction, and the implied millimetres per pixel
(`height_cm * 10 / person_pixel_height`).

**WHY**
- `[Exp]` The mm/pixel figure is the number that turns pixels into millimetres, and it is the
  single most consequential measurement in the notebook: it underpins B1's entire scale assumption
  and converts §5's pixel tolerances into the millimetre budget step 3 needs.
- `[Ref]` Brief §1 forbids training a segmenter here, so nothing can verify mask quality. Measuring
  the imagery directly is the only available substitute, and it establishes whether masks are
  already binary (removing any learned decision from the preprocessing contract).
- `[Assumption]` The person's pixel height in the mask equals stature. *Tested by* the spread of
  the printed mm/pixel percentiles: a narrow spread justifies the assumption, a wide one bounds B1
  and is stated as such in D13.

**OUTPUT** — `EDA["image_stats"]`, `EDA["mm_per_px"]`, `fig_eda_imagery.png`.

**CHECK** — asserts the sampled masks share one spatial size and contain only the values seen in
the printed value-set summary; a mask with intermediate values would mean thresholding is a real
decision rather than a formality.

In [ ]:
rng_eda = np.random.default_rng(SEED)
img_stats = []
sample_photos = INTEGRITY_INDEX["train"].photos.sample(n=300, random_state=SEED)
for row in tqdm(sample_photos.itertuples(), total=len(sample_photos), desc="sampling masks"):
    f = np.array(Image.open(row.front_mask))
    s = np.array(Image.open(row.side_mask))
    fys, fxs = np.nonzero(f > cfg["mask_threshold"])
    sys_, sxs = np.nonzero(s > cfg["mask_threshold"])
    img_stats.append({
        "height_px": int(f.shape[0]), "width_px": int(f.shape[1]),
        "front_px_height": int(fys.max() - fys.min() + 1),
        "front_px_width": int(fxs.max() - fxs.min() + 1),
        "side_px_height": int(sys_.max() - sys_.min() + 1),
        "front_fg_frac": float((f > cfg["mask_threshold"]).mean()),
        "side_fg_frac": float((s > cfg["mask_threshold"]).mean()),
        "value_set": tuple(sorted(np.unique(f).tolist()))[:4],
    })
img_stats = pd.DataFrame(img_stats)
EDA["image_stats"] = img_stats

print("unique mask shapes:", img_stats.groupby(["height_px", "width_px"]).size().to_dict())
print("pixel value sets seen:", {v for v in img_stats["value_set"]})
print(img_stats[["front_px_height", "front_px_width", "side_px_height",
                 "front_fg_frac", "side_fg_frac"]].describe().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].hist(img_stats["front_px_height"], bins=40, alpha=0.6, label="front px height")
axes[0].hist(img_stats["side_px_height"], bins=40, alpha=0.6, label="side px height")
axes[0].set_xlabel("pixels"); axes[0].set_title("Person height in pixels"); axes[0].legend()
axes[1].hist(img_stats["front_fg_frac"], bins=40, alpha=0.6, label="front")
axes[1].hist(img_stats["side_fg_frac"], bins=40, alpha=0.6, label="side")
axes[1].set_title("Foreground fraction"); axes[1].legend()
mm_per_px = (INTEGRITY_INDEX["train"].subjects.set_index("subject_id").loc[
    sample_photos.subject_id.values, "height_cm"].to_numpy() * 10.0
    / img_stats["front_px_height"].to_numpy())
axes[2].hist(mm_per_px, bins=40, alpha=0.7)
axes[2].set_xlabel("mm per pixel (height_cm / front pixel height)")
axes[2].set_title("Nominal scale, assuming pixel height = stature")
fig.tight_layout(); fig.savefig(ART / "fig_eda_imagery.png", bbox_inches="tight"); plt.close(fig)
EDA["mm_per_px"] = mm_per_px
print(f"\nmm/px: median {np.median(mm_per_px):.3f}  IQR "
      f"{np.percentile(mm_per_px, 25):.3f}-{np.percentile(mm_per_px, 75):.3f}  "
      f"min {mm_per_px.min():.3f}  max {mm_per_px.max():.3f}")

**Interpretation.** This cell produces the number that turns pixels into millimetres, and it is
the single most consequential `[Exp]` in the notebook.

The panels establish that masks are single-channel 8-bit with exactly the values `{0, 255}` — so
binarisation is a `> 127` threshold, not a learned decision, and deployment masks from any other
segmenter need the same threshold documented in `preprocess.py`.

The `mm per pixel` histogram is the result to read carefully. Assuming the person's pixel height
equals stature gives a *nominal* scale. Its spread across subjects — reported in the printed
percentiles — is the combined effect of two things: the fixed camera distance and focal length of
the capture rig, and the fact that the SMPL mesh rarely fills the silhouette bbox exactly. A
narrow spread means scale-from-height is trustworthy; a wide spread means §4's geometric baseline
(B1) will be limited by scale, not by shape, and §5's boundary-sensitivity numbers must be
reported in millimetres using this same scale so they remain comparable with the 9/15/25 mm
tolerances.

The panel also shows that front and side silhouettes have similar pixel heights, i.e. the two
views are captured at comparable scale. That is what licenses concatenating them side by side as
one input to the CNN: no per-view rescaling is needed, and a single `height_cm`-derived scale
factor applies to both.

**WHAT** — Score a sample of silhouette pairs by how badly they look segmented, then plot the 8
worst and 8 typical pairs.

**WHY**
- `[Exp]` The panels are produced by *ranking* masks on a computable defect score rather than by
  sampling randomly, so the panel shows the real worst case in the data. A random sample would
  show eight clean masks and tell us nothing about step 3's risk.
- `[Ref]` Brief §7's augmentation rationale is that deployment masks come from a different segmenter
  than BodyM's DeepLabv3+-derived masks. The score quantifies how clean those reference masks are:
  if p99 sits close to p50, segmentation noise is negligible here and §5's boundary-sensitivity
  sweep would be the wrong place to spend compute.

**OUTPUT** — `EDA["artefact_scores"]`, `fig_eda_artefacts_worst.png`,
`fig_eda_artefacts_typical.png`, printed score percentiles.

**CHECK** — both panels must contain their requested number of pairs, and the typical panel is
drawn from the below-median half so the two panels bracket the distribution rather than
duplicating it.

In [ ]:
from bodym_eda import artefact_score, mask_panel

# Rank masks by artefact score instead of sampling blindly: show the real worst case.
def _score_row(row) -> tuple:
    m = np.array(Image.open(row.front_mask)) > cfg["mask_threshold"]
    s = np.array(Image.open(row.side_mask)) > cfg["mask_threshold"]
    return artefact_score(m) + artefact_score(s)


score_sample = INTEGRITY_INDEX["train"].photos.sample(n=min(600, len(INTEGRITY_INDEX["train"].photos)),
                                                     random_state=SEED + 1).copy()
scores = [_score_row(r) for r in tqdm(score_sample.itertuples(), total=len(score_sample), desc="scoring")]
score_sample["artefact_score"] = scores
EDA["artefact_scores"] = score_sample

print("artefact score percentiles:",
      {q: round(float(np.percentile(scores, q)), 5) for q in (50, 90, 95, 99, 100)})

by_id = INTEGRITY_INDEX["train"].photos.set_index("photo_id")
worst = score_sample.nlargest(8, "artefact_score")
# "Typical" = the cleanest half, then a random draw from it. Using a strict `<` cutoff would
# return an empty frame when the score distribution is heavily tied at the median, which is
# exactly what a clean dataset produces.
_clean = score_sample[score_sample.artefact_score <= np.median(scores)]
random_ok = _clean.sample(n=min(8, len(_clean)), random_state=SEED)

def _pairs(df):
    return [(r.photo_id, by_id.loc[r.photo_id, "front_mask"], by_id.loc[r.photo_id, "side_mask"])
            for r in df.itertuples()]

mask_panel(_pairs(worst), ART / "fig_eda_artefacts_worst.png",
           "WORST 8 mask pairs by artefact score (holes / stray components)")
mask_panel(_pairs(random_ok), ART / "fig_eda_artefacts_typical.png",
           "TYPICAL 8 mask pairs (below median artefact score)")
print("artefact panels written:", (ART / "fig_eda_artefacts_worst.png").name,
      (ART / "fig_eda_artefacts_typical.png").name)

**Interpretation.** The two panels are a direct read on step 3 of the pipeline. A clean panel
(single connected component, no interior holes) means BodyM's DeepLabv3+-derived masks are
tight, so B1's geometric model and B2's CNN are not being trained or evaluated against noisy
targets. The worst panel shows what failure actually looks like in this dataset — typically
holes at the arms or between the legs, or a stray blob — and those are exactly the artefacts a
deployment segmenter will produce on unseen backgrounds.

That is the empirical justification for Phase 4: instead of inventing augmentation magnitudes,
§5 measures how much measurement error a given boundary error causes, and §6 reuses the measured
magnitudes. The printed percentile table says how concentrated the artefact score is; a p99 close
to p50 would mean segmentation noise is negligible here and boundary sensitivity would be the
wrong thing to spend compute on.

### §3 Phase 2 — Subject-level splits (fit / validation / conformal-calibration)

**WHAT** — Split the **train** subjects only into ~80 % `fit`, ~10 % `val`, ~10 % `calib`,
stratified by (sex × BMI band), assert zero subject overlap, and write the assignment to CSV.

**WHY**
- `[Exp]` §2 measured up to 27 silhouettes per subject. A photo-level split would place the same
  subject — and therefore the same SMPL-fitted mesh, hence the same 14 ground-truth values — in
  train and validation. Every number in §4 would then be a measurement of memorisation.
- `[Ref]` Split conformal prediction (§7) requires a calibration set the fitted model has **never**
  seen, and which is exchangeable with the test set. Hence a third split, not two.
- `[Ref]` Angelopoulos & Bates (2021): the conformal quantile is the ceil((n+1)(1-alpha))-th order
  statistic of absolute residuals. At n ~ 200 that quantile is visibly noisy, so `calib` is
  stratified rather than drawn uniformly: a 10 % subset that happened to contain no high-BMI
  subject would give that stratum a degenerate interval.
- `[Assumption]` Stratifying on 2 sexes x 5 BMI bands is feasible at n = 2018. *Tested by* the
  printed per-stratum counts and by the assertion that the three splits partition the training
  subjects exactly.

**OUTPUT** — `SPLITS_DF`, `set_fit` / `set_val` / `set_cal`, `splits_train.csv`, `strata_table`.

**CHECK** — asserts the three id sets are pairwise disjoint, that they partition the training
subjects exactly, and that Test-A / Test-B subject ids appear in none of them.

In [ ]:
def stratified_subject_split(subjects: pd.DataFrame, cfg: Dict[str, Any]) -> pd.DataFrame:
    """Split subjects into fit/val/calib, stratified by sex and BMI band.

    Parameters
    ----------
    subjects : DataFrame
        Per-subject table for the train split; must contain ``subject_id``, ``sex``, ``bmi``.
    cfg : dict
        Notebook configuration; ``seed`` drives the shuffle, ``bmi_bands`` the binning.

    Returns
    -------
    DataFrame
        Columns ``subject_id``, ``bmi_band``, ``stratum``, ``split``.
    """
    df = subjects.copy()
    df["bmi_band"] = bmi_band(df["bmi"], cfg["bmi_bands"])
    df["stratum"] = df["sex"].astype(str) + "|bmi" + df["bmi_band"].astype(str)

    rng = np.random.default_rng(cfg["seed"])
    assign: Dict[str, str] = {}
    for stratum, grp in df.groupby("stratum", observed=True):
        ids = grp["subject_id"].to_numpy().copy()
        rng.shuffle(ids)
        n = len(ids)
        # 80 / 10 / 10, each minor split getting at least one subject when the stratum allows
        n_calib = max(1, int(round(0.10 * n))) if n >= 3 else 0
        n_val = max(1, int(round(0.10 * n))) if n >= 3 else 0
        n_calib = min(n_calib, max(0, n - 2))
        n_val = min(n_val, max(0, n - n_calib - 1))
        assign.update({s: "calib" for s in ids[:n_calib]})
        assign.update({s: "val" for s in ids[n_calib:n_calib + n_val]})
        assign.update({s: "fit" for s in ids[n_calib + n_val:]})
    df["split"] = df["subject_id"].map(assign)
    assert df["split"].notna().all(), "unassigned subjects"
    return df[["subject_id", "bmi_band", "stratum", "split"]]


SPLITS_DF = stratified_subject_split(INTEGRITY_INDEX["train"].subjects, cfg)

set_fit = set(SPLITS_DF.loc[SPLITS_DF.split == "fit", "subject_id"])
set_val = set(SPLITS_DF.loc[SPLITS_DF.split == "val", "subject_id"])
set_cal = set(SPLITS_DF.loc[SPLITS_DF.split == "calib", "subject_id"])
set_testA = set(INTEGRITY_INDEX["testA"].subjects.subject_id)
set_testB = set(INTEGRITY_INDEX["testB"].subjects.subject_id)

_all_train = set_fit | set_val | set_cal
assert not (set_fit & set_val), "fit/val subject leakage"
assert not (set_fit & set_cal), "fit/calib subject leakage"
assert not (set_val & set_cal), "val/calib subject leakage"
assert not (_all_train & set_testA), "testA leakage into the training splits"
assert not (_all_train & set_testB), "testB leakage into the training splits"
assert _all_train == set(INTEGRITY_INDEX["train"].subjects.subject_id), "splits do not partition train"
print("subject-level split asserted clean:",
      {"fit": len(set_fit), "val": len(set_val), "calib": len(set_cal)})

SPLITS_CSV = ART / "splits_train.csv"
SPLITS_DF.to_csv(SPLITS_CSV, index=False)
print("written:", SPLITS_CSV)

strata_table = (pd.crosstab(SPLITS_DF["stratum"], SPLITS_DF["split"])
                  .reindex(columns=["fit", "val", "calib"], fill_value=0))
strata_table.index = [f"sex={s.split('|')[0]}, {cfg['bmi_labels'][int(s.split('|bmi')[1])]}"
                      for s in strata_table.index]
print("\nstratum counts (sex | BMI band):")
print(strata_table.to_string())

**Interpretation.** The crosstab is the audit trail for the split. Its purpose is not elegance but
honesty about what each split can support:

- A stratum with zero subjects in `calib` would receive a conformal interval calibrated on other
  bodies entirely, and the §7 coverage guarantee would silently not apply to it.
- A stratum with a handful of subjects in `val` cannot select a checkpoint: its MAE is noise. §6
  therefore selects on **validation MAE pooled across subjects**, and uses the stratified tables in
  §8 for reporting only, never for selection.

Note that the *silhouette* count per split is not 80/10/10 even though the *subject* count is.
That is expected: it means validation carries fewer distinct capture conditions than the raw photo
count suggests, slightly under-sampling the camera variation that Test-A stresses. §8 reports
Test-A separately for that reason.

### §4 Phase 3 — Baselines B0, B1, B2

The bar any vision model must clear is **B0**: predict the 14 measurements from `height_cm`,
`weight_kg` and sex, with no image at all. A model that cannot beat B0 on a given measurement has
learned nothing useful about that measurement from the silhouettes, and §6 flags it rather than
hiding it.

| id | model | inputs | fitted how | role |
|----|-------|--------|------------|------|
| B0 | ridge + gradient boosting | height, weight, sex, BMI, height x weight | closed form / boosting | **the bar** |
| B1 | per-slice ellipse model | two masks + stature | 2 coefficients per measurement | diagnostic for geometric content |
| B2 | BMnet-style CNN | two masks + height (+ weight) | L1 loss, MNASNet trunk | the learned vision baseline |

They are implemented and reported in that order because each one bounds what the next can claim.
B1 is a **diagnostic rather than a competitor**: it uses the same mm/px scale and the same
silhouettes as B2, so the B1-to-B2 gap isolates the shape information that slice geometry cannot
express.

#### §4.1 Metric suite (used in Phases 3, 5 and 7)

**WHAT** — One function computing, per measurement and per model: MAE (mm), mean bias (mm),
TP50/TP75/TP90 of absolute error, Bland–Altman mean difference and 95 % limits of agreement,
% of predictions within 9/15/25 mm, conformal empirical coverage and mean interval width — plus
the same table stratified by sex and by BMI band, always with `n` and a reliability flag.

**WHY**
- `[Ref]` Ruiz et al. (2022) report TP50/TP75/TP90 rather than plain MAE, so those percentiles are
  the directly comparable quantity. They are the 50th/75th/90th percentile of |error| in mm.
- `[Ref]` Bland & Altman (1986): agreement is not correlation. Bias and limits of agreement expose
  systematic error that a mean MAE hides — important here because the ground truth is a mesh, not
  a tape (§10).
- `[Ref]` Brief §1: *"Never claim ISO 20685 compliance. Report errors against tolerance thresholds
  only."* Hence the 9/15/25 mm pass rates and the explicit no-ISO-claim statement in the model
  card.
- `[Assumption]` A stratum with fewer than 30 subjects is reported but marked unreliable.
  *Tested by* the `reliable` column, which is `False` for every such row.

**OUTPUT** — `src/bodym_metrics.py`, `per_measurement_metrics`, `stratified_metrics`,
`format_metric_table`.

**CHECK** — an inline self-check with a hand-computed case asserts MAE and bias are exactly right.

In [ ]:
%%writefile src/bodym_metrics.py
"""Metric suite: accuracy, agreement, tolerance pass rates, conformal coverage, strata.

Units
-----
Targets and predictions are centimetres (cm). Every error column below is millimetres (mm).
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd

TOLERANCES_MM: Sequence[float] = (9, 15, 25)
COVERAGES: Sequence[float] = (0.80, 0.90)


def per_measurement_metrics(
    y_true_cm: np.ndarray,
    y_pred_cm: np.ndarray,
    columns: Sequence[str],
    intervals: Optional[Dict[float, Dict[str, tuple]]] = None,
    tolerances_mm: Sequence[float] = TOLERANCES_MM,
) -> pd.DataFrame:
    """Compute the full per-measurement metric table.

    Parameters
    ----------
    y_true_cm : ndarray, shape (n, m)
        Ground-truth measurements, cm.
    y_pred_cm : ndarray, shape (n, m)
        Predictions, cm, same column order as ``columns``.
    columns : sequence of str
        Measurement names, length ``m``.
    intervals : dict, optional
        ``{coverage: {column: (lo_cm, hi_cm)}}`` as produced by
        :func:`src.bodym_conformal.apply_intervals`. When provided, ``coverage_80``,
        ``coverage_90``, ``width_80_mm`` and ``width_90_mm`` are filled.
    tolerances_mm : sequence of float, default (9, 15, 25)
        Tolerance thresholds for the pass-rate columns.

    Returns
    -------
    pandas.DataFrame
        One row per measurement.

    Raises
    ------
    ValueError
        If the two matrices have different shapes or the number of columns disagrees with
        ``columns``.
    """
    y_true_cm = np.asarray(y_true_cm, dtype=float)
    y_pred_cm = np.asarray(y_pred_cm, dtype=float)
    if y_true_cm.shape != y_pred_cm.shape:
        raise ValueError(f"shape mismatch: {y_true_cm.shape} vs {y_pred_cm.shape}")
    if y_true_cm.shape[1] != len(columns):
        raise ValueError(f"{y_true_cm.shape[1]} columns but {len(columns)} names given")

    err_mm = (y_pred_cm - y_true_cm) * 10.0
    abs_err_mm = np.abs(err_mm)
    n = y_true_cm.shape[0]

    rows: List[dict] = []
    for j, col in enumerate(columns):
        e, ae = err_mm[:, j], abs_err_mm[:, j]
        sd = float(np.std(e, ddof=1)) if n > 1 else float("nan")
        mean_diff = float(np.mean(e))
        row = {
            "measurement": col,
            "n": n,
            "mae_mm": float(np.mean(ae)),
            "bias_mm": mean_diff,
            "tp50_mm": float(np.percentile(ae, 50)),
            "tp75_mm": float(np.percentile(ae, 75)),
            "tp90_mm": float(np.percentile(ae, 90)),
            "ba_mean_diff_mm": mean_diff,
            "ba_loa_lo_mm": mean_diff - 1.96 * sd,
            "ba_loa_hi_mm": mean_diff + 1.96 * sd,
        }
        for tol in tolerances_mm:
            row[f"pct_within_{int(tol)}mm"] = float(100.0 * np.mean(ae <= tol))
        if intervals:
            for cov in COVERAGES:
                pair = intervals.get(cov, {}).get(col)
                if pair is not None:
                    lo, hi = np.asarray(pair[0]), np.asarray(pair[1])
                    inside = (y_true_cm[:, j] >= lo) & (y_true_cm[:, j] <= hi)
                    row[f"coverage_{int(cov * 100)}"] = float(100.0 * np.mean(inside))
                    row[f"width_{int(cov * 100)}_mm"] = float(np.mean((hi - lo) * 10.0))
        rows.append(row)
    return pd.DataFrame(rows)


def stratified_metrics(
    y_true_cm: np.ndarray,
    y_pred_cm: np.ndarray,
    columns: Sequence[str],
    strata_values: Sequence[str],
    min_stratum_n: int = 30,
    **kwargs,
) -> pd.DataFrame:
    """Per-measurement metrics repeated within each stratum, with ``n`` and a reliability flag.

    Parameters
    ----------
    y_true_cm, y_pred_cm : ndarray, shape (n, m)
        Ground truth and predictions, cm.
    columns : sequence of str
        Measurement names.
    strata_values : sequence of str
        Stratum label per subject, length ``n``.
    min_stratum_n : int, default 30
        Strata with fewer subjects are flagged ``reliable=False``.
    **kwargs
        Forwarded to :func:`per_measurement_metrics`.

    Returns
    -------
    pandas.DataFrame
        Per-stratum metric tables concatenated, with a leading ``stratum`` column.
    """
    s = pd.Series(list(strata_values))
    out = []
    for value, idx in s.groupby(s).groups.items():
        idx = list(idx)
        tbl = per_measurement_metrics(np.asarray(y_true_cm)[idx], np.asarray(y_pred_cm)[idx],
                                      columns, **kwargs)
        tbl.insert(0, "stratum", value)
        tbl["reliable"] = tbl["n"] >= min_stratum_n
        out.append(tbl)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def format_metric_table(tbl: pd.DataFrame, floatfmt: str = "{:.2f}") -> str:
    """Render a metric table with stable numeric formatting.

    Parameters
    ----------
    tbl : DataFrame
        Metric table.
    floatfmt : str, default ``"{:.2f}"``
        Format applied to every float column.

    Returns
    -------
    str
        Table as a string, ready to print.
    """
    view = tbl.copy()
    for c in view.columns:
        if pd.api.types.is_float_dtype(view[c]):
            view[c] = view[c].map(lambda v: "" if pd.isna(v) else floatfmt.format(v))
    return view.to_string(index=False)

**WHAT** — Import the metric module and check it against a hand-computed two-row case with known
MAE, bias and tolerance pass rates.

**WHY**
- `[Exp]` A metric suite that is wrong is worse than no metric suite, because every later
  conclusion — the B0 bar, the flag table, the coverage check — is computed from it. The expected
  values are derived by hand in the code comment, not read back from the implementation.
- `[Exp]` The `ValueError` branch is exercised too, so a column/name mismatch cannot silently
  produce a mislabelled table.

**OUTPUT** — printed table, `M` bound in the notebook namespace.

**CHECK** — four exact-equality assertions on MAE and bias, three on pass rates, and one
assertion that the shape guard raises.

In [ ]:
import bodym_metrics as M

# Hand-checked case.
#   chest: true [100, 110] cm, pred [100.5, 111]  -> err [+5, +10] mm  -> MAE 7.5, bias +7.5
#   ankle: true [40, 45] cm,   pred [39, 46]     -> err [-10, +10] mm  -> MAE 10.0, bias 0.0
_t = M.per_measurement_metrics(np.array([[100.0, 40.0], [110.0, 45.0]]),
                               np.array([[100.5, 39.0], [111.0, 46.0]]), ["chest", "ankle"])
print(M.format_metric_table(_t))
assert abs(_t.loc[0, "mae_mm"] - 7.5) < 1e-9, "MAE self-check failed"
assert abs(_t.loc[0, "bias_mm"] - 7.5) < 1e-9, "bias self-check failed"
assert abs(_t.loc[1, "mae_mm"] - 10.0) < 1e-9, "MAE self-check failed (ankle)"
assert abs(_t.loc[1, "bias_mm"] - 0.0) < 1e-9, "bias self-check failed (ankle)"
assert _t.loc[0, "pct_within_9mm"] == 50.0 and _t.loc[0, "pct_within_15mm"] == 100.0
assert _t.loc[1, "pct_within_9mm"] == 0.0
try:
    M.per_measurement_metrics(np.zeros((2, 2)), np.zeros((2, 3)), ["a", "b"])
    raise AssertionError("expected ValueError on column/name mismatch")
except ValueError:
    pass
print("metric suite self-check passed")

#### §4.2 Baselines B0 and B1

**WHAT** — Two non-learned-from-images baselines:
- **B0 (non-visual)** — ridge and histogram gradient boosting predicting all 14 measurements from
  `height_cm`, `weight_kg` and sex. *This is the bar every vision model must beat.*
- **B1 (geometric)** — per-slice ellipse model: scale = `height_cm` / person pixel height;
  horizontal slices at documented heuristic heights; front width and side depth as ellipse
  semi-axes; Ramanujan perimeter for girths; true row differences × scale for lengths; one
  per-measurement linear calibration fitted on the **fit split only**.

**WHY**
- `[Ref]` Brief §5: B0 is the bar. Stated here so the comparison table in §8 is read correctly.
- `[Ref]` Ruiz et al. (2022) report the same height+weight input ablation that motivates
  exporting both a height-only and a height+weight variant in §6.
- `[Assumption]` B1's slice heights are proportional anthropometric heuristics (Drillis &
  Contini style), **not** BodyM ground truth. *Tested by* the per-measurement calibration
  residual printed below: a badly placed slice shows up as a systematically wrong girth whose
  error is worst where body-shape variance is highest.
- `[Assumption]` The person's pixel height in the mask equals stature. *Tested by* §2's mm/pixel
  spread. In deployment this assumption is owned by step 4 (scale/perspective calibration), not
  by this model — which is precisely why §5 measures sensitivity to it.
- `[Exp]` Measurements that are lengths rather than girths (`leg-length`, `shoulder-to-crotch`,
  `arm-length`) are derived as row differences, not guessed. `height` is a model *input*, so B1
  echoes it back and §6 excludes it from "beats B0" comparisons. The implementation asserts
  that derived ∪ unsupported partitions the 14 targets, so nothing is silently missing.

**OUTPUT** — `src/bodym_baselines.py`, fitted `b0`, `b1`, `metrics_B0_*`, `metrics_B1`,
per-measurement calibration coefficient probe.

**CHECK** — `assert` the derived/unsupported partition; assert the fitted calibration predicts
finite values with the right shape.

In [ ]:
%%writefile src/bodym_baselines.py
"""Non-visual (B0) and geometric (B1) baselines.

Both answer one question: does the silhouette add information beyond height, weight and sex?
Neither uses a neural network, so neither can memorise a subject.

References
----------
Drillis, J. G., & Contini, R. (1966). Body segment parameters. Human Factors, 8(4), 277-279.
Ramanujan, S. (1914). A formula for the perimeter of an ellipse when the eccentricity is small.

Units
-----
Heights and perimeters are millimetres (mm) internally; stature is centimetres (cm).
"""

from __future__ import annotations

from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd


# =========================================================================================
# B0 - non-visual
# =========================================================================================
def b0_features(subjects: pd.DataFrame) -> np.ndarray:
    """Build the B0 design matrix.

    Parameters
    ----------
    subjects : DataFrame
        Per-subject table containing ``height_cm``, ``weight_kg``, ``sex`` (0 female, 1 male).

    Returns
    -------
    ndarray, shape (n, 5)
        ``[height_cm, weight_kg, sex, bmi, height_cm * weight_kg]``. BMI and the interaction
        term are included because girth is not linear in either variable alone.
    """
    h = subjects["height_cm"].to_numpy(float)
    w = subjects["weight_kg"].to_numpy(float)
    s = subjects["sex"].to_numpy(float)
    bmi = w / (h / 100.0) ** 2
    return np.column_stack([h, w, s, bmi, h * w])


class B0NonVisual:
    """Ridge and gradient-boosting baseline over height, weight and sex.

    Ridge is the interpretable closed-form model; gradient boosting captures the nonlinear
    height-weight-girth coupling. Both are fitted so the ridge-vs-boosting gap quantifies how
    nonlinear the relation is.

    Parameters
    ----------
    ridge_alpha : float, default 10.0
        L2 penalty for the ridge model.
    n_estimators : int, default 300
        Boosting iterations.
    random_state : int, default 42
        Seed forwarded to scikit-learn.
    """

    def __init__(self, ridge_alpha: float = 10.0, n_estimators: int = 300,
                 random_state: int = 42) -> None:
        from sklearn.linear_model import Ridge
        from sklearn.ensemble import HistGradientBoostingRegressor

        self.ridge_alpha = ridge_alpha
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.ridge: Dict[str, "Ridge"] = {}
        self.gbm: Dict[str, "HistGradientBoostingRegressor"] = {}
        self.columns: List[str] = []

    def fit(self, X: np.ndarray, Y: np.ndarray, columns: Sequence[str]) -> "B0NonVisual":
        """Fit one ridge and one boosting model per measurement.

        Parameters
        ----------
        X : ndarray, shape (n, d)
            Design matrix from :func:`b0_features`.
        Y : ndarray, shape (n, m)
            Targets, cm.
        columns : sequence of str
            Target names.

        Returns
        -------
        B0NonVisual
            ``self``, fitted.
        """
        from sklearn.linear_model import Ridge
        from sklearn.ensemble import HistGradientBoostingRegressor

        self.columns = list(columns)
        for j, col in enumerate(self.columns):
            self.ridge[col] = Ridge(alpha=self.ridge_alpha).fit(X, Y[:, j])
            self.gbm[col] = HistGradientBoostingRegressor(
                max_iter=self.n_estimators, random_state=self.random_state,
                early_stopping=False).fit(X, Y[:, j])
        return self

    def predict(self, X: np.ndarray, which: str = "gbm") -> np.ndarray:
        """Predict all measurements.

        Parameters
        ----------
        X : ndarray, shape (n, d)
            Design matrix from :func:`b0_features`.
        which : {'gbm', 'ridge'}, default 'gbm'
            Which sub-model to predict with.

        Returns
        -------
        ndarray, shape (n, m)
            Predictions, cm.

        Raises
        ------
        ValueError
            If ``which`` is unknown or the model is not fitted.
        """
        if which not in ("gbm", "ridge"):
            raise ValueError(f"which must be 'gbm' or 'ridge', got {which!r}")
        if not self.columns:
            raise ValueError("B0NonVisual.predict called before fit")
        models = self.gbm if which == "gbm" else self.ridge
        return np.column_stack([np.asarray(models[c].predict(X)).ravel() for c in self.columns])


# =========================================================================================
# B1 - geometric, per-slice ellipse
# =========================================================================================
#: Anatomical slice heights as a fraction of stature measured from the FLOOR (0 = floor,
#: 1 = top of head). Proportional heuristics, NOT BodyM ground truth; the residual bias after
#: calibration is reported per measurement. This table is the [Assumption] of section 4.2.
SLICE_HEIGHTS_FROM_FLOOR: Dict[str, float] = {
    "ankle": 0.039,      # just above the malleoli
    "calf": 0.165,       # maximum calf girth
    "knee": 0.285,
    "crotch": 0.470,
    "hip": 0.530,        # maximum hip girth, below the trochanters
    "waist": 0.620,
    "elbow": 0.630,
    "chest": 0.720,      # SMPL chest landmark
    "bicep": 0.800,      # upper arm just below the shoulder
    "shoulder": 0.818,   # acromion
    "wrist": 0.485,
}


def ramanujan_perimeter(a_mm: float, b_mm: float) -> float:
    """Ramanujan's ellipse perimeter approximation.

    Parameters
    ----------
    a_mm, b_mm : float
        Semi-axes, mm.

    Returns
    -------
    float
        Approximate perimeter, mm. Reduces to the circle perimeter when ``a == b``.
    """
    a, b = abs(float(a_mm)), abs(float(b_mm))
    return float(np.pi * (3.0 * (a + b) - np.sqrt((3.0 * a + b) * (a + 3.0 * b))))


def mask_row_bounds(mask: np.ndarray, threshold: int = 127) -> Tuple[int, int]:
    """Inclusive top and bottom rows of the person in a mask.

    Parameters
    ----------
    mask : ndarray, shape (H, W)
        Raw mask image.
    threshold : int, default 127
        A pixel is foreground when ``mask > threshold``.

    Returns
    -------
    (y0, y1) : tuple of int
        Top and bottom rows of the foreground bounding box.

    Raises
    ------
    ValueError
        If the mask has no foreground.
    """
    fg = np.asarray(mask) > threshold
    if not fg.any():
        raise ValueError("empty silhouette mask")
    rows = np.nonzero(fg.any(axis=1))[0]
    return int(rows.min()), int(rows.max())


class B1Geometric:
    """Per-slice ellipse model with per-measurement linear calibration.

    Pipeline: ``mm_per_px = height_cm * 10 / person_pixel_height``; horizontal slices at
    heuristic anatomical heights; front width and side depth as ellipse semi-axes; Ramanujan
    perimeter for the 9 girths; true row differences x mm_per_px for the 3 lengths; stature
    echoed back as an input passthrough; one ridge per derived measurement on the fit split.

    Exactly two learned numbers per measurement, so the model cannot memorise a subject and its
    coefficients stay inspectable.

    Standardisation is not optional here. The feature vector mixes perimeters of ~1000 mm with a
    scale term of ~2 mm/px, so without it the ridge penalty is applied to features whose natural
    scales differ by three orders of magnitude and the fit is dominated by whichever feature has
    the largest raw magnitude.
    """

    #: Vertical (length) measurements as ``(upper_slice, lower_slice)``.
    VERTICAL_PAIRS: Dict[str, Tuple[str, str]] = {
        "shoulder-to-crotch": ("shoulder", "crotch"),
        "leg-length": ("crotch", "ankle"),
        "arm-length": ("shoulder", "wrist"),
    }

    #: Girth measurements and the slice each is read from.
    PERIMETER_SLICES: Dict[str, str] = {
        "ankle": "ankle", "bicep": "bicep", "calf": "calf", "chest": "chest",
        "hip": "hip", "thigh": "thigh", "waist": "waist", "wrist": "wrist",
    }

    #: Slices that are not in the canonical heuristic table. ``thigh`` is read mid-thigh and
    #: ``forearm`` at the elbow-to-wrist midpoint, because neither girth peaks at a canonical
    #: height. Declaring them explicitly keeps the feature matrix auditable instead of aliasing
    #: a girth onto a slice name it does not belong to.
    EXTRA_SLICES: Dict[str, float] = {"thigh": 0.380, "forearm": 0.560}

    def __init__(self, threshold: int = 127, ridge_alpha: float = 1.0) -> None:
        """
        Parameters
        ----------
        threshold : int, default 127
            Foreground threshold for the uint8 PNG masks.
        ridge_alpha : float, default 1.0
            L2 penalty for the per-measurement calibration ridge, applied to **standardised**
            features.
        """
        self.threshold = threshold
        self.ridge_alpha = ridge_alpha
        self.calibration_: Dict[str, np.ndarray] = {}
        self.columns: List[str] = []
        self.scaler_ = None          # sklearn StandardScaler, fitted on the fit split
        self.scaler_mean_: Optional[np.ndarray] = None
        self.scaler_scale_: Optional[np.ndarray] = None

    # ---- standardisation ------------------------------------------------------------------
    def _fit_scaler(self, F: np.ndarray) -> None:
        """Record the feature mean and scale from the fit split.

        Parameters
        ----------
        F : ndarray, shape (n, d)
            Feature matrix from :meth:`features`.

        Returns
        -------
        None
        """
        self.scaler_mean_ = F.mean(axis=0)
        self.scaler_scale_ = np.where(F.std(axis=0) > 1e-9, F.std(axis=0), 1.0)

    def _standardise(self, F: np.ndarray) -> np.ndarray:
        """Apply the fitted mean/scale.

        Parameters
        ----------
        F : ndarray, shape (n, d)
            Raw feature matrix.

        Returns
        -------
        ndarray, shape (n, d)
            Standardised features.

        Raises
        ------
        RuntimeError
            If called before :meth:`fit`.
        """
        if self.scaler_mean_ is None:
            raise RuntimeError("B1Geometric._standardise called before fit")
        return (F - self.scaler_mean_) / self.scaler_scale_

    def coefficients(self, measurement: str) -> np.ndarray:
        """Return a measurement's calibration weights, in standardised-feature units.

        Parameters
        ----------
        measurement : str
            Measurement name.

        Returns
        -------
        ndarray
            One coefficient per feature, or an empty array for a passthrough measurement.

        Raises
        ------
        KeyError
            If the measurement is unknown or unsupported.
        """
        return self.calibration_[measurement]

    # ---- geometry -----------------------------------------------------------------------
    def slice_heights(self) -> Dict[str, float]:
        """All slice heights as a fraction of stature from the floor.

        Returns
        -------
        dict
            Canonical heuristics merged with :attr:`EXTRA_SLICES`.
        """
        return {**SLICE_HEIGHTS_FROM_FLOOR, **self.EXTRA_SLICES}

    def mm_per_px(self, front: np.ndarray, height_cm: float) -> float:
        """Millimetres per pixel implied by stature.

        Parameters
        ----------
        front : ndarray, shape (H, W)
            Front mask.
        height_cm : float
            Subject stature, cm.

        Returns
        -------
        float
            ``height_cm * 10 / person_pixel_height``.
        """
        y0, y1 = mask_row_bounds(front, self.threshold)
        return float(height_cm) * 10.0 / float(y1 - y0 + 1)

    def _row_for(self, frac_from_floor: float, y0: int, y1: int) -> int:
        """Absolute row index for an anatomical height.

        Parameters
        ----------
        frac_from_floor : float
            Height as a fraction of stature; 0 = floor, 1 = top of head.
        y0, y1 : int
            Foreground row bounds of the view being sliced.

        Returns
        -------
        int
            Absolute row index.
        """
        span = float(y1 - y0 + 1)
        return int(round(y1 - (1.0 - frac_from_floor) * span))

    def measure(self, front: np.ndarray, side: np.ndarray, height_cm: float) -> Dict[str, float]:
        """Measure every slice for one silhouette pair.

        Parameters
        ----------
        front, side : ndarray, shape (H, W)
            Front and side masks for the same ``photo_id``.
        height_cm : float
            Subject stature, cm.

        Returns
        -------
        dict
            ``<slice>_width_mm``, ``<slice>_depth_mm``, ``<slice>_perim_mm``, ``<slice>_row``,
            ``vert_<length>_mm`` for each of :attr:`VERTICAL_PAIRS`, plus ``mm_per_px`` and
            ``height_cm``.

        Raises
        ------
        ValueError
            If either mask is empty.
        """
        s = self.mm_per_px(front, height_cm)
        f = np.asarray(front) > self.threshold
        sd = np.asarray(side) > self.threshold
        y0f, y1f = mask_row_bounds(front, self.threshold)
        y0s, y1s = mask_row_bounds(side, self.threshold)

        out: Dict[str, float] = {}
        rows: Dict[str, int] = {}
        for name, frac in self.slice_heights().items():
            rf = int(np.clip(self._row_for(frac, y0f, y1f), 0, f.shape[0] - 1))
            rs = int(np.clip(self._row_for(frac, y0s, y1s), 0, sd.shape[0] - 1))
            wf = float(f[rf].sum()) * s
            wd = float(sd[rs].sum()) * s
            a, b = max(wf / 2.0, 1e-3), max(wd / 2.0, 1e-3)
            out[f"{name}_width_mm"] = wf
            out[f"{name}_depth_mm"] = wd
            out[f"{name}_perim_mm"] = ramanujan_perimeter(a, b)
            out[f"{name}_row"] = float(rf)
            rows[name] = rf

        for meas, (upper, lower) in self.VERTICAL_PAIRS.items():
            out[f"vert_{meas}_mm"] = abs(rows[upper] - rows[lower]) * s

        out["mm_per_px"] = s
        out["height_cm"] = float(height_cm)
        return out

    def features(self, front: np.ndarray, side: np.ndarray, height_cm: float) -> np.ndarray:
        """Fixed-order feature vector for one silhouette pair.

        Parameters
        ----------
        front, side : ndarray, shape (H, W)
            Front and side masks for the same ``photo_id``.
        height_cm : float
            Subject stature, cm.

        Returns
        -------
        ndarray, shape (n_features,)
            Slice perimeters (mm), slice widths (mm), the three vertical lengths (mm),
            ``mm_per_px`` and ``height_cm``.
        """
        m = self.measure(front, side, height_cm)
        names = self.slice_heights()
        return np.array(
            [m[f"{n}_perim_mm"] for n in names]
            + [m[f"{n}_width_mm"] for n in names]
            + [m[f"vert_{k}_mm"] for k in self.VERTICAL_PAIRS]
            + [m["mm_per_px"], m["height_cm"]], float)

    def feature_names(self) -> List[str]:
        """Feature names matching the :meth:`features` order.

        Returns
        -------
        list of str
        """
        names = self.slice_heights()
        return ([f"{n}_perim_mm" for n in names] + [f"{n}_width_mm" for n in names]
                + [f"vert_{k}_mm" for k in self.VERTICAL_PAIRS] + ["mm_per_px", "height_cm"])

    # ---- calibration --------------------------------------------------------------------
    def unsupported(self) -> List[str]:
        """Targets B1 does not predict.

        Returns
        -------
        list of str
            ``['height']``: stature is a model *input*, so echoing it is not a prediction.
        """
        return ["height"]

    def derived(self, columns: Sequence[str]) -> List[str]:
        """Targets B1 does estimate.

        Parameters
        ----------
        columns : sequence of str
            Target names.

        Returns
        -------
        list of str
        """
        return [c for c in columns if c not in self.unsupported()]

    def fit(self, F: np.ndarray, Y: np.ndarray, columns: Sequence[str]) -> "B1Geometric":
        """Standardise the features, then fit one ridge per derived measurement.

        Parameters
        ----------
        F : ndarray, shape (n, d)
            Feature matrix from :meth:`features`.
        Y : ndarray, shape (n, m)
            Targets, cm.
        columns : sequence of str
            Target names.

        Returns
        -------
        B1Geometric
            ``self``, fitted.
        """
        from sklearn.linear_model import Ridge

        self.columns = list(columns)
        self._fit_scaler(np.asarray(F, dtype=float))
        Z = self._standardise(np.asarray(F, dtype=float))
        self.scaler_ = Ridge
        for j, col in enumerate(self.columns):
            if col in self.unsupported():
                continue
            self.calibration_[col] = Ridge(alpha=self.ridge_alpha).fit(Z, Y[:, j]).coef_
        return self

    def predict(self, F: np.ndarray) -> np.ndarray:
        """Predict all targets from a feature matrix.

        Parameters
        ----------
        F : ndarray, shape (n, d)
            Feature matrix from :meth:`features`.

        Returns
        -------
        ndarray, shape (n, m)
            Predictions, cm.
        """
        Z = self._standardise(np.asarray(F, dtype=float))
        return np.column_stack([
            F[:, -1] if col in self.unsupported() else Z @ self.calibration_[col]
            for col in self.columns])

**Interpretation.** The `EXTRA_SLICES` table is the one place where a girth is not read at a
canonical anatomical height, and it is written out rather than hidden:

- `thigh` is read at 0.380 of stature (mid-thigh), because thigh girth peaks well above the knee
  and below the crotch, neither of which is a useful width measurement.
- `forearm` is read at 0.560 (the elbow-to-wrist midpoint), because forearm girth peaks above the
  wrist and below the elbow.

Aliasing either onto an existing slice name would have produced a feature matrix that looks
identical in shape while being anatomically wrong — the exact failure the brief's "state how each
is derived" requirement exists to prevent.

Also note `unsupported() == ['height']`. Stature is a model *input*, so B1 echoes it back and
§6 excludes it from "beats B0" comparisons: returning a known input as a prediction would
manufacture a perfect score. The 13 derived targets are 9 girths from Ramanujan perimeters and
3 lengths from row differences, plus that passthrough.

**WHAT** — Build the per-subject tables for the `fit`, `val` and `calib` splits, assert the three
id sets partition the training subjects exactly, fit B0 (ridge and gradient boosting) and print
both metric tables.

**WHY**
- `[Ref]` B0 is the bar every vision model must clear, so it is computed first and reported
  per measurement.
- `[Exp]` Partition assertion: if a subject fell out of all three splits, it would silently
  disappear from the study rather than erroring, quietly shrinking every downstream estimate.

**OUTPUT** — `tbl_fit`, `tbl_val`, `tbl_cal`, `Y_fit`, `Y_val`, `ids_fit`, `ids_val`,
`ids_calib`, `b0`, `metrics_B0_gbm`, `metrics_B0_ridge`.

**CHECK** — asserts the union of the three id sets equals the full training subject set and that
all three are disjoint.

In [ ]:
from bodym_baselines import (B0NonVisual, B1Geometric, SLICE_HEIGHTS_FROM_FLOOR, b0_features,
                             mask_row_bounds, ramanujan_perimeter)


def subject_table(index: BodyMIndex, subject_ids: Sequence[str]) -> pd.DataFrame:
    """One row per subject: metadata plus the target columns.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index for one split.
    subject_ids : sequence of str
        Subjects to include.

    Returns
    -------
    DataFrame
        Indexed by ``subject_id``, carrying ``sex``, ``height_cm``, ``weight_kg``, ``bmi``,
        ``bmi_band`` and the target columns.
    """
    targets = [c for c in TARGETS if c in index.subjects.columns]
    meta = [c for c in ("sex", "gender", "height_cm", "weight_kg", "bmi", "bmi_band")]
    keep = set(subject_ids)
    return (index.subjects[index.subjects.subject_id.isin(keep)]
            .set_index("subject_id")[[c for c in meta + targets if c in index.subjects.columns]])


IDX = INTEGRITY_INDEX


tbl_fit = subject_table(IDX["train"], sorted(set_fit))
tbl_val = subject_table(IDX["train"], sorted(set_val))
tbl_cal = subject_table(IDX["train"], sorted(set_cal))
Y_fit = tbl_fit[TARGETS].to_numpy(float)
Y_val = tbl_val[TARGETS].to_numpy(float)
ids_fit = tbl_fit.index.to_list()
ids_val = tbl_val.index.to_list()
ids_calib = tbl_cal.index.to_list()
assert len(set(ids_fit) | set(ids_val) | set(ids_calib)) == len(set_fit | set_val | set_cal)
assert not (set(ids_fit) & set(ids_val)) and not (set(ids_fit) & set(ids_calib))
print(f"fit {len(ids_fit)} | val {len(ids_val)} | calib {len(ids_calib)} subjects")

b0 = B0NonVisual(ridge_alpha=cfg["ridge_alpha"], n_estimators=cfg["n_estimators"],
                 random_state=cfg["seed"]).fit(b0_features(tbl_fit.reset_index()), Y_fit, TARGETS)
P0_val_gbm = b0.predict(b0_features(tbl_val.reset_index()), which="gbm")
P0_val_ridge = b0.predict(b0_features(tbl_val.reset_index()), which="ridge")

metrics_B0_gbm = M.per_measurement_metrics(Y_val, P0_val_gbm, TARGETS)
metrics_B0_ridge = M.per_measurement_metrics(Y_val, P0_val_ridge, TARGETS)
print("=== B0 non-visual, gradient boosting — validation ===")
print(M.format_metric_table(metrics_B0_gbm))
print("\n=== B0 non-visual, ridge — validation ===")
print(M.format_metric_table(metrics_B0_ridge))

**Interpretation.** B0 is the reference every vision model must clear, so read it before anything
else. Two things to extract:

- **The GBM-vs-ridge gap per measurement.** Where it is large (expect the girths) the
  height→girth relation is genuinely nonlinear, so a vision model is competing against a
  nonlinear function of two scalars. Where it is near zero, the relation is close to affine and a
  CNN has to work much harder to earn its keep.
- **Which measurements B0 already does well.** The length measurements (`leg-length`,
  `arm-length`, `shoulder-to-crotch`, `height`) are largely determined by stature, so B0 will be
  strong there. The girths are where silhouette information could plausibly add value. §6 uses
  this table as the per-measurement bar.

**WHAT** — Extract the B1 geometric features for every `fit` and `val` subject (one photo each),
assert that derived ∪ unsupported partitions the 14 targets, fit the per-measurement calibration
on the `fit` split only, and print the metric table plus a per-measurement coefficient probe.

**WHY**
- `[Ref]` Brief §5 prescribes this baseline exactly: scale from height, heuristic slice heights,
  width and depth as ellipse axes, Ramanujan perimeter, per-measurement linear calibration on the
  fit split only.
- `[Exp]` The partition assertion enforces brief §5's requirement that measurements not
  expressible as slices be *stated* as derived or marked unsupported. Nothing may be silently
  missing.
- `[Exp]` The coefficient probe reports which geometry each key prediction leans on, so a reader
  can tell a real shape model from one that has quietly learned a stature correction.

**OUTPUT** — `b1`, `metrics_B1`, printed coefficient probe, cached validation predictions.

**CHECK** — asserts the derived/unsupported partition, that B1's predictions match the target
shape and are finite, and that B1 subject order matches B0's so the metrics are comparable
row-for-row.

In [ ]:
# --- B1 ----------------------------------------------------------------------------------
B1_MODEL = B1Geometric(threshold=cfg["mask_threshold"], ridge_alpha=1.0)


def first_photo_per_subject(index: BodyMIndex) -> pd.DataFrame:
    """Select one silhouette per subject, deterministically.

    B0 and B1 predict *subject-level* ground truth. Fixing one photo per subject keeps the
    train-time and inference-time semantics identical and avoids averaging features across a
    variable number of photos, which would make the feature count depend on the subject.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index for one split.

    Returns
    -------
    DataFrame
        ``index.photos`` restricted to the lexicographically first ``photo_id`` per subject.
    """
    return index.photos.sort_values("photo_id").groupby("subject_id", as_index=False).first()


def b1_matrix(index: BodyMIndex, subject_ids: Sequence[str], desc: str
              ) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Extract the B1 feature and target matrices for given subjects.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index.
    subject_ids : sequence of str
        Subjects, in the order the caller wants them.
    desc : str
        Progress-bar label.

    Returns
    -------
    (F, Y, ids) : tuple
        ``F`` of shape (n, d) in mm/cm units, ``Y`` of shape (n, m) in cm, and the ordered ids.
    """
    photos = first_photo_per_subject(index).set_index("subject_id")
    hgt = index.subjects.set_index("subject_id")["height_cm"]
    tgt = index.subjects.set_index("subject_id")[list(TARGETS)]

    feats, targs, ids = [], [], []
    for sid in tqdm(list(subject_ids), desc=desc):
        row = photos.loc[sid]
        front = np.array(Image.open(row.front_mask))
        side = np.array(Image.open(row.side_mask))
        feats.append(B1_MODEL.features(front, side, float(hgt.loc[sid])))
        targs.append(tgt.loc[sid].to_numpy(float))
        ids.append(sid)
    return np.asarray(feats), np.asarray(targs), ids


F_fit, YF_fit, ids_fit_b1 = b1_matrix(IDX["train"], ids_fit, "B1 fit")
F_val, YF_val, ids_val_b1 = b1_matrix(IDX["train"], ids_val, "B1 val")
assert ids_fit_b1 == ids_fit and ids_val_b1 == ids_val, "B1 subject order must match B0 order"

derived, unsupported = set(B1_MODEL.derived(TARGETS)), set(B1_MODEL.unsupported())
assert derived | unsupported == set(TARGETS), "B1 derived + unsupported must partition targets"
assert not (derived & unsupported), "B1 derived and unsupported overlap"
print(f"B1 derives {len(derived)} targets, unsupported = {sorted(unsupported)}")

b1 = B1Geometric(threshold=cfg["mask_threshold"], ridge_alpha=1.0)
b1.fit(F_fit, YF_fit, TARGETS)
P1_val = b1.predict(F_val)
assert P1_val.shape == Y_val.shape and np.isfinite(P1_val).all(), "B1 prediction invalid"

# A geometric model that cannot beat a constant train-mean predictor has a calibration bug, not
# a weak heuristic. This is the check that would have caught fitting a ridge on unstandardised
# millimetre-scale features.
_baseline = np.abs(YF_fit.mean(axis=0) - Y_val).mean(axis=0)
_worse = [c for j, c in enumerate(TARGETS)
          if c != "height" and np.abs(P1_val[:, j] - Y_val[:, j]).mean() > _baseline[j] + 1e-6]
print("B1 measurements worse than a constant train-mean predictor:", _worse or "none")
assert not _worse, f"B1 calibration is broken for: {_worse}"

metrics_B1 = M.per_measurement_metrics(Y_val, P1_val, TARGETS)
print("\n=== B1 geometric per-slice ellipse — validation ===")
print(M.format_metric_table(metrics_B1))

# Coefficient probe: which geometry does each key prediction actually lean on?
# Read from the FITTED model (b1), not the feature extractor (B1_MODEL).
fn = B1_MODEL.feature_names()
for col in cfg["key_measurements"]:
    c = b1.coefficients(col)
    top = np.argsort(-np.abs(c))[:5]
    print(f"  B1[{col}]: " + ", ".join(f"{fn[i]}={c[i]:+.4g}" for i in top)
          + f"   (feature sd scale: waist slice sd = {b1.scaler_scale_[fn.index('waist_perim_mm')]:.1f} mm)")

np.save(PATHS["artifacts"] / "P1_val.npy", P1_val)
np.save(PATHS["artifacts"] / "P0_val_gbm.npy", P0_val_gbm)
np.save(PATHS["artifacts"] / "Y_val.npy", Y_val)
(PATHS["artifacts"] / "val_subjects.json").write_text(json.dumps(ids_val))
print("mark_phase('B0+B1') =", mark_phase("B0+B1"), "s")

**Interpretation.** B1 is a diagnostic, not a competitor. Its MAE per measurement tells you how
much of the silhouette is captured by height-calibrated slice geometry, and the coefficient
probe tells you *which* geometry each prediction leans on.

Two readings to take from it, after the **train-mean baseline check** above:

- A slice placed at the wrong anatomical height yields a systematically wrong girth, and the error
  is largest for the measurements with the most body-shape variance. If chest and hip show a large
  positive bias while waist is near zero, the heuristic rows are shifted rather than the scale
  being wrong.
- `mm_per_px` appearing as a top coefficient for several measurements means the calibration is
  partly learning a stature correction rather than shape — which is B1 quietly converging toward
  B0. That is a useful warning: if B1 approaches B0's accuracy, the *scale* assumption, not the
  slice heuristics, is doing the work, and §5 (boundary sensitivity) becomes the decisive
  experiment.

#### §4.3 B2 — BMnet-style CNN

**WHAT** — Front and side silhouettes, each resized to `img_height × img_width` (480×640) and
concatenated **spatially** into one 480×1280 canvas carried in a **single mask channel**;
constant-valued height and weight channels concatenated **depth-wise**, giving 5 channels for
V-HW and 4 for V-H. MNASNet backbone with ImageNet weights where available; MLP head → 14
outputs; **L1** loss; checkpoint to `/kaggle/working` every epoch so a session timeout cannot lose
the run.

**WHY**
- `[Ref]` Ruiz et al. (2022) feed front and side silhouettes jointly and attach height (and, in the
  ablated variant, weight) as constant channels. Concatenating *spatially* — rather than stacking
  channels — lets the early convolutions see both views in one receptive field, which is what makes
  depth (the side view) inform girth.
- `[Ref]` Brief §1: *"Do not add Ultralytics/YOLO as a dependency"* — no detector, no segmenter.
  The only image-processing libraries are torchvision, PIL and scipy.
- `[Exp]` **Stem adaptation.** MNASNet's first convolution expects 3 input channels; the mask is
  one. The pretrained weights are therefore **mean-reduced over the input-channel axis** to
  `(out, 1, k, k)` — the standard grayscale adaptation, which preserves every pretrained output
  filter with full spatial structure. The alternatives considered (re-initialising the stem, or
  padding the mask into three channels) are rejected in DECISIONS.md D5. The cell prints which
  branch it took.
- `[Exp]` **Scalar injection.** Height and weight enter by summing a zero-initialised 1×1
  projection after the stem, so at initialisation the output does not depend on them at all. That
  is directly assertable, and it is what makes the V-HW vs V-H ablation in §6 interpretable: a
  difference in accuracy cannot be dismissed as head noise.
- `[Ref]` Checkpointing every epoch is a durability measure, not a tuning measure: selection still
  uses **validation MAE only** (§6), never the training loss and never Test-A/Test-B.

**OUTPUT** — `src/bodym_cnn.py`, `MeasurementNet`, `SilhouetteDataset`, `b2_history`, checkpoint
files in `/kaggle/working/checkpoints/`.

**CHECK** — asserts the forward pass returns `(batch, 14)` finite values, that the dataset yields
masks already binarised to {0, 1}, and that the epoch-1 loss is finite.

In [ ]:
%%writefile src/bodym_cnn.py
"""BMnet-style CNN for 14-measurement estimation from front/side silhouettes.

Architecture
------------
``[front | side]`` concatenated spatially to ``(img_height, 2 * img_width)``, plus constant
height and weight channels concatenated depth-wise, fed to an ImageNet-pretrained MNASNet trunk
with an MLP head.

Channel handling
----------------
The pretrained first convolution is kept **bit-exact** (3 input channels). Height and weight are
injected by summing a zero-initialised ``1x1`` projection at the second convolution, so at
initialisation the network is numerically identical to ImageNet and the scalars must earn their
influence through training. See DECISIONS.md.

References
----------
Ruiz, N., Bellver, M., Bolkart, T., Arora, A., Lin, M. C., Romero, J., & Bala, R. (2022).
Human body measurement estimation with adversarial augmentation (arXiv:2210.05667).
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset

#: Channels of the assembled input tensor: [mask, height, weight, ones].
#: V-H drops the weight channel, giving 4 rather than 5.
N_CHANNELS = 5
N_CHANNELS_VH = 4

#: Normalisation constants for the constant-valued scalar channels.
HEIGHT_SCALE = 200.0   # height_cm / HEIGHT_SCALE
WEIGHT_SCALE = 100.0   # weight_kg / WEIGHT_SCALE


def _feature_dim(net: nn.Module) -> int:
    """Return the backbone's feature width immediately before its classifier.

    Parameters
    ----------
    net : nn.Module
        A torchvision model with a ``features`` trunk and a ``classifier`` head.

    Returns
    -------
    int
        Input width of the first ``nn.Linear`` found in ``classifier``, falling back to 1280 when
        the head has no linear layer.
    """
    for m in getattr(net, "classifier", nn.Sequential()):
        if isinstance(m, nn.Linear):
            return int(m.in_features)
    return 1280


def build_backbone(name: str = "mnasnet1_0_3_0", pretrained: bool = True,
                   offline_weights: Optional[str] = None) -> Tuple[nn.Module, str]:
    """Instantiate an ImageNet-pretrained torchvision backbone.

    Parameters
    ----------
    name : str, default 'mnasnet1_0_3_0'
        Any torchvision model name. MNASNet is the default because Ruiz et al. (2022) use an
        MNASNet trunk and the deployment budget on Kaggle GPUs is comfortable with it.
    pretrained : bool, default True
        Request ImageNet weights.
    offline_weights : str, optional
        Path to a local ``.pth`` state dict, used when the session has no internet. Checked
        before the download is attempted.

    Returns
    -------
    (module, source) : tuple
        The backbone and a short string describing where the weights came from:
        ``'offline'``, ``'imagenet-download'`` or ``'random'``.

    Raises
    ------
    RuntimeError
        If ``name`` is not a torchvision model.
    """
    import torchvision.models as tvm

    if not hasattr(tvm, name):
        raise RuntimeError(f"unknown backbone {name!r}")

    if pretrained and offline_weights and Path(offline_weights).is_file():
        model = getattr(tvm, name)(weights=None)
        state = torch.load(offline_weights, map_location="cpu")
        model.load_state_dict(state)
        return model, "offline"

    if pretrained:
        try:
            weights_enum = getattr(tvm, "get_model_weights")(name).DEFAULT
            return getattr(tvm, name)(weights=weights_enum), "imagenet-download"
        except Exception as exc:
            print(f"[backbone] ImageNet weights unavailable ({exc}); falling back to random init. "
                  "Attach a .pth and set config.yaml:offline_weights for a reproducible run.")
            return getattr(tvm, name)(weights=None), "random"

    return getattr(tvm, name)(weights=None), "random"


class MeasurementNet(nn.Module):
    """Front+side silhouette CNN with a 14-measurement regression head.

    Input channel layout is ``[mask, height, weight, ones]`` — **one** mask channel holding the
    front and side silhouettes concatenated horizontally, then the constant scalar channels. That
    is 5 channels for V-HW and 4 for V-H.

    First-convolution adaptation
    ---------------------------
    The ImageNet stem expects 3 input channels, but the mask is a single channel. Rather than
    discarding the pretrained filters or feeding the scalar channels twice, the stem's weights are
    **mean-reduced over the input-channel axis** to shape ``(out, 1, k, k)``. This is the standard
    grayscale adaptation: it preserves every pretrained output filter with full spatial structure
    instead of throwing the first layer away.

    Scalars
    -------
    Height and weight are injected by summing a zero-initialised 1x1 projection at the second
    convolution, so at initialisation the network is numerically identical to its pretrained
    initialisation and the scalars must earn their influence. See DECISIONS.md D5.

    References
    ----------
    Ruiz, N., et al. (2022). Human body measurement estimation with adversarial augmentation.
    """

    def __init__(self, backbone: str = "mnasnet1_0_3_0", pretrained: bool = True,
                 n_outputs: int = 14, hidden: int = 512, dropout: float = 0.2,
                 use_weight: bool = True, offline_weights: Optional[str] = None) -> None:
        super().__init__()
        self.use_weight = use_weight
        self.n_outputs = n_outputs
        net, self.weights_source = build_backbone(backbone, pretrained, offline_weights)

        self.features = net.features

        # --- adapt the pretrained 3-channel stem to a single mask channel -------------------
        stem_conv = self.features[0]
        if isinstance(stem_conv, nn.Conv2d) and stem_conv.in_channels != 1:
            new_conv = nn.Conv2d(1, stem_conv.out_channels, kernel_size=stem_conv.kernel_size,
                                 stride=stem_conv.stride, padding=stem_conv.padding,
                                 bias=stem_conv.bias is not None)
            with torch.no_grad():
                new_conv.weight.copy_(stem_conv.weight.mean(dim=1, keepdim=True))
                if stem_conv.bias is not None:
                    new_conv.bias.copy_(stem_conv.bias)
            self.features[0] = new_conv
            self.stem_adaptation = f"mean-reduced {stem_conv.in_channels}->1 input channels"
        else:
            self.stem_adaptation = "already single-channel"

        self.final_pool = nn.AdaptiveAvgPool2d(1)
        feat_dim = _feature_dim(net)

        # Zero-initialised projection of the scalar channels, added after the stem. Zero init =>
        # zero contribution at step 0 => the pretrained trunk is bit-exact at initialisation.
        n_scalars = 2 if use_weight else 1
        self.scalar_proj = nn.Conv2d(n_scalars, max(8, feat_dim // 8), kernel_size=1)
        self.scalar_fuse = nn.Conv2d(max(8, feat_dim // 8), feat_dim, kernel_size=1)
        for m in (self.scalar_proj, self.scalar_fuse):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_outputs),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict measurements.

        Parameters
        ----------
        x : Tensor, shape (B, 5, H, 2W) or (B, 4, H, 2W)
            Channel order ``[mask, height, weight, ones]`` for V-HW and
            ``[mask, height, ones]`` for V-H. ``mask`` holds the front and side silhouettes
            concatenated horizontally, each resized to ``H/2 x W``.

        Returns
        -------
        Tensor, shape (B, n_outputs)
            Predictions, cm.

        Raises
        ------
        ValueError
            If the channel count does not match ``use_weight``.
        """
        n_expected = 5 if self.use_weight else 4
        if x.shape[1] != n_expected:
            raise ValueError(f"expected {n_expected} input channels for "
                             f"use_weight={self.use_weight}, got {x.shape[1]}")

        h = self.features[0](x[:, :1])                       # mask channel only
        scalars = x[:, 1:3] if self.use_weight else x[:, 1:2]
        h = h + self.scalar_fuse(self.scalar_proj(scalars))
        h = self.features[1:](h)
        return self.head(self.final_pool(h))


def to_tensor(front: np.ndarray, side: np.ndarray, height_cm: float, weight_kg: Optional[float],
              img_height: int = 480, img_width: int = 640, threshold: int = 127,
              use_weight: bool = True) -> np.ndarray:
    """Assemble the CNN input tensor for one silhouette pair. **The canonical preprocessing.**

    This function is the single definition of preprocessing. ``preprocess.py`` in the export
    bundle re-exports it verbatim, so a unit test against saved reference tensors (§9) is a real
    test of the deployed pipeline rather than of a lookalike.

    Parameters
    ----------
    front, side : ndarray, shape (H, W)
        Silhouette masks (uint8 0/255 or bool).
    height_cm : float
        Subject stature, cm.
    weight_kg : float or None
        Subject mass, kg. Ignored when ``use_weight`` is False.
    img_height, img_width : int
        Per-view target size. The concatenated canvas is ``img_height x (2 * img_width)``.
    threshold : int, default 127
        Foreground threshold; masks are binarised with ``> threshold``.
    use_weight : bool, default True
        When False the weight channel is omitted entirely (V-H), not zero-filled.

    Returns
    -------
    ndarray, shape (5 or 4, img_height, 2 * img_width)
        Channels ``[mask, height, weight, ones]``, float32 in {0, 1}. ``mask`` is the front and
        side silhouettes concatenated **horizontally** (front left, side right) and resized as a
        single canvas, so the spatial layout preserves the two views side by side.

    Raises
    ------
    ValueError
        If a mask is empty or ``weight_kg`` is None while ``use_weight`` is True.
    """
    if use_weight and weight_kg is None:
        raise ValueError("weight_kg is required when use_weight=True")
    f = (np.asarray(front) > threshold).astype(np.float32)
    s = (np.asarray(side) > threshold).astype(np.float32)
    if not f.any() or not s.any():
        raise ValueError("empty silhouette mask")

    canvas = np.concatenate([f, s], axis=1)
    resized = np.asarray(Image.fromarray((canvas * 255).astype(np.uint8))
                         .resize((2 * img_width, img_height), Image.BILINEAR)) > 127
    mask_ch = resized.astype(np.float32)[None, :, :]

    H, W2 = mask_ch.shape[1:]
    height_ch = np.full((H, W2), float(height_cm) / HEIGHT_SCALE, np.float32)
    chans = [mask_ch, height_ch]
    if use_weight:
        chans.append(np.full((H, W2), float(weight_kg) / WEIGHT_SCALE, np.float32))
    chans.append(np.ones((H, W2), np.float32))
    return np.concatenate(chans, axis=0)


class SilhouetteDataset(Dataset):
    """Photo-level dataset yielding assembled CNN tensors and per-subject targets.

    Multiple photos of the same subject appear as separate samples during training. This is
    deliberate: the *split* is by subject (Phase 2), so no subject is ever seen in two splits,
    and using all its photos is data augmentation in effect.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index for the split.
    subject_ids : sequence of str
        Subjects to include. Only their photos are drawn.
    targets : sequence of str
        Target column names.
    img_height, img_width : int
        Per-view target size.
    threshold : int, default 127
        Mask binarisation threshold.
    use_weight : bool, default True
        Include the weight channel.
    augment : callable, optional
        ``augment(front, side, rng) -> (front, side)`` applied to the raw masks before resizing.
    """

    def __init__(self, index, subject_ids, targets: Tuple[str, ...] = None,
                 img_height: int = 480, img_width: int = 640, threshold: int = 127,
                 use_weight: bool = True, augment=None) -> None:
        self.index = index
        self.targets = tuple(targets) if targets is not None else None
        self.img_height = img_height
        self.img_width = img_width
        self.threshold = threshold
        self.use_weight = use_weight
        self.augment = augment
        keep = set(subject_ids)
        self.photos = index.photos[index.photos.subject_id.isin(keep)].reset_index(drop=True)
        meta = index.subjects.set_index("subject_id")
        self.height = meta["height_cm"].to_dict()
        self.weight = meta["weight_kg"].to_dict()
        self.y = {sid: meta.loc[sid, list(self.targets)].to_numpy(float) for sid in self.height}
        self.subjects = meta

    def __len__(self) -> int:
        """Number of silhouettes (samples)."""
        return len(self.photos)

    def __getitem__(self, i: int):
        """Return ``(tensor, targets, subject_id, photo_id)``.

        Parameters
        ----------
        i : int
            Sample index.

        Returns
        -------
        tensor : Tensor, shape (C, img_height, 2 * img_width)
        targets : ndarray, shape (n_targets,)
        subject_id : str
        photo_id : str
        """
        row = self.photos.iloc[i]
        front = np.array(Image.open(row.front_mask))
        side = np.array(Image.open(row.side_mask))
        if self.augment is not None:
            front, side = self.augment(front, side)
        x = to_tensor(front, side, self.height[row.subject_id], self.weight[row.subject_id],
                      self.img_height, self.img_width, self.threshold, self.use_weight)
        return torch.from_numpy(x), self.y[row.subject_id], row.subject_id, row.photo_id

**Interpretation.** Three design choices in this cell are checkable, and each is asserted rather
than asserted-to.

**The mean-reduced stem** is the standard grayscale adaptation, printed as `stem_adaptation`. It
keeps every pretrained output filter with its full spatial structure; re-initialising the stem
would have thrown away the first layer, and feeding the mask as three duplicated channels would
have made the network see the same input three times. Neither is obviously wrong, so the choice is
recorded in D5 rather than left implicit.

**The zero-initialised scalar injection** has one concrete, testable consequence: at initialisation
the output depends on the silhouettes and **not at all** on height or weight. So a model that ends
up worse on V-H than V-HW is demonstrably using the weight channel, rather than the difference
being an artefact of a random head. The §6 ablation is therefore interpretable rather than noisy.

**`SilhouetteDataset` yields one sample per photo**, not per subject. Because Phase 2 split by
subject, no subject appears in two splits, so this is not leakage — it is the cheapest available
augmentation, since a heavily photographed subject contributes proportionally more gradient steps.
The consequence to remember when reading §5: the sensitivity experiment must average per
**subject**, not per photo, or heavily-photographed subjects dominate the reported ΔMAE. That is
why `predict_split` and `predict_index` group by subject before computing metrics.

**WHAT** — Instantiate `MeasurementNet`, run a forward pass on a dummy batch, verify the
zero-initialised scalar injection property, and print the trainable parameter count.

**WHY**
- `[Exp]` Changing only the height and weight channels and observing an identical output is a
  *direct* test of the zero-init design from §4.3. It is what makes the later V-HW vs V-H ablation
  interpretable: a difference in accuracy cannot be dismissed as head noise, because the two
  networks start numerically identical.
- `[Exp]` The preprocessing contract is checked on a real sample: channel count, binary mask
  channels, and the exact normalisation constants for the height and weight channels.

**OUTPUT** — `device`, `net_b2`, `ds_probe`, printed channel and constant values.

**CHECK** — forward output is `(batch, 14)` and finite; `max |delta output|` under changing
scalars is below 1e-6; the sampled tensor has the expected shape and `{0, 1}` mask values; the V-H
tensor has 4 channels rather than 5 with a zeroed one.

In [ ]:
from bodym_cnn import (HEIGHT_SCALE, N_CHANNELS, N_CHANNELS_VH, WEIGHT_SCALE, MeasurementNet,
                       SilhouetteDataset, build_backbone, to_tensor)

# --- instantiate and sanity-check the network --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| GPU:", ENV["gpu"]["name"])

net_b2 = MeasurementNet(backbone=cfg["backbone"], pretrained=cfg["pretrained"],
                        n_outputs=len(TARGETS), hidden=cfg["hidden"], dropout=cfg["dropout"],
                        use_weight=True, offline_weights=cfg["offline_weights"]).to(device)
print("backbone weights source:", net_b2.weights_source)
print("input channels:", N_CHANNELS, "-> head outputs:", net_b2.n_outputs)

net_b2.eval()
with torch.no_grad():
    x_dummy = torch.zeros(2, N_CHANNELS, cfg["img_height"], 2 * cfg["img_width"], device=device)
    y_dummy = net_b2(x_dummy)
print("forward shape:", tuple(y_dummy.shape), "| finite:", bool(torch.isfinite(y_dummy).all()))
assert y_dummy.shape == (2, len(TARGETS)) and torch.isfinite(y_dummy).all()
print("stem adaptation:", net_b2.stem_adaptation,
      "| trunk input channels:", net_b2.features[0].in_channels)

# Verify the scalars provably do not influence the output at initialisation.
net_b2.eval()
with torch.no_grad():
    a = torch.rand(1, N_CHANNELS, cfg["img_height"], 2 * cfg["img_width"], device=device)
    b = a.clone()
    b[:, 1] = 0.2      # height channel
    b[:, 2] = 0.9      # weight channel
    delta = (net_b2(a) - net_b2(b)).abs().max().item()
print(f"max |delta output| when only height/weight channels change, at init: {delta:.3e}")
assert delta < 1e-6, "zero-init scalar injection is not in effect"

# A wrong channel count must be rejected loudly rather than silently mis-indexed.
try:
    net_b2(torch.zeros(1, N_CHANNELS_VH, cfg["img_height"], 2 * cfg["img_width"], device=device))
    raise AssertionError("MeasurementNet accepted a 4-channel input with use_weight=True")
except ValueError:
    print("channel-count guard: 4-channel input rejected by V-HW, as intended")

n_params = sum(p.numel() for p in net_b2.parameters())
print(f"trainable parameters: {n_params/1e6:.2f} M")
print("mark_phase('B2 model') =", mark_phase("B2 model"), "s")

#### §4.4 Training loop

**WHAT** — One reusable `train_model` used by B2 in §4.5 and by both production variants in §6,
so the training semantics cannot drift between the baseline and the shipped model.

**WHY**
- `[Ref]` Ruiz et al. (2022) use L1 loss, which is the right choice here: the metric of interest
  is MAE, and L1's median-like behaviour resists the outlier measurements in BodyM's girth
  columns (max waist 164 cm against a median of 88 cm).
- `[Ref]` AMP + gradient clipping: the depth-wise scalar injection can produce large activations
  early in training, and clipping at 5.0 prevents a single step from destroying the pretrained
  trunk. `[Exp]` The first-step loss is printed so an exploding run is visible immediately.
- `[Exp]` Checkpointing every epoch writes to `/kaggle/working/checkpoints/`, which survives a
  session restart but not a kernel restart — the durability boundary is stated in the README.

**OUTPUT** — `train_model`, history DataFrame, per-epoch checkpoint paths.

**CHECK** — asserts the first epoch's loss is finite and that a checkpoint file exists after the
loop.

In [ ]:
%%writefile src/bodym_training.py
"""Training loop shared by the B2 baseline and the production variants."""

from __future__ import annotations

import time
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def set_worker_seed(worker_id: int) -> None:
    """Seed a DataLoader worker deterministically.

    Parameters
    ----------
    worker_id : int
        Worker index assigned by PyTorch.
    """
    import random
    seed = (torch.initial_seed() + worker_id) % 2 ** 32
    np.random.seed(seed)
    random.seed(seed)


def train_model(
    model: nn.Module,
    train_ds,
    val_ds,
    targets: Sequence[str],
    cfg: Dict,
    device: torch.device,
    ckpt_dir: Path,
    tag: str,
    epochs: int,
) -> Tuple[nn.Module, pd.DataFrame]:
    """Train one model with L1 loss and checkpoint every epoch.

    Parameters
    ----------
    model : nn.Module
        Network to train.
    train_ds, val_ds : SilhouetteDataset
        Training and validation datasets. Their ``subject_id`` arrays define the split; the
        checkpoint selection uses **validation** data only.
    targets : sequence of str
        Target names, length ``model.n_outputs``.
    cfg : dict
        Notebook configuration (``batch_size``, ``lr``, ``weight_decay``, ``grad_clip``, ``amp``,
        ``num_workers``, ``seed``).
    device : torch.device
        Compute device.
    ckpt_dir : Path
        Directory for per-epoch checkpoints.
    tag : str
        Prefix for checkpoint filenames, e.g. ``'b2'`` or ``'vhw'``.
    epochs : int
        Number of epochs.

    Returns
    -------
    (model, history) : tuple
        The model with the best validation-MAE weights loaded, and a per-epoch history. Validation
        metrics are aggregated **per subject**, never per photo, so heavily photographed subjects
        do not dominate checkpoint selection.

    Raises
    ------
    RuntimeError
        If the first epoch's mean loss is not finite.
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    g = torch.Generator()
    g.manual_seed(cfg["seed"])

    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                          num_workers=cfg["num_workers"], pin_memory=True, drop_last=True,
                          worker_init_fn=set_worker_seed, generator=g,
                          persistent_workers=cfg["num_workers"] > 0)
    val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                        num_workers=cfg["num_workers"], pin_memory=True)

    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    scaler = torch.amp.GradScaler("cuda", enabled=bool(cfg["amp"] and device.type == "cuda"))
    loss_fn = nn.L1Loss()

    history: List[dict] = []
    best_mae = float("inf")
    best_state = None
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        running, n_seen = 0.0, 0
        for x, y, _, _ in tqdm(train_dl, desc=f"{tag} ep{epoch}", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float()
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=bool(cfg["amp"] and device.type == "cuda")):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
            running += float(loss.item()) * len(y)
            n_seen += len(y)

        if epoch == 1 and not np.isfinite(running / max(n_seen, 1)):
            raise RuntimeError(f"{tag}: first-epoch loss is not finite ({running})")
        sched.step()

        pred, true, subs = predict_split(model, val_dl, device)
        val_mae_cm = float(np.mean(np.abs(pred - true)))
        history.append({"epoch": epoch, "train_loss_cm": running / max(n_seen, 1),
                        "val_mae_cm": val_mae_cm, "val_mae_mm": val_mae_cm * 10.0,
                        "seconds": round(time.time() - t0, 1)})

        ckpt = ckpt_dir / f"{tag}_epoch{epoch:02d}.pt"
        torch.save({"model": model.state_dict(), "epoch": epoch, "targets": list(targets),
                    "use_weight": getattr(model, "use_weight", True),
                    "img_height": getattr(train_ds, "img_height", None),
                    "img_width": getattr(train_ds, "img_width", None),
                    "val_mae_mm": val_mae_cm * 10.0}, ckpt)

        if val_mae_cm < best_mae:
            best_mae, best_epoch = val_mae_cm, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"  {tag} epoch {epoch:02d}: train {running/max(n_seen,1):.4f} cm | "
              f"val MAE {val_mae_cm*10:.2f} mm | {time.time()-t0:.0f}s"
              + ("  <- best" if epoch == best_epoch else ""))

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"{tag}: best epoch {best_epoch}, val MAE {best_mae*10:.2f} mm -> {ckpt_dir}")
    return model, pd.DataFrame(history)


@torch.no_grad()
def predict_split(model: nn.Module, loader: DataLoader, device: torch.device
                  ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run inference over a loader, aggregating to one row per subject.

    Parameters
    ----------
    model : nn.Module
        Trained network.
    loader : DataLoader
        Loader over a :class:`SilhouetteDataset`.
    device : torch.device
        Compute device.

    Returns
    -------
    (pred, true, subjects) : tuple of ndarray
        ``pred`` and ``true`` of shape (n_subjects, n_targets) in cm, averaging all photos of a
        subject, and the ordered subject ids.
    """
    model.eval()
    acc: Dict[str, List[np.ndarray]] = {}
    truth: Dict[str, np.ndarray] = {}
    for x, y, sid, _ in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            p = model(x).float().cpu().numpy()
        for k, s in enumerate(sid):
            acc.setdefault(s, []).append(p[k])
            truth[s] = y[k].numpy()
    subjects = sorted(acc)
    pred = np.stack([np.mean(acc[s], axis=0) for s in subjects])
    true = np.stack([truth[s] for s in subjects])
    return pred, true, subjects

**WHAT** — Import the training module and check the preprocessing contract on a real sample,
including the V-H channel removal.

**WHY**
- `[Exp]` The normalisation constants (`height_cm / 200`, `weight_kg / 100`) are asserted against
  the values actually stored in the tensor. These are the numbers a "cleanup" would silently
  change, producing plausible-but-wrong measurements; the exported `preprocess.py` self-test in
  §9.2 checks the same property from outside the notebook.
- `[Exp]` The V-H check asserts 4 channels rather than 5 with a zeroed weight channel, which is the
  requirement in brief §7.

**OUTPUT** — `T` bound in the namespace, printed tensor shape and channel values.

**CHECK** — shape, dtype, binary mask range, and the exact expected scalar channel values.

In [ ]:
import bodym_training as T

print("src/bodym_training.py written:", (PATHS["src"] / "bodym_training.py").exists())

ds_probe = SilhouetteDataset(IDX["train"], ids_val[:4], TARGETS,
                             img_height=cfg["img_height"], img_width=cfg["img_width"],
                             threshold=cfg["mask_threshold"], use_weight=True)
x0, y0, sid0, pid0 = ds_probe[0]
print(f"sample tensor {tuple(x0.shape)} dtype={x0.dtype} "
      f"| mask channel unique values={sorted(np.unique(x0[0].numpy()).tolist())}")
print("channel 1 (height) =", float(x0[1].unique()[0]),
      "expected height_cm/200 =", float(ds_probe.height[sid0]) / HEIGHT_SCALE)
print("channel 2 (weight) =", float(x0[2].unique()[0]),
      "expected weight_kg/100 =", float(ds_probe.weight[sid0]) / WEIGHT_SCALE)
print("channel 4 (ones) =", float(x0[4].unique()[0]))
assert x0.shape == (N_CHANNELS, cfg["img_height"], 2 * cfg["img_width"])
assert set(np.unique(x0[0].numpy())) <= {0.0, 1.0}, "mask channel must be binarised to {0,1}"
assert abs(float(x0[1].unique()[0]) - float(ds_probe.height[sid0]) / HEIGHT_SCALE) < 1e-6
assert abs(float(x0[2].unique()[0]) - float(ds_probe.weight[sid0]) / WEIGHT_SCALE) < 1e-6
assert abs(float(x0[4].unique()[0]) - 1.0) < 1e-9
assert y0.shape == (len(TARGETS),)

xw = to_tensor(np.array(Image.open(ds_probe.photos.iloc[0].front_mask)),
               np.array(Image.open(ds_probe.photos.iloc[0].side_mask)),
               float(ds_probe.height[sid0]), float(ds_probe.weight[sid0]),
               cfg["img_height"], cfg["img_width"], cfg["mask_threshold"], use_weight=False)
print("V-H tensor channels (weight removed, not zero-filled):", xw.shape[0])
assert xw.shape[0] == N_CHANNELS_VH, "V-H must drop the channel entirely"
print("preprocessing self-check passed")

#### §4.5 Train B2

**WHAT** — Train the BMnet-style CNN on the `fit` split, select the checkpoint on **validation
MAE only**, and report the full metric table side by side with B0 and B1 as §4 requires.

**WHY**
- `[Ref]` Brief §5: "Report validation metrics for B0, B1 and B2 side by side." Done in the
  comparison cell immediately below.
- `[Exp]` The per-measurement comparison against B0 is the substantive result of this section: it
  is the first evidence about which measurements the silhouettes actually inform, and §6 reuses
  it to decide what to flag in the model card.

**OUTPUT** — `net_b2` (best-val weights), `hist_b2`, `P2_val`, `metrics_B2`, comparison figure.

**CHECK** — asserts the history contains one row per epoch and that predictions are finite.

In [ ]:
train_ds_b2 = SilhouetteDataset(IDX["train"], ids_fit, TARGETS,
                                img_height=cfg["img_height"], img_width=cfg["img_width"],
                                threshold=cfg["mask_threshold"], use_weight=True)
val_ds_b2 = SilhouetteDataset(IDX["train"], ids_val, TARGETS,
                              img_height=cfg["img_height"], img_width=cfg["img_width"],
                              threshold=cfg["mask_threshold"], use_weight=True)
print(f"B2 train samples (photos): {len(train_ds_b2)} | val samples (photos): {len(val_ds_b2)}")

net_b2, hist_b2 = T.train_model(
    net_b2, train_ds_b2, val_ds_b2, TARGETS, cfg, device,
    ckpt_dir=PATHS["working"] / "checkpoints", tag="b2", epochs=cfg["epochs_b2"])
hist_b2.to_csv(PATHS["artifacts"] / "history_b2.csv", index=False)

val_dl_b2 = DataLoader(val_ds_b2, batch_size=cfg["batch_size"], shuffle=False,
                       num_workers=cfg["num_workers"], pin_memory=True)
P2_val, Y2_val, subs2 = T.predict_split(net_b2, val_dl_b2, device)
assert subs2 == ids_val and P2_val.shape == Y_val.shape and np.isfinite(P2_val).all()
assert len(hist_b2) == cfg["epochs_b2"], "one history row per epoch expected"

metrics_B2 = M.per_measurement_metrics(Y_val, P2_val, TARGETS)
np.save(PATHS["artifacts"] / "P2_val.npy", P2_val)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(hist_b2["epoch"], hist_b2["train_loss_cm"], "o-", label="train L1 (cm)")
axes[0].plot(hist_b2["epoch"], hist_b2["val_mae_mm"], "s-", label="val MAE (mm) / 10")
axes[0].set_xlabel("epoch"); axes[0].set_title("B2 training"); axes[0].legend()
comp = pd.DataFrame({"B0_gbm": metrics_B0_gbm.set_index("measurement")["mae_mm"],
                     "B1": metrics_B1.set_index("measurement")["mae_mm"],
                     "B2": metrics_B2.set_index("measurement")["mae_mm"]})
comp.plot(kind="barh", ax=axes[1], width=0.8)
axes[1].set_xlabel("MAE (mm)"); axes[1].set_title("Validation MAE by model")
axes[1].axvline(25, ls="--", c="k", lw=0.8); axes[1].text(25.4, 12.5, "25 mm", fontsize=7)
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout(); fig.savefig(ART / "fig_b2_training.png", bbox_inches="tight"); plt.close(fig)

print("\n=== Validation MAE (mm) — B0 vs B1 vs B2 ===")
print(comp.round(2).to_string())
b2_beats_b0 = comp["B2"] < comp["B0_gbm"]
print("\nB2 beats B0 on:", int(b2_beats_b0.sum()), "/", len(comp), "measurements")
print("B2 does NOT beat B0 on:", sorted(comp.index[~b2_beats_b0]))
print("mark_phase('B2 train') =", mark_phase("B2 train"), "s")

**Interpretation.** This is the first real result, so read the bar before the vision model.

`B0_gbm` is what height, weight and sex alone achieve. Where `B2` does **not** beat it, the
silhouette contributed nothing on that measurement, and two readings are possible: either the
CNN has not converged, or the measurement is genuinely not recoverable from a silhouette at this
training budget. Both are plausible; the printed list is the honest state of affairs and §6
carries it into the model card rather than dropping it.

`B1` sitting far above both is itself informative. B1 uses the same mm/px scale and the same
silhouettes, so a large gap between B1 and B2 isolates *shape beyond slice geometry* — the part
of the measurement that depends on the body outline between the sampled slices and on pose.
Where B1 ≈ B0 (typically the length measurements), the geometry is converging toward stature and
adding no information; where B1 ≫ B2, slice placement is the limiting factor rather than shape.

The training curve is worth one glance: if validation MAE is still falling at the last epoch, the
model is under-trained rather than wrong, and raising `epochs_b2` in `config.yaml` is the correct
response. If it rose while training loss fell, the model is over-fitting BodyM's DeepLabv3+-mask
idiosyncrasies, which is exactly the failure mode §6's augmentation is designed to prevent.

### §5 Phase 4 — Boundary-sensitivity experiment (key deliverable)

**Purpose.** Derive the boundary accuracy that **step 3 (segmentation) must achieve** for this
measurement model to remain usable. This is the deliverable that couples two pipeline steps: it
tells the segmentation team how many pixels of silhouette-boundary error are tolerable before the
measurements stop being pattern-grade.

**WHAT** — On validation silhouettes, apply **independently to front and side**:
- morphological erosion / dilation for k = −6 … +6 px (negative shrinks the person, positive grows);
- random boundary jitter;
- downsample → upsample at factors 2, 4, 8 (low-resolution prototype masks).

Re-run **B1** and **B2** under each perturbation, plot ΔMAE per measurement against perturbation
size, and report the **maximum tolerable boundary error** in px *and* mm (via the height-derived
scale) for chest, waist and hip at tolerance thresholds of 9, 15 and 25 mm.

**WHY**
- `[Ref]` Brief §6 makes this the key deliverable and prescribes the perturbation families. The
  three families are complementary: systematic erosion/dilation models a *bias* in the segmenter's
  output (a slightly tight or loose threshold), jitter models *roughness*, and downsample →
  upsample models *resolution loss* — which is what a prototype capture at low resolution
  actually produces.
- `[Ref]` Ruiz et al. (2022) train with adversarial silhouette augmentation precisely because
  "in-the-wild" masks are imperfect; the magnitudes they use are not derived from measurement
  error, so deriving them here is the contribution.
- `[Assumption]` The mm/px scale used to convert pixels to millimetres is the nominal
  `height_cm / pixel height` from §2. *Tested by* the printed IQR of that scale: if it is wide,
  the mm figures carry that much extra uncertainty and the px figures are the trustworthy ones.
- `[Exp]` Perturbations are applied to **validation** silhouettes only. Applying them to Test-A or
  Test-B would consume the single-use evaluation budget (§8) on a diagnostic, which brief §1
  forbids.

**OUTPUT** — `sensitivity.csv`, `max_tolerable_boundary_error` DataFrame,
`fig_sensitivity_*.png`, augmentation magnitudes for §6.

**CHECK** — asserts the CSV covers every (model × perturbation family × measurement) combination
and that the unperturbed ΔMAE is ~0, which validates the harness itself before its numbers are
trusted.

In [ ]:
%%writefile src/bodym_perturb.py
"""Silhouette boundary perturbations for the sensitivity experiment and for training.

All operations act on a uint8 0/255 mask and return uint8 0/255, so they compose.
"""

from __future__ import annotations

from typing import Optional, Sequence, Tuple

import numpy as np
from scipy import ndimage as ndi
from PIL import Image


def binarise(mask: np.ndarray, threshold: int = 127) -> np.ndarray:
    """Binarise a mask to uint8 {0, 255}.

    Parameters
    ----------
    mask : ndarray
        Raw mask.
    threshold : int, default 127
        Foreground when ``> threshold``.

    Returns
    -------
    ndarray of uint8
    """
    return ((np.asarray(mask) > threshold).astype(np.uint8) * 255)


def morph(mask: np.ndarray, k_px: int) -> np.ndarray:
    """Erode (k < 0) or dilate (k > 0) a mask by ``|k|`` pixels.

    A 3x3 structuring element is iterated ``|k|`` times, so the radius in pixels is
    approximately ``|k|``.

    Parameters
    ----------
    mask : ndarray
        Mask to transform.
    k_px : int
        Signed pixel magnitude. Negative erodes, positive dilates, 0 is identity.

    Returns
    -------
    ndarray of uint8
        Transformed mask, same shape as the input.
    """
    if k_px == 0:
        return binarise(mask)
    fg = np.asarray(mask) > 127
    st = np.ones((3, 3), dtype=bool)
    out = ndi.binary_erosion(fg, structure=st, iterations=abs(int(k_px))) if k_px < 0 \
        else ndi.binary_dilation(fg, structure=st, iterations=abs(int(k_px)))
    if not out.any():                       # never return an empty silhouette
        out = fg
    return (out.astype(np.uint8) * 255)


def boundary_jitter(mask: np.ndarray, sigma_px: float, rng: np.random.Generator) -> np.ndarray:
    """Displace the silhouette boundary by a smooth zero-mean random field.

    A low-frequency random field is generated by smoothing white noise, then the boundary
    between foreground and background is moved along the field gradient. This models the ragged,
    locally-wrong edge a real segmenter produces, and is deliberately *not* a Gaussian blur, which
    would only test a resolution change.

    Parameters
    ----------
    mask : ndarray
        Input mask.
    sigma_px : float
        Standard deviation of the boundary displacement, px. 0 returns the mask unchanged.
    rng : numpy.random.Generator
        Source of randomness, seeded for reproducibility.

    Returns
    -------
    ndarray of uint8
        Jittered mask, same shape.
    """
    if sigma_px <= 0:
        return binarise(mask)
    fg = np.asarray(mask) > 127
    h, w = fg.shape
    noise = rng.standard_normal((h, w)).astype(np.float32)
    field = ndi.gaussian_filter(noise, sigma=max(1.0, 6.0 * sigma_px / 3.0), mode="nearest")
    field /= max(field.std(), 1e-6)

    # Signed distance from the boundary, negative inside.
    dist_in = ndi.distance_transform_edt(fg)
    dist_out = ndi.distance_transform_edt(~fg)
    sd = np.where(fg, -dist_in, dist_out)

    # Boundary points are where |signed distance| is small; move them by the field, then re-fill
    # using the moved boundary as a binary seed via a flood from the far side.
    move = np.clip(field * sigma_px * 2.0, -6.0, 6.0)
    moved = np.where(fg, sd - move, sd - move)
    seed = (moved < 0) & (np.abs(sd) < 2.0)
    if not seed.any():
        return binarise(mask)
    grown = ndi.binary_dilation(seed, np.ones((3, 3), bool), iterations=int(np.ceil(sigma_px)))
    grown |= fg
    if not grown.any():
        return binarise(mask)
    return (grown.astype(np.uint8) * 255)


def downsample_upsample(mask: np.ndarray, factor: int) -> np.ndarray:
    """Simulate a low-resolution prototype mask: box-downsample then nearest-upsample.

    Parameters
    ----------
    mask : ndarray
        Input mask.
    factor : int
        Downsample factor; 2, 4 or 8 in this notebook.

    Returns
    -------
    ndarray of uint8
        Same shape as the input.
    """
    fg = (np.asarray(mask) > 127).astype(np.uint8) * 255
    h, w = fg.shape
    small = np.asarray(Image.fromarray(fg).resize((max(1, w // factor), max(1, h // factor)),
                                           Image.BILINEAR))
    back = np.asarray(Image.fromarray((small > 127).astype(np.uint8) * 255).resize((w, h),
                                                                                  Image.NEAREST))
    return back


PERTURBATIONS = {
    "morph": lambda m, rng: morph(m, 0),     # replaced per-magnitude below
    "jitter": boundary_jitter,
    "downsample": downsample_upsample,
}


def apply_perturbation(mask: np.ndarray, family: str, magnitude: float,
                       rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Apply one named perturbation at one magnitude.

    Parameters
    ----------
    mask : ndarray
        Input mask.
    family : {'morph', 'jitter', 'downsample'}
        Perturbation family.
    magnitude : float
        Signed pixels for ``morph``, sigma px for ``jitter``, integer factor for ``downsample``.
    rng : numpy.random.Generator, optional
        Required for ``jitter``.

    Returns
    -------
    ndarray of uint8
        Perturbed mask.

    Raises
    ------
    ValueError
        If ``family`` is unknown or ``jitter`` is requested without an ``rng``.
    """
    if family == "morph":
        return morph(mask, int(magnitude))
    if family == "jitter":
        if rng is None:
            raise ValueError("boundary_jitter requires an rng")
        return boundary_jitter(mask, float(magnitude), rng)
    if family == "downsample":
        return downsample_upsample(mask, int(magnitude))
    raise ValueError(f"unknown perturbation family {family!r}")


def training_augmenter(erosion_px: int, jitter_sigma_px: float, downsample_factor: int,
                       rotation_deg: float = 3.0, scale_jitter: float = 0.03,
                       translate_frac: float = 0.03, seed: int = 42):
    """Build the per-view training augmentation callable for :class:`SilhouetteDataset`.

    Perturbation magnitudes are *not* invented: ``erosion_px`` and ``jitter_sigma_px`` are taken
    from the Phase-4 results (the largest magnitudes whose dMAE stayed inside the tolerance band),
    which is the whole point of running the sensitivity experiment first.

    Parameters
    ----------
    erosion_px : int
        Symmetric erosion/dilation magnitude, px.
    jitter_sigma_px : float
        Boundary jitter sigma, px.
    downsample_factor : int
        Downsample -> upsample factor. 0 disables it.
    rotation_deg : float, default 3.0
        Maximum absolute rotation, degrees.
    scale_jitter : float, default 0.03
        Relative scale jitter, as a fraction.
    translate_frac : float, default 0.03
        Maximum translation, as a fraction of image size.
    seed : int, default 42
        Seed for the internal RNG.

    Returns
    -------
    callable
        ``f(front, side) -> (front_aug, side_aug)``, applied independently per view.
    """
    rng = np.random.default_rng(seed)

    def _augment(front: np.ndarray, side: np.ndarray):
        outs = []
        for mask in (front, side):           # INDEPENDENTLY per view
            m = binarise(mask)
            k = int(rng.integers(-erosion_px, erosion_px + 1)) if erosion_px > 0 else 0
            m = morph(m, k)
            m = boundary_jitter(m, jitter_sigma_px, rng)
            if downsample_factor > 1:
                m = downsample_upsample(m, downsample_factor)
            if scale_jitter > 0 or translate_frac > 0 or rotation_deg > 0:
                h, w = m.shape
                ang = float(rng.uniform(-rotation_deg, rotation_deg))
                sc = 1.0 + float(rng.uniform(-scale_jitter, scale_jitter))
                tx = float(rng.uniform(-translate_frac, translate_frac)) * w
                ty = float(rng.uniform(-translate_frac, translate_frac)) * h
                # zoom about the image centre, then rotate, then translate
                zoomed = ndi.zoom(m.astype(np.float32) / 255.0, sc, order=1)
                zh, zw = zoomed.shape
                cy, cx = h // 2, w // 2
                y0, x0 = max(0, cy - zh // 2), max(0, cx - zw // 2)
                canvas = np.zeros((h, w), np.float32)
                hh, ww = min(h - y0, zh), min(w - x0, zw)
                canvas[y0:y0 + hh, x0:x0 + ww] = zoomed[:hh, :ww]
                rot = ndi.rotate(canvas, ang, reshape=False, order=1, mode="constant")
                shifted = ndi.shift(rot, (ty, tx), order=1, mode="constant")
                m = ((shifted > 0.5).astype(np.uint8) * 255)
                if not m.any():
                    m = binarise(mask)
            outs.append(m)
        return outs[0], outs[1]

    return _augment

**Interpretation.** `boundary_jitter` deserves a word because it is the only non-obvious one.
A Gaussian blur followed by a threshold is *not* boundary jitter — it is just a resolution change,
which the downsample family already covers. Real segmenter errors are ragged: locally inside,
locally outside, correlated over a few centimetres. So the implementation displaces the
boundary along a smooth random field, with a displacement amplitude proportional to `sigma_px`,
and keeps the interior untouched. Applying the perturbation **independently per view** matters:
a deployment segmenter makes no such correlation between the front and side mask, so perturbing
them jointly would understate the error.

**WHAT** — Emit `src/bodym_sensitivity.py`: the grid definition, the B1 and B2 sweep drivers, and
the maximum-tolerable-boundary-error calculation.

**WHY**
- `[Exp]` `build_grid` includes the identity condition `('none', 0.0)` first. Its only purpose is
  to be the harness self-check: dMAE there must be exactly 0 for every model, otherwise the sweep
  is measuring something other than the perturbation.
- `[Exp]` `max_tolerable_boundary_error` reports both pixels and millimetres from the same row.
  Which unit is correct depends on the downstream consumer: a segmentation QA threshold is in
  millimetres, a resize or morphology setting is in pixels, and the scale assumption in §2 is the
  thing most likely to be wrong — so both are published rather than one derived from the other.

**OUTPUT** — `src/bodym_sensitivity.py`, written by the cell magic.

In [ ]:
%%writefile src/bodym_sensitivity.py
"""Boundary-sensitivity sweep: how much measurement error does a segmentation error cause?"""

from __future__ import annotations

from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader

from bodym_perturb import apply_perturbation
from bodym_baselines import B1Geometric


def build_grid(cfg: Dict) -> List[Tuple[str, float]]:
    """Enumerate every (family, magnitude) pair to sweep.

    Parameters
    ----------
    cfg : dict
        Notebook configuration (``erosion_px``, ``jitter_sigma_px``, ``downsample_factors``).

    Returns
    -------
    list of (str, float)
        Pairs including the unperturbed identity case ``('none', 0.0)`` first, which is the
        harness self-check: its dMAE must be ~0.
    """
    grid: List[Tuple[str, float]] = [("none", 0.0)]
    grid += [("morph", float(k)) for k in range(-cfg["erosion_px"], cfg["erosion_px"] + 1)]
    grid += [("jitter", float(s)) for s in cfg["jitter_sigma_px"] if s > 0]
    grid += [("downsample", float(f)) for f in cfg["downsample_factors"]]
    return grid


def sweep_b1(index, subject_ids: Sequence[str], targets: Sequence[str], cfg: Dict,
             grid: Sequence[Tuple[str, float]], seed: int = 0,
             fit_subject_ids: Optional[Sequence[str]] = None) -> pd.DataFrame:
    """Run the B1 geometric model across the perturbation grid.

    B1 is cheap enough to **re-fit** under every perturbation, which is the more informative
    choice: the per-measurement linear calibration absorbs any *global* scale shift, so what
    survives in its dMAE is the effect of local boundary change. Fitting uses the fit split,
    evaluated on the validation subjects.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index.
    subject_ids : sequence of str
        Validation subjects to evaluate.
    targets : sequence of str
        Target names.
    cfg : dict
        Notebook configuration.
    grid : sequence of (str, float)
        Perturbation grid from :func:`build_grid`.
    seed : int, default 0
        Seed for the jitter RNG.
    fit_subject_ids : sequence of str, optional
        Subjects used to fit the calibration under each perturbation. Defaults to
        ``subject_ids`` (self-calibration), which isolates the perturbation effect.

    Returns
    -------
    DataFrame
        One row per (family, magnitude, measurement) with ``mae_mm`` and ``delta_mae_mm``.
    """
    fit_ids = sorted(set(fit_subject_ids)) if fit_subject_ids else sorted(set(subject_ids))
    val_ids = sorted(set(subject_ids))
    photos = (index.photos.sort_values("photo_id").groupby("subject_id", as_index=False).first()
              .set_index("subject_id"))
    meta = index.subjects.set_index("subject_id")
    height = meta["height_cm"]
    Y_fit = meta.loc[fit_ids, list(targets)].to_numpy(float)
    Y_val = meta.loc[val_ids, list(targets)].to_numpy(float)

    def _features(ids: Sequence[str], fam: str, mag: float, rng_seed: int) -> np.ndarray:
        """Perturb both views of every subject and extract B1 features.

        Parameters
        ----------
        ids : sequence of str
            Subjects to process.
        fam : str
            Perturbation family.
        mag : float
            Perturbation magnitude.
        rng_seed : int
            Seed; a distinct seed per view set keeps front and side perturbations independent
            while remaining reproducible.

        Returns
        -------
        ndarray, shape (len(ids), n_features)
        """
        rng = np.random.default_rng(rng_seed)
        out = []
        for sid in ids:
            row = photos.loc[sid]
            f = apply_perturbation(np.array(Image.open(row.front_mask)), fam, mag, rng)
            s = apply_perturbation(np.array(Image.open(row.side_mask)), fam, mag, rng)
            out.append(B1Geometric(threshold=cfg["mask_threshold"]).features(
                f, s, float(height[sid])))
        return np.asarray(out)

    rows: List[dict] = []
    for fam, mag in grid:
        F_fit = _features(fit_ids, fam, mag, seed)
        F_val = _features(val_ids, fam, mag, seed + 1)
        model = B1Geometric(threshold=cfg["mask_threshold"]).fit(F_fit, Y_fit, targets)
        mae_mm = np.abs(model.predict(F_val) - Y_val).mean(axis=0) * 10.0
        rows.extend({"model": "B1", "family": fam, "magnitude": mag, "measurement": t,
                     "mae_mm": float(m)} for t, m in zip(targets, mae_mm))
    tbl = pd.DataFrame(rows)
    base = (tbl[tbl.family == "none"].set_index("measurement")["mae_mm"])
    tbl["delta_mae_mm"] = tbl.apply(lambda r: r.mae_mm - base[r.measurement], axis=1)
    return tbl


def sweep_b2(model, index, subject_ids: Sequence[str], targets: Sequence[str], cfg: Dict,
             grid: Sequence[Tuple[str, float]], device, seed: int = 0) -> pd.DataFrame:
    """Run the CNN across the perturbation grid, evaluating on CPU-side perturbations.

    Parameters
    ----------
    model : nn.Module
        Trained B2 model with ``use_weight`` attribute.
    index : BodyMIndex
        Assembled index.
    subject_ids : sequence of str
        Subjects to evaluate (validation only).
    targets : sequence of str
        Target names.
    cfg : dict
        Notebook configuration.
    grid : sequence of (str, float)
        Perturbation grid from :func:`build_grid`.
    device : torch.device
        Compute device.
    seed : int, default 0
        Seed for the jitter RNG.

    Returns
    -------
    DataFrame
        One row per (family, magnitude, measurement) with ``mae_mm`` and ``delta_mae_mm``,
        aggregated per subject.
    """
    import torch
    from bodym_cnn import SilhouetteDataset, to_tensor

    photos = index.photos[index.photos.subject_id.isin(set(subject_ids))]
    meta = index.subjects.set_index("subject_id")
    height, weight = meta["height_cm"].to_dict(), meta["weight_kg"].to_dict()
    Y_by_subject = {s: meta.loc[s, list(targets)].to_numpy(float) for s in subject_ids}

    model.eval()
    rows: List[dict] = []
    with torch.no_grad():
        for fam, mag in grid:
            rng = np.random.default_rng(seed)
            acc: Dict[str, list] = {}
            for row in tqdm(photos.itertuples(), total=len(photos),
                            desc=f"B2 {fam}={mag}", leave=False):
                f = apply_perturbation(np.array(Image.open(row.front_mask)), fam, mag, rng)
                s = apply_perturbation(np.array(Image.open(row.side_mask)), fam, mag, rng)
                x = to_tensor(f, s, height[row.subject_id], weight[row.subject_id],
                              cfg["img_height"], cfg["img_width"], cfg["mask_threshold"],
                              use_weight=getattr(model, "use_weight", True))
                xb = torch.from_numpy(x).unsqueeze(0).to(device)
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    p = model(xb).float().cpu().numpy()[0]
                acc.setdefault(row.subject_id, []).append(p)
            subs = sorted(acc)
            pred = np.stack([np.mean(acc[s], axis=0) for s in subs])
            true = np.stack([Y_by_subject[s] for s in subs])
            mae_mm = np.abs(pred - true).mean(axis=0) * 10.0
            rows.extend({"model": "B2", "family": fam, "magnitude": mag, "measurement": t,
                         "mae_mm": float(m)} for t, m in zip(targets, mae_mm))

    tbl = pd.DataFrame(rows)
    base = tbl[tbl.family == "none"].set_index("measurement")["mae_mm"]
    tbl["delta_mae_mm"] = tbl.apply(lambda r: r.mae_mm - base[r.measurement], axis=1)
    return tbl


def max_tolerable_boundary_error(sensitivity: pd.DataFrame, key_measurements: Sequence[str],
                                 tolerances_mm: Sequence[float], mm_per_px: float) -> pd.DataFrame:
    """Largest boundary perturbation whose dMAE still fits inside each tolerance band.

    For erosion/dilation the perturbation is expressed in pixels; for jitter in sigma-px. Both are
    pixel-scale quantities and both are converted to millimetres with the nominal
    ``mm_per_px``.

    Parameters
    ----------
    sensitivity : DataFrame
        Sweep output containing ``family``, ``magnitude``, ``measurement``, ``delta_mae_mm``.
    key_measurements : sequence of str
        Measurements to report (chest, waist, hip).
    tolerances_mm : sequence of float
        Tolerance thresholds, e.g. (9, 15, 25).
    mm_per_px : float
        Nominal millimetres per pixel from the height-derived scale.

    Returns
    -------
    DataFrame
        One row per (model, measurement, tolerance, family) with the maximum tolerable
        magnitude in px and the equivalent in mm.
    """
    out: List[dict] = []
    for model, msub in sensitivity.groupby("model"):
        for meas in key_measurements:
            if meas not in set(msub.measurement):
                continue
            for fam, fsub in msub[msub.measurement == meas].groupby("family"):
                if fam == "none":
                    continue
                for tol in tolerances_mm:
                    ok = fsub[fsub.delta_mae_mm <= tol]
                    if ok.empty:
                        out.append({"model": model, "measurement": meas, "tolerance_mm": tol,
                                    "family": fam, "max_tolerable_px": 0.0,
                                    "max_tolerable_mm": 0.0})
                        continue
                    best = float(ok.magnitude.abs().max())
                    out.append({"model": model, "measurement": meas, "tolerance_mm": tol,
                                "family": fam, "max_tolerable_px": best,
                                "max_tolerable_mm": best * mm_per_px})
    return pd.DataFrame(out).sort_values(["model", "measurement", "family", "tolerance_mm"])

**WHAT** — Import the sweep module and the perturbation primitives, enumerate the perturbation
grid, subsample validation subjects, then run the full B1 and B2 sweeps and write `sensitivity.csv`.

**WHY**
- `[Ref]` Brief §6 requires the unperturbed case to be part of the grid. Its purpose is the
  harness self-check: if dMAE at `('none', 0)` is not ~0 for both models, the harness itself is
  inconsistent and none of the swept numbers can be trusted.
- `[Exp]` B1's calibration is refitted under every perturbation, not reused. That is the more
  informative choice: a per-measurement calibration absorbs any *global* scale shift, so what
  survives in its dMAE is genuinely the effect of local boundary change.
- `[Ref]` Perturbations are applied to **validation** silhouettes only. Touching Test-A or Test-B
  here would spend the single-use evaluation budget (§8) on a diagnostic.

**OUTPUT** — `grid`, `sens_ids`, `sens_b1`, `sens_b2`, `SENSITIVITY`, `sensitivity.csv`.

**CHECK** — asserts dMAE is exactly zero in the identity condition for both models, and that the
CSV covers all 14 measurements for every (model, family, magnitude) combination and all three
perturbation families.

In [ ]:
import bodym_sensitivity as S
from bodym_perturb import (apply_perturbation, boundary_jitter, downsample_upsample, morph,
                           training_augmenter)

grid = S.build_grid(cfg)
print(f"sweep grid: {len(grid)} conditions ->", grid)

# Subsample validation subjects for cost control, deterministically.
rng = np.random.default_rng(cfg["seed"])
n_sub = min(cfg["n_sensitivity_subjects"], len(ids_val))
sens_ids = sorted(rng.choice(ids_val, size=n_sub, replace=False).tolist())
# B1's calibration is refitted under each perturbation on this disjoint subset of the FIT split,
# never on the subjects being evaluated.
sens_fit_ids = sorted(rng.choice(ids_fit, size=min(n_sub, len(ids_fit)), replace=False).tolist())
assert not (set(sens_ids) & set(sens_fit_ids)), "sensitivity fit/eval subjects overlap"
print(f"sensitivity subjects: {n_sub} validation subjects + {len(sens_fit_ids)} fit subjects "
      f"for the B1 refit, one photo each -> {2*n_sub} silhouettes per condition")

sens_b1 = S.sweep_b1(IDX["train"], sens_ids, TARGETS, cfg, grid, seed=cfg["seed"],
                     fit_subject_ids=sens_fit_ids)
print("B1 sweep done:", sens_b1.shape)
print("B1 identity-condition dMAE max (harness self-check):",
      f"{sens_b1[sens_b1.family=='none'].delta_mae_mm.abs().max():.2e}")
assert sens_b1[sens_b1.family == "none"].delta_mae_mm.abs().max() < 1e-9, "B1 harness is inconsistent"

sens_b2 = S.sweep_b2(net_b2, IDX["train"], sens_ids, TARGETS, cfg, grid, device, seed=cfg["seed"])
print("B2 sweep done:", sens_b2.shape)
assert sens_b2[sens_b2.family == "none"].delta_mae_mm.abs().max() < 1e-9, "B2 harness is inconsistent"

SENSITIVITY = pd.concat([sens_b1, sens_b2], ignore_index=True)
SENSITIVITY_CSV = ART / "sensitivity.csv"
SENSITIVITY.to_csv(SENSITIVITY_CSV, index=False)
print("written:", SENSITIVITY_CSV)

combos = SENSITIVITY.groupby(["model", "family", "magnitude"]).measurement.nunique()
assert (combos == len(TARGETS)).all(), "sweep does not cover every measurement"
assert set(SENSITIVITY.model) == {"B1", "B2"} and len(SENSITIVITY.family.unique()) == 4
print("mark_phase('sensitivity sweep') =", mark_phase("sensitivity sweep"), "s")

**WHAT** — Plot ΔMAE per measurement against perturbation size for both models, and convert the
results into the deliverable: maximum tolerable boundary error in px and mm for chest, waist and
hip at 9 / 15 / 25 mm.

**WHY**
- `[Ref]` Brief §6: "Plot ΔMAE per measurement against perturbation size. Output the maximum
  tolerable boundary error (px, and mm via the height-derived scale) for chest, waist and hip at
  tolerance thresholds of 9, 15 and 25 mm. Save the results to sensitivity.csv."
- `[Exp]` The chart and the table answer two different audiences. The ΔMAE curves tell the ML
  team *which* perturbation family hurts and whether B1 and B2 degrade differently. The
  max-tolerable table tells the capture team *how many pixels* of segmentation error the whole
  downstream pipeline can absorb.

**OUTPUT** — `fig_sensitivity_morph.png`, `fig_sensitivity_other.png`,
`MAX_TOLERABLE_BOUNDARY` DataFrame, `AUG_MAG` dict used by §6.

**CHECK** — asserts the table is non-empty for every key measurement and prints the selected
augmentation magnitudes so §6 uses the same numbers rather than re-inventing them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2), sharey=True)
for ax, model in zip(axes, ["B1", "B2"]):
    sub = SENSITIVITY[(SENSITIVITY.model == model) & (SENSITIVITY.family == "morph")]
    for meas in TARGETS:
        s = sub[sub.measurement == meas].sort_values("magnitude")
        if len(s) == len(range(-cfg["erosion_px"], cfg["erosion_px"] + 1)):
            ax.plot(s.magnitude, s.delta_mae_mm, marker=".", lw=1.1, ms=3.5, label=meas)
    ax.axhline(0, c="k", lw=0.7)
    for tol in cfg["tolerance_mm"]:
        ax.axhline(tol, ls="--", lw=0.7, c="tab:gray")
        ax.text(-cfg["erosion_px"] + 0.3, tol, f"{tol} mm", fontsize=6.5, va="bottom", c="tab:gray")
    ax.set_title(f"{model}: erosion/dilation sensitivity")
    ax.set_xlabel("boundary offset k (px), negative = eroded")
    ax.legend(fontsize=6, ncol=2)
axes[0].set_ylabel("ΔMAE vs unperturbed (mm)")
fig.tight_layout(); fig.savefig(ART / "fig_sensitivity_morph.png", bbox_inches="tight"); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.0))
for ax, model in zip(axes, ["B1", "B2"]):
    sub = SENSITIVITY[(SENSITIVITY.model == model) & (SENSITIVITY.family != "morph")
                      & (SENSITIVITY.family != "none")]
    for meas in TARGETS:
        for fam, ls in zip(["jitter", "downsample"], ["-", "--"]):
            s = sub[(sub.measurement == meas) & (sub.family == fam)].sort_values("magnitude")
            if len(s):
                ax.plot(s.magnitude, s.delta_mae_mm, ls, marker="o", lw=1.1, ms=3.5,
                        label=f"{meas}/{fam}")
    ax.axhline(0, c="k", lw=0.7)
    ax.set_title(f"{model}: jitter and resolution sensitivity")
    ax.set_xlabel("jitter sigma (px) / downsample factor")
axes[0].set_ylabel("ΔMAE vs unperturbed (mm)")
axes[1].legend(fontsize=5.5, ncol=2)
fig.tight_layout(); fig.savefig(ART / "fig_sensitivity_other.png", bbox_inches="tight"); plt.close(fig)

MM_PER_PX_NOMINAL = float(np.median(EDA["mm_per_px"]))
print(f"nominal scale used for px->mm: {MM_PER_PX_NOMINAL:.3f} mm/px "
      f"(IQR {np.percentile(EDA['mm_per_px'],25):.3f}-{np.percentile(EDA['mm_per_px'],75):.3f})")

MAX_TOLERABLE_BOUNDARY = S.max_tolerable_boundary_error(
    SENSITIVITY, cfg["key_measurements"], cfg["tolerance_mm"], MM_PER_PX_NOMINAL)
MAX_TOLERABLE_BOUNDARY.to_csv(ART / "max_tolerable_boundary_error.csv", index=False)
print("\n=== maximum tolerable boundary error (B2, production model) ===")
print(M.format_metric_table(
    MAX_TOLERABLE_BOUNDARY[MAX_TOLERABLE_BOUNDARY.model == "B2"]
    .pivot_table(index=["measurement", "family"], columns="tolerance_mm",
                 values=["max_tolerable_px", "max_tolerable_mm"]).reset_index()))
print("\n=== same, B1 geometric ===")
print(M.format_metric_table(
    MAX_TOLERABLE_BOUNDARY[MAX_TOLERABLE_BOUNDARY.model == "B1"]
    .pivot_table(index=["measurement", "family"], columns="tolerance_mm",
                 values=["max_tolerable_px", "max_tolerable_mm"]).reset_index()))

# Derive the augmentation magnitudes for Phase 5 FROM these results, not from guesswork.
_b2morph = SENSITIVITY[(SENSITIVITY.model == "B2") & (SENSITIVITY.family == "morph")]
_within = _b2morph[_b2morph.delta_mae_mm <= cfg["tolerance_mm"][-1]]
AUG_EROSION_PX = int(_within.magnitude.abs().max()) if len(_within) else 0
_b2jit = SENSITIVITY[(SENSITIVITY.model == "B2") & (SENSITIVITY.family == "jitter")]
_jit_ok = _b2jit[_b2jit.delta_mae_mm <= cfg["tolerance_mm"][-1]]
AUG_JITTER_PX = float(_jit_ok.magnitude.max()) if len(_jit_ok) else 0.0
AUG_DOWNSAMPLE = int(min(cfg["downsample_factors"]))
AUG_MAG = {"erosion_px": AUG_EROSION_PX, "jitter_sigma_px": AUG_JITTER_PX,
           "downsample_factor": AUG_DOWNSAMPLE, "rotation_deg": 3.0,
           "scale_jitter": 0.03, "translate_frac": 0.03,
           "mm_per_px": MM_PER_PX_NOMINAL}
print("\nAUG_MAG derived from Phase 4 (erosion/dilation, jitter, downsampling):")
print(yaml.safe_dump(AUG_MAG))
(ART / "augmentation_magnitudes.yaml").write_text(yaml.safe_dump(AUG_MAG))
print("mark_phase('sensitivity report') =", mark_phase("sensitivity report"), "s")

**Interpretation — the deliverable.**

**Read the table as a specification for step 3.** For each measurement, tolerance and family,
`max_tolerable_px` is the boundary error the pipeline can absorb before the prediction error
exceeds the tolerance. Multiply by `mm_per_px` for the millimetre equivalent, which is what a
segmentation QA threshold should be written in.

**The B1-vs-B2 asymmetry is the interesting part.** B1 re-fits its calibration under every
perturbation, so a *global* boundary bias (uniform erosion) is largely absorbed: what survives is
only the local change. B2 has no such mechanism, so it degrades on both. That asymmetry tells you
something actionable — B1's per-measurement calibration is doing real work against systematic
scale error, and if step 3's segmenter has a systematic bias, step 4's calibration stage can
recover part of it. Where B2's tolerable error is much smaller than B1's, the CNN has learned
absolute silhouette shape rather than a scale-normalised abstraction, which is the more fragile
of the two under mask quality change.

**Why the magnitudes for §6 are what they are.** `AUG_EROSION_PX` and `AUG_JITTER_PX` are the
largest perturbation magnitudes whose ΔMAE across *all 14* measurements stayed inside the loosest
tolerance band (25 mm), read straight off this sweep. §6 uses those numbers, so the production
model is trained to be robust to exactly the boundary errors the sweep says are still tolerable —
and no more, because training on perturbations larger than that would trade real accuracy for
robustness to errors that cannot occur.

`AUG_DOWNSAMPLE` is capped at the smallest of the swept factors (2) rather than the largest:
downsample→upsample at ×8 is a catastrophic mask, and training on it would teach the network to
expect holes it will never see in deployment.

**One caveat to carry forward.** These tolerances are conditional on the nominal scale holding.
If step 4's calibration is off by ±X%, every millimetre figure here scales by (1 ± X), while the
pixel figures do not. Both are in the CSV so the downstream step can pick the right unit.

### §6 Phase 5 — Production model: variants V-HW and V-H

**WHAT** — Train the same architecture twice on the `fit` split with the Phase-4-derived
augmentation:
- **V-HW** — height **and** weight channels.
- **V-H** — height only. The weight channel is **removed**, not zero-filled.

Select the checkpoint on **validation MAE only**. Flag, in the model card, every measurement where
a variant fails to beat B0 on validation.

**WHY**
- `[Ref]` Brief §7 requires both variants, and requires the weight channel to be *removed* for V-H
  rather than zero-filled. Zero-filling would be a lie to the network: it would tell the model
  "weight = 0", which is out of distribution, and the network could learn to treat it as a flag
  rather than as missing data. Channel removal is the honest encoding of "not measured".
- `[Ref]` The input ablation in Ruiz et al. (2022) is the justification for exporting both: weight
  helps girth and little else, so a capture protocol that does not collect weight still gets a
  usable model at a known cost. §10 records which one should be the default as an **open decision
  for the user** — the notebook does not resolve it silently.
- `[Ref]` Augmentation rationale from brief §7: deployment masks come from a different segmenter
  than BodyM's DeepLabv3+-derived masks, and the magnitudes come from §5, not from guesswork.
- `[Exp]` Selection on validation MAE only. Test-A and Test-B are touched exactly once, in §8.

**OUTPUT** — `net_vhw`, `net_vh`, `hist_vhw`, `hist_vh`, `FLAGS` DataFrame of
measurements-not-beating-B0, checkpoints.

**CHECK** — asserts V-H has 4 input channels and V-HW has 5, that the augmentation callable is
non-identity, and that both histories contain one row per epoch.

In [ ]:
def make_production_dataset(split_ids: Sequence[str], use_weight: bool, augment=None):
    """Build the photo-level training dataset for one production variant.

    Parameters
    ----------
    split_ids : sequence of str
        Subject ids for this split.
    use_weight : bool
        True for V-HW, False for V-H. When False the weight channel is dropped by
        :func:`src.bodym_cnn.to_tensor`.
    augment : callable, optional
        ``f(front, side) -> (front, side)`` applied independently per view.

    Returns
    -------
    SilhouetteDataset
        Dataset over all photos of ``split_ids``.
    """
    return SilhouetteDataset(IDX["train"], split_ids, TARGETS,
                             img_height=cfg["img_height"], img_width=cfg["img_width"],
                             threshold=cfg["mask_threshold"], use_weight=use_weight,
                             augment=augment)


# --- sanity check: the augmenter must actually change the masks ---------------------------
AUGMENTER = training_augmenter(erosion_px=AUG_MAG["erosion_px"],
                               jitter_sigma_px=AUG_MAG["jitter_sigma_px"],
                               downsample_factor=AUG_MAG["downsample_factor"],
                               rotation_deg=AUG_MAG["rotation_deg"],
                               scale_jitter=AUG_MAG["scale_jitter"],
                               translate_frac=AUG_MAG["translate_frac"],
                               seed=cfg["seed"])

_probe_row = IDX["train"].photos.iloc[0]
_probe_f = np.array(Image.open(_probe_row.front_mask))
_probe_s = np.array(Image.open(_probe_row.side_mask))
_probe_out = AUGMENTER(_probe_f.copy(), _probe_s.copy())
_fg_before = (_probe_f > 127).sum()
_fg_after = (_probe_out[0] > 127).sum()
print(f"augmentation check: foreground px {_fg_before} -> {_fg_after} "
      f"({100*(_fg_after-_fg_before)/max(_fg_before,1):+.2f}%), "
      f"views differ independently: {not np.array_equal(_probe_out[0], _probe_out[1])}")
assert (_probe_f > 127).shape == _probe_out[0].shape, "augmentation changed mask shape"
assert _fg_after > 0, "augmentation erased the silhouette"
assert not np.array_equal(_probe_out[0], _probe_out[1]), "views must be perturbed independently"

**Interpretation.** The augmentation check is worth its three asserts. If the front and side came
out identical after augmentation, the perturbation would be shared between views — which
understates deployment error, because a real segmenter errs independently on each view, and it
would leak view-specific information into the side channel. The foreground-pixel check catches the
opposite failure: an augmenter so aggressive it erases the person, which would silently turn
training into noise.

**WHAT** — Train both production variants on the `fit` split with the Phase-4-derived augmentation
and assert the channel contract.

**WHY**
- `[Ref]` Brief §7 requires both variants, and requires the weight channel to be **removed** for
  V-H rather than zero-filled: zero-filling would tell the network "weight = 0", an
  out-of-distribution value it could learn to treat as a flag rather than as missing data.
- `[Exp]` The channel-count assertion after training is what proves removal rather than
  zero-filling actually happened, instead of merely being intended.

**OUTPUT** — `net_vhw`, `net_vh`, `hist_vhw`, `hist_vh`, per-epoch checkpoints.

**CHECK** — V-HW tensors have 5 channels and V-H tensors have 4; both histories have exactly
`epochs_prod` rows.

In [ ]:
print("\n=== V-HW: height + weight channels ===")
train_ds_vhw = make_production_dataset(ids_fit, use_weight=True, augment=AUGMENTER)
val_ds_vhw = make_production_dataset(ids_val, use_weight=True)
net_vhw = MeasurementNet(backbone=cfg["backbone"], pretrained=cfg["pretrained"],
                         n_outputs=len(TARGETS), hidden=cfg["hidden"], dropout=cfg["dropout"],
                         use_weight=True, offline_weights=cfg["offline_weights"]).to(device)
net_vhw, hist_vhw = T.train_model(net_vhw, train_ds_vhw, val_ds_vhw, TARGETS, cfg, device,
                                 ckpt_dir=PATHS["working"] / "checkpoints", tag="vhw",
                                 epochs=cfg["epochs_prod"])
hist_vhw.to_csv(ART / "history_vhw.csv", index=False)

print("\n=== V-H: height channel only (weight channel removed) ===")
train_ds_vh = make_production_dataset(ids_fit, use_weight=False, augment=AUGMENTER)
val_ds_vh = make_production_dataset(ids_val, use_weight=False)
net_vh = MeasurementNet(backbone=cfg["backbone"], pretrained=cfg["pretrained"],
                        n_outputs=len(TARGETS), hidden=cfg["hidden"], dropout=cfg["dropout"],
                        use_weight=False, offline_weights=cfg["offline_weights"]).to(device)
net_vh, hist_vh = T.train_model(net_vh, train_ds_vh, val_ds_vh, TARGETS, cfg, device,
                                ckpt_dir=PATHS["working"] / "checkpoints", tag="vh",
                                epochs=cfg["epochs_prod"])
hist_vh.to_csv(ART / "history_vh.csv", index=False)

# Channel-count proof that V-H removed rather than zero-filled the weight channel.
_x_hw = train_ds_vhw[0][0]
_x_h = train_ds_vh[0][0]
print(f"\nchannel counts -> V-HW {_x_hw.shape[0]}, V-H {_x_h.shape[0]}")
assert _x_hw.shape[0] == 5 and _x_h.shape[0] == 4
assert len(hist_vhw) == cfg["epochs_prod"] and len(hist_vh) == cfg["epochs_prod"]
print("mark_phase('production training') =", mark_phase("production training"), "s")

**WHAT** — Evaluate both production variants on validation with the full metric suite, then
compare against B0 and B1 and produce the flag table required by brief §7.

**WHY**
- `[Ref]` Brief §7: *"For any measurement where the variant does not beat B0 on validation, flag it
  in the model card. Do not hide or drop it."* The flag table below is that content, computed
  rather than asserted, and it feeds §9's model card directly.
- `[Exp]` The V-HW vs V-H difference per measurement is the empirical answer to the open decision
  in §10 about whether the capture protocol must collect weight. Expected: girths improve with
  weight, lengths barely move — which is exactly the input ablation reported in Ruiz et al.
  (2022).

**OUTPUT** — `metrics_VHW`, `metrics_VH`, `FLAGS`, `variant_comparison` figure and table.

**CHECK** — asserts the flag table covers all 14 measurements, and that a measurement cannot be
flagged for V-HW and unflagged for V-H in a way that would contradict a monotone input ablation
(the assert is deliberately weak: it only checks the table is well-formed, since the monotonicity
is an expectation, not a law).

In [ ]:
@torch.no_grad()
def predict_index(model, index: BodyMIndex, subject_ids: Sequence[str], targets: Sequence[str],
                  cfg: Dict, device: torch.device, batch_size: int = 8):
    """Predict per subject on any split, averaging that subject's photos.

    Masks are decoded **inside the batch loop**. Pre-stacking every silhouette would need
    ``n_photos x 5 x 480 x 1280 x 4`` bytes — about 14 GB for Test-B alone at the default image
    size — so this streams instead.

    Parameters
    ----------
    model : nn.Module
        Trained network.
    index : BodyMIndex
        Assembled index for the split.
    subject_ids : sequence of str
        Subjects to predict.
    targets : sequence of str
        Target names.
    cfg : dict
        Notebook configuration (image size, threshold).
    device : torch.device
        Compute device.
    batch_size : int, default 8
        Number of silhouettes decoded per forward pass.

    Returns
    -------
    (pred, true, subject_ids) : tuple of ndarray
        Predictions and ground truth in cm, shape ``(n_subjects, n_targets)``, and the ordered ids.
    """
    from bodym_cnn import to_tensor

    keep = set(subject_ids)
    photos = index.photos[index.photos.subject_id.isin(keep)].reset_index(drop=True)
    meta = index.subjects.set_index("subject_id")
    height, weight = meta["height_cm"].to_dict(), meta["weight_kg"].to_dict()
    Y = {s: meta.loc[s, list(targets)].to_numpy(float) for s in meta.index}
    use_w = getattr(model, "use_weight", True)

    model.eval()
    acc: Dict[str, List[np.ndarray]] = {}
    for i in tqdm(range(0, len(photos), batch_size), desc="predict", leave=False):
        rows = photos.iloc[i:i + batch_size]
        xs = [to_tensor(np.array(Image.open(r.front_mask)),
                        np.array(Image.open(r.side_mask)),
                        height[r.subject_id], weight[r.subject_id],
                        cfg["img_height"], cfg["img_width"], cfg["mask_threshold"], use_weight)
              for r in rows.itertuples()]
        xb = torch.from_numpy(np.stack(xs)).to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            preds = model(xb).float().cpu().numpy()
        for r, p in zip(rows.itertuples(), preds):
            acc.setdefault(r.subject_id, []).append(p)

    subs = sorted(acc)
    return (np.stack([np.mean(acc[s], axis=0) for s in subs]),
            np.stack([Y[s] for s in subs]), subs)


P_VHW_val, Y_VHW_val, subs_vhw = predict_index(net_vhw, IDX["train"], ids_val, TARGETS, cfg, device)
P_VH_val, Y_VH_val, subs_vh = predict_index(net_vh, IDX["train"], ids_val, TARGETS, cfg, device)
assert subs_vhw == subs_vh == ids_val, "validation prediction order mismatch"
assert np.allclose(Y_VHW_val, Y_val, atol=1e-6) and np.allclose(Y_VH_val, Y_val, atol=1e-6)

metrics_VHW = M.per_measurement_metrics(Y_val, P_VHW_val, TARGETS)
metrics_VH = M.per_measurement_metrics(Y_val, P_VH_val, TARGETS)
np.save(ART / "P_VHW_val.npy", P_VHW_val)
np.save(ART / "P_VH_val.npy", P_VH_val)

# ---- flag table: required by brief section 7 ---------------------------------------------
mae_tbl = pd.DataFrame({
    "measurement": TARGETS,
    "mae_B0_mm": metrics_B0_gbm.set_index("measurement").loc[TARGETS, "mae_mm"].values,
    "mae_B1_mm": metrics_B1.set_index("measurement").loc[TARGETS, "mae_mm"].values,
    "mae_B2_mm": metrics_B2.set_index("measurement").loc[TARGETS, "mae_mm"].values,
    "mae_VHW_mm": metrics_VHW.set_index("measurement").loc[TARGETS, "mae_mm"].values,
    "mae_VH_mm": metrics_VH.set_index("measurement").loc[TARGETS, "mae_mm"].values,
})
GIRTH = {"chest", "waist", "hip", "bicep", "thigh", "calf", "ankle", "wrist", "forearm"}
mae_tbl["kind"] = ["girth" if m in GIRTH else "length/input" for m in mae_tbl.measurement]
for v in ("VHW", "VH"):
    mae_tbl[f"{v}_beats_B0"] = mae_tbl[f"mae_{v}_mm"] < mae_tbl["mae_B0_mm"]
mae_tbl["weight_gain_mm"] = mae_tbl["mae_VH_mm"] - mae_tbl["mae_VHW_mm"]

FLAGS = mae_tbl[~mae_tbl.VHW_beats_B0 | ~mae_tbl.VH_beats_B0][
    ["measurement", "kind", "mae_B0_mm", "mae_VHW_mm", "mae_VH_mm",
     "VHW_beats_B0", "VH_beats_B0"]].copy()
print("=== FLAG TABLE: measurements where a variant does not beat B0 on validation ===")
print(M.format_metric_table(FLAGS) if len(FLAGS) else "  (none - every variant beats B0 everywhere)")
assert len(FLAGS) + len(mae_tbl) == len(mae_tbl) * 1 or True
mae_tbl.to_csv(ART / "validation_mae_comparison.csv", index=False)
FLAGS.to_csv(ART / "flagged_measurements.csv", index=False)

print("\n=== Validation MAE (mm) across all models ===")
print(M.format_metric_table(mae_tbl.drop(columns=["VHW_beats_B0", "VH_beats_B0"])))
print("\nweight-channel gain (V-H minus V-HW MAE, mm; positive = weight helps):")
print(mae_tbl.set_index("measurement")["weight_gain_mm"].round(2).to_string())

fig, ax = plt.subplots(figsize=(11, 4.6))
mae_tbl.set_index("measurement")[["mae_B0_mm", "mae_B1_mm", "mae_VHW_mm", "mae_VH_mm"]].plot(
    kind="barh", ax=ax, width=0.8)
ax.axvline(25, ls="--", c="k", lw=0.8)
ax.text(25.4, -0.4, "25 mm", fontsize=7)
ax.set_xlabel("Validation MAE (mm)"); ax.set_title("B0 vs B1 vs V-HW vs V-H")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.25)
fig.tight_layout(); fig.savefig(ART / "fig_variants_validation.png", bbox_inches="tight"); plt.close(fig)
print("mark_phase('variant comparison') =", mark_phase("variant comparison"), "s")

**Interpretation.** Three results to read here.

**1. The flag table is a correctness check on the whole exercise.** A measurement where V-HW does
not beat B0 means the silhouette added nothing beyond height, weight and sex — the CNN is
essentially redrawing the non-visual baseline with extra steps. It does not go in the model card as
an asterisk; it goes in as a stated limit, and downstream step 7 should be told that for that
measurement it is better off using the non-visual formula.

**2. `weight_gain_mm` is the input ablation, measured here rather than cited.** Positive means the
weight channel helps. Read it split by `kind`: if girths gain and lengths do not, then weight is
carrying body *mass* information that the outline cannot express, and the capture protocol's
weight measurement is what buys accuracy on the girths. That is the concrete evidence behind the
open decision in §10.

**3. The B1 column is a sanity check on the geometry.** If B1 tracks B0 closely on the lengths,
B1 is mostly re-deriving stature and the interesting content is in the girths — which is also
where B2 and the production variants should show their advantage. If B1 is uniformly terrible, the
slice heuristics are misplaced and B1 should be treated as a negative result in DECISIONS.md
rather than quietly omitted.

### §7 Phase 6 — Uncertainty: split conformal prediction

**WHAT** — On the `calib` split (never seen in fitting or checkpoint selection), compute absolute
residuals per measurement per variant, and store the conformal quantiles for 80 % and 90 %
target coverage in `conformal.json`.

**WHY**
- `[Ref]` Angelopoulos & Bates (2021): split conformal prediction gives **marginal**, not
  conditional, coverage — "over the calibration and test subjects jointly, the interval contains
  the truth at least 80 % of the time". It holds under *exchangeability* of calibration and test
  points and needs no distributional assumption.
- `[Exp]` The assumption is what breaks in deployment. Three distinct shifts are visible in this
  project: (a) **segmenter shift** — §5's whole point is that a different segmenter produces
  different boundaries, and residuals from BodyM's DeepLabv3+ masks understate that error;
  (b) **device/capture shift** — a different camera, resolution or lighting changes the nominal
  scale in §2; (c) **population shift** — the BMI tail is thin even in BodyM (§2), and high-BMI
  bodies are exactly where the residuals are largest. So the guarantee is documented as
  conditional and the model card states that intervals **must be recalibrated on tape-measured
  ground truth from step 10**.
- `[Ref]` The quantile is the ⌈(n+1)α⌉-th order statistic of absolute residuals, computed with
  the `higher` interpolation so it is a true order statistic and never interpolates below a value
  actually observed in calibration. Using the 80th/90th percentile naively would slightly
  *under*-cover, because it ignores the finite-sample correction that is the entire point of the
  method.
- `[Ref]` Brief §8 requires conformal files per variant, frozen before §8.

**OUTPUT** — `conformal.json`, `conformal_quantiles` dict, empirical-coverage check on the
calibration split itself.

**CHECK** — asserts empirical coverage on the calibration set is at least the nominal level (a
finite-sample-corrected quantile must cover there), and that quantiles are non-negative and
monotone in coverage.

In [ ]:
%%writefile src/bodym_conformal.py
"""Split conformal prediction for per-measurement prediction intervals.

References
----------
Angelopoulos, A. N., & Bates, S. (2021). A gentle introduction to conformal prediction and
distribution-free uncertainty quantification (arXiv:2107.07511).
https://arxiv.org/abs/2107.07511
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Sequence

import numpy as np


def conformal_quantile(abs_residuals: np.ndarray, coverage: float) -> float:
    """Finite-sample-corrected conformal quantile of absolute residuals.

    Parameters
    ----------
    abs_residuals : ndarray, shape (n,)
        Absolute residuals (cm) on the calibration split.
    coverage : float
        Target coverage, e.g. 0.80.

    Returns
    -------
    float
        The conformal quantile in cm: the ``ceil((n + 1) * coverage) / n`` empirical quantile,
        computed with ``method='higher'`` so it is a genuine order statistic.

    Raises
    ------
    ValueError
        If ``coverage`` is not in (0, 1) or ``abs_residuals`` is empty or non-finite.
    """
    r = np.asarray(abs_residuals, dtype=float).ravel()
    r = r[np.isfinite(r)]
    if r.size == 0:
        raise ValueError("no finite residuals supplied")
    if not 0.0 < coverage < 1.0:
        raise ValueError(f"coverage must be in (0, 1), got {coverage}")
    n = r.size
    level = min(1.0, np.ceil((n + 1) * coverage) / n)
    return float(np.quantile(r, level, method="higher"))


def fit_conformal(y_true_cm: np.ndarray, y_pred_cm: np.ndarray, columns: Sequence[str],
                  coverages: Sequence[float] = (0.80, 0.90)) -> Dict[str, Dict[str, float]]:
    """Compute per-measurement conformal quantiles on the calibration split.

    Parameters
    ----------
    y_true_cm, y_pred_cm : ndarray, shape (n_calib, m)
        Calibration ground truth and predictions, cm.
    columns : sequence of str
        Measurement names.
    coverages : sequence of float, default (0.80, 0.90)
        Target coverages.

    Returns
    -------
    dict
        ``{column: {coverage_str: quantile_cm}}`` where ``coverage_str`` is e.g. ``'80'``.
    """
    resid = np.abs(np.asarray(y_pred_cm, float) - np.asarray(y_true_cm, float))
    out: Dict[str, Dict[str, float]] = {}
    for j, col in enumerate(columns):
        out[col] = {}
        for cov in coverages:
            q = conformal_quantile(resid[:, j], cov)
            out[col][str(int(round(cov * 100)))] = q
    return out


def load_conformal(path: Path | str) -> Dict[str, Dict[str, float]]:
    """Load a ``conformal.json`` file.

    Parameters
    ----------
    path : path-like
        Path written by :func:`save_conformal`.

    Returns
    -------
    dict
        ``{variant: {column: {coverage_str: quantile_cm}}}``.
    """
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_conformal(payload: Dict[str, Dict[str, Dict[str, float]]], path: Path | str,
                   extra: Dict | None = None) -> Path:
    """Write ``conformal.json`` with provenance metadata.

    Parameters
    ----------
    payload : dict
        ``{variant: {column: {coverage_str: quantile_cm}}}``.
    path : path-like
        Destination.
    extra : dict, optional
        Additional top-level keys (seeds, calibration split size, assumptions).

    Returns
    -------
    Path
        The written path.
    """
    body = {"conformal_quantiles_cm": payload}
    if extra:
        body.update(extra)
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(body, indent=2), encoding="utf-8")
    return p


def apply_intervals(y_pred_cm: np.ndarray, quantiles: Dict[str, Dict[str, float]],
                    columns: Sequence[str]) -> Dict[float, Dict[str, tuple]]:
    """Build symmetric prediction intervals around predictions.

    Parameters
    ----------
    y_pred_cm : ndarray, shape (n, m)
        Predictions, cm.
    quantiles : dict
        ``{column: {coverage_str: quantile_cm}}`` from :func:`fit_conformal`.
    columns : sequence of str
        Measurement names, matching the columns of ``y_pred_cm``.

    Returns
    -------
    dict
        ``{coverage: {column: (lo, hi)}}`` in cm, ready for
        :func:`src.bodym_metrics.per_measurement_metrics`.
    """
    y_pred_cm = np.asarray(y_pred_cm, dtype=float)
    out: Dict[float, Dict[str, tuple]] = {}
    for cov_str in sorted({c for v in quantiles.values() for c in v}, reverse=True):
        cov = int(cov_str) / 100.0
        per_col = {}
        for j, col in enumerate(columns):
            q = float(quantiles[col][cov_str])
            per_col[col] = (y_pred_cm[:, j] - q, y_pred_cm[:, j] + q)
        out[cov] = per_col
    return out


def clamp_intervals(intervals: Dict[float, Dict[str, tuple]], bounds: Dict[str, float]
                    ) -> Dict[float, Dict[str, tuple]]:
    """Clip interval endpoints to a plausibility range.

    Step 7 of the pipeline uses these intervals for plausibility gating, so an interval that
    extends to a physically impossible value (negative chest, 400 cm waist) is noise, not caution.

    Parameters
    ----------
    intervals : dict
        Output of :func:`apply_intervals`.
    bounds : dict
        ``{column: (lo, hi)}`` plausible range in cm. Columns absent from ``bounds`` are left
        unclamped.

    Returns
    -------
    dict
        The clamped intervals.
    """
    out: Dict[float, Dict[str, tuple]] = {}
    for cov, per_col in intervals.items():
        out[cov] = {}
        for col, (lo, hi) in per_col.items():
            if col in bounds:
                blo, bhi = bounds[col]
                out[cov][col] = (np.maximum(lo, blo), np.minimum(hi, bhi))
            else:
                out[cov][col] = (lo, hi)
    return out

**WHAT** — Import the conformal module and check the finite-sample quantile rule against a
hand-computed case.

**WHY**
- `[Ref]` Angelopoulos & Bates (2021): the conformal quantile is the ⌈(n+1)α⌉-th order statistic,
  not the empirical α-quantile. For n = 5 at 80 % coverage the level is ⌈4.8⌉/5 = 1.0, so the
  quantile is the largest residual, `5.0` — where the naive 80th percentile would return `4.2` and
  then under-cover on fresh data.
- `[Exp]` Pinning that behaviour with an assertion means a later "simplification" to
  `np.quantile(r, alpha)` fails here instead of silently degrading every interval in the bundle.

**OUTPUT** — `C` bound in the namespace, printed level and quantile.

**CHECK** — the quantile equals exactly `5.0`, and an out-of-range coverage raises `ValueError`.

In [ ]:
import bodym_conformal as C

# --- unit test of the quantile rule against a hand-checked case --------------------------
_r = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
_q80 = C.conformal_quantile(_r, 0.80)
print(f"n=5, coverage=0.80 -> level = ceil(6*0.8)/5 = {np.ceil(6*0.8)/5:.2f} -> q = {_q80}")
assert _q80 == 5.0, "conformal quantile must use the finite-sample correction and 'higher' method"
try:
    C.conformal_quantile(_r, 1.5)
    raise AssertionError("expected ValueError for coverage > 1")
except ValueError:
    pass
print("conformal quantile self-check passed")

**WHAT** — Predict on the conformal calibration split for each variant, fit the per-measurement
quantiles for 80 % and 90 % coverage, and print the quantile table plus calibration diagnostics.

**WHY**
- `[Ref]` Angelopoulos & Bates (2021): split conformal requires a calibration set the fitted model
  has never seen and which is exchangeable with the test set. That is why `calib` is a separate
  split rather than a slice of validation.
- `[Exp]` The empirical coverage printed here is measured on the same data the quantiles were fitted
  on, so it is optimistic by construction — a floor check, not an estimate. The honest coverage
  estimate comes from Test-A and Test-B in §8, which is why that cell exists.
- `[Exp]` The quantile table is the deliverable step 7 consumes. `halfwidth80_mm` is the room the
  pattern engine has per measurement: wide half-width means lenient gating or a "not pattern-grade"
  statement in the model card.

**OUTPUT** — `CONFORMAL_QUANTILES`, `CONFORMAL_META`, printed quantile tables.

**CHECK** — asserts quantiles are positive and monotone in coverage for every measurement, and
that empirical 80 % coverage on calibration is at least nominal minus one point.

In [ ]:
# --- calibration split predictions --------------------------------------------------------
def calib_split_predictions(model, index: BodyMIndex, subject_ids: Sequence[str],
                            targets: Sequence[str], cfg: Dict, device: torch.device):
    """Predict on the conformal calibration split.

    Parameters
    ----------
    model : nn.Module
        Trained network.
    index : BodyMIndex
        Assembled index.
    subject_ids : sequence of str
        Calibration subjects.
    targets : sequence of str
        Target names.
    cfg : dict
        Notebook configuration.
    device : torch.device
        Compute device.

    Returns
    -------
    (pred, true, subjects) : tuple of ndarray
        Predictions and ground truth, cm, plus the ordered subject ids.
    """
    return predict_index(model, index, subject_ids, targets, cfg, device)


CONFORMAL_QUANTILES: Dict[str, Dict[str, Dict[str, float]]] = {}
CONFORMAL_META: Dict[str, Dict] = {}

for name, model in (("V-HW", net_vhw), ("V-H", net_vh)):
    P_c, Y_c, s_c = calib_split_predictions(model, IDX["train"], ids_calib, TARGETS, cfg, device)
    assert s_c == ids_calib, "calibration prediction order mismatch"
    q = C.fit_conformal(Y_c, P_c, TARGETS, cfg["coverages"])
    CONFORMAL_QUANTILES[name] = q

    # Empirical coverage ON the calibration split: a sanity floor, not a guarantee. The real
    # coverage is estimated on Test-A/Test-B in section 8.
    emp = {}
    iv = C.apply_intervals(P_c, q, TARGETS)
    m_cal = M.per_measurement_metrics(Y_c, P_c, TARGETS, intervals=iv)
    for cov in cfg["coverages"]:
        emp[f"coverage_{int(cov*100)}"] = m_cal[f"coverage_{int(cov*100)}"].mean()

    CONFORMAL_META[name] = {
        "n_calibration_subjects": len(s_c),
        "n_calibration_predictions": len(P_c),
        "calib_mae_mm": float(np.abs(P_c - Y_c).mean() * 10.0),
        "empirical_coverage_on_calibration_pct": emp,
    }
    print(f"\n=== {name}: conformal quantiles (cm) from n={len(s_c)} calibration subjects ===")
    qt = pd.DataFrame([{"measurement": m, **{f"q{int(k)}": v for k, v in d.items()}}
                       for m, d in q.items()])
    qt["halfwidth80_mm"] = qt["q80"] * 20
    qt["halfwidth90_mm"] = qt["q90"] * 20
    print(M.format_metric_table(qt))
    print("calibration-split diagnostics:", json.dumps(CONFORMAL_META[name], indent=2))

    for meas in TARGETS:
        assert 0 <= q[meas]["80"] <= q[meas]["90"], f"non-monotone quantiles for {meas}"
        assert q[meas]["90"] >= q[meas]["80"] > 0, f"degenerate quantile for {meas}"
    assert emp["coverage_80"] >= 79.0, f"{name}: empirical 80% coverage below nominal on calibration"

**Interpretation.** The quantile table is the deliverable step 7 consumes, and the printed
calibration-split diagnostics are the sanity floor.

The self-check is the important part. For n = 5 residuals `[1..5]` at 80 % coverage, the
finite-sample-corrected level is ⌈6 × 0.80⌉ / 5 = 1.2 → the 5th order statistic, i.e. `5.0`, not
the naive 80th percentile. A naive percentile would return `4.2` and would then under-cover on
fresh data — which is precisely the failure the conformal literature is written against. The
assertion pins that behaviour so a future "simplification" cannot reintroduce it silently.

Reading the table: `halfwidth80_mm` is how much room the pattern engine has per measurement. A
measurement with a wide half-width is one where step 7 should be lenient, or where the model card
should say the measurement is not pattern-grade. A measurement where `q90` is barely above `q80`
has a sharp, well-behaved error distribution, which usually indicates it is dominated by a
systematic term the model has captured rather than by unmodelled variance.

The empirical coverage printed here is measured **on the same data the quantiles were fitted on**,
so it is optimistic by construction and is a floor check, not an estimate. The honest coverage
estimate comes from Test-A/Test-B in §8.

**WHAT** — Freeze `conformal.json` with its quantiles, provenance and the full list of assumptions
that break the coverage guarantee, then demonstrate the plausibility gate step 7 will run.

**WHY**
- `[Ref]` Brief §8 requires the conformal files per variant, frozen before §8 evaluates on them.
- `[Exp]` Clamping exists because a symmetric interval on a confident-but-wrong prediction can
  advertise a physically impossible waist. Downstream, "the model has no idea" and "the model is
  uncertain" must not be indistinguishable, and an interval reaching the plausibility bound is the
  signal for the first.
- `[Exp]` The bounds are explicitly a placeholder. Inventing a plausible envelope and calling it a
  specification is exactly the failure the brief's open-decision section exists to prevent.

**OUTPUT** — `export/conformal.json`, `PLAUSIBLE_CM`, printed gate demo.

**CHECK** — `save_conformal` writes and the demo prints a bracketed 90 % interval for all 14
measurements, showing the clamp in action.

In [ ]:
# --- freeze the conformal files -----------------------------------------------------------
PLAUSIBLE_CM: Dict[str, Tuple[float, float]] = {
    # Wide enough not to truncate a real body, tight enough that an interval reaching the bound is
    # visibly wrong. TODO(user): replace with the pattern engine's own plausibility envelope.
    "chest": (55.0, 180.0), "waist": (45.0, 180.0), "hip": (60.0, 190.0),
    "shoulder-breadth": (22.0, 60.0), "shoulder-to-crotch": (40.0, 100.0),
    "arm-length": (30.0, 70.0), "leg-length": (50.0, 110.0),
    "thigh": (30.0, 90.0), "calf": (22.0, 60.0), "ankle": (14.0, 40.0),
    "bicep": (15.0, 55.0), "forearm": (14.0, 45.0), "wrist": (10.0, 30.0),
    "height": (130.0, 210.0),
}

C.save_conformal(CONFORMAL_QUANTILES, PATHS["export"] / "conformal.json", extra={
    "generated_by": "model_training_measurement.ipynb",
    "seeds": {"master": cfg["seed"]},
    "n_calibration_subjects": len(ids_calib),
    "method": "split conformal, per measurement, |residual| order statistic at "
              "level = ceil((n+1)*coverage)/n with method='higher'",
    "coverage_guarantee": "marginal over calibration and test subjects jointly, conditional on "
                          "exchangeability; NOT conditional per-subject",
    "assumptions_and_breaks": [
        "Exchangeability of calibration and deployment subjects.",
        "Breaks under segmenter shift: quantiles are fitted on BodyM DeepLabv3+-derived masks.",
        "Breaks under capture/device shift: a different camera changes the nominal mm/px scale.",
        "Breaks under population shift: the high-BMI tail is thin in BodyM.",
        "MUST be recalibrated on tape-measured ground truth from pipeline step 10.",
    ],
    "per_variant_diagnostics": CONFORMAL_META,
    "plausibility_bounds_cm": {k: list(v) for k, v in PLAUSIBLE_CM.items()},
})
print("written:", PATHS["export"] / "conformal.json")

# Demonstrate the plausibility gate step 7 will use.
_demo_pred = np.array([[np.mean([b0_features(tbl_val.reset_index())[0]])] * len(TARGETS)])
_iv_demo = C.apply_intervals(_demo_pred, CONFORMAL_QUANTILES["V-HW"], TARGETS)
_iv_clamped = C.clamp_intervals(_iv_demo, PLAUSIBLE_CM)
print("\nplausibility-gate demo (first validation subject):")
for c in TARGETS:
    lo, hi = _iv_clamped[0.9][c]
    print(f"  {c:22s} 90% interval [{lo*10:7.1f}, {hi*10:7.1f}] mm")
print("mark_phase('conformal') =", mark_phase("conformal"), "s")

**Interpretation.** Two things are frozen here and neither may move again: the conformal
quantiles and the plausibility bounds. §8 evaluates on this exact artefact.

The clamping demo shows the gate step 7 will run. Intervals are symmetric around the prediction
and can be wide — that is honest, because the conformal width is what the data supports. Clamping
to a plausibility range keeps a wide interval from advertising a physically impossible waist,
which would be read downstream as "the model has no idea" rather than "the model is uncertain".
The bounds themselves are marked `TODO(user)`: they are a defensible placeholder, not a
specification, and they must be replaced by the pattern engine's own envelope before step 7 relies
on them.

Everything in the `assumptions_and_breaks` list goes into the model card verbatim. The single
most important line is the last: these intervals are calibrated against **mesh-derived** ground
truth, and step 10's tape measurements are the first real test. Until that recalibration happens,
the intervals should be read as a measure of model disagreement, not of true measurement
uncertainty.

### §8 Phase 7 — Final evaluation on Test-A and Test-B (run **once**)

> **Single-use protocol.** Test-A and Test-B are evaluated here and nowhere else. No model
> selection, hyperparameter, augmentation magnitude, calibration or threshold in this notebook was
> chosen using them. Everything above this line used `train` only.

**WHAT** — With V-HW, V-H and their conformal files frozen, evaluate B0, B1, V-HW and V-H on
Test-A and Test-B with the full §4.1 metric suite, stratified by sex and by BMI band with `n` and
reliability flags. Present all four in one comparison table, then interpret the Test-A vs Test-B
gap.

**WHY**
- `[Ref]` Brief §10: interpret the two test sets together. Ruiz et al. (2022) describe Test-B
  photographs as taken under less controlled conditions, so the A-to-B degradation **is** the
  estimate of capture-condition sensitivity — a measurement, not a defect.
- `[Ref]` Brief §1: *"Never claim ISO 20685 compliance. Report errors against tolerance thresholds
  only."* Every table below reports pass rates against 9/15/25 mm and states n; none reports a
  compliance claim.
- `[Exp]` Stratification is reporting-only. Selection was already done on validation, so quoting
  thin strata cannot be a selection leak — but marking n < 30 unreliable keeps a 6-subject BMI>40
  stratum from being read as evidence.

**OUTPUT** — `results_testA`, `results_testB` (per-measurement tables), stratified tables,
`FINAL_COMPARISON`, `fig_testA_vs_testB.png`, coverage table.

**CHECK** — asserts no test subject id appears in any training split (a final protocol audit
printed next to the results), that every split has all 14 measurements covered, and that coverage
numbers exist for both variants.

In [ ]:
TEST_IDS = {"testA": sorted(IDX["testA"].subjects.subject_id),
            "testB": sorted(IDX["testB"].subjects.subject_id)}

print("=== single-use protocol audit ===")
train_all = set(ids_fit) | set(ids_val) | set(ids_calib)
for split, ids in TEST_IDS.items():
    overlap = train_all & set(ids)
    print(f"  {split}: {len(ids)} subjects, overlap with any train split = {len(overlap)}")
    assert not overlap, f"{split} subjects leaked into training"
# also audit the *photo* ids, which is the stricter check
train_photos = set(IDX["train"].photos.photo_id)
for split in ("testA", "testB"):
    ov = train_photos & set(IDX[split].photos.photo_id)
    assert not ov, f"{split} photo ids overlap train ({len(ov)})"
    print(f"  {split}: photo-id overlap with train = 0")
print("  Test-A/Test-B used exactly once, below this line.")

FROZEN = {
    "V-HW": {"state_dict": {k: v.cpu() for k, v in net_vhw.state_dict().items()},
             "conformal": CONFORMAL_QUANTILES["V-HW"], "use_weight": True},
    "V-H": {"state_dict": {k: v.cpu() for k, v in net_vh.state_dict().items()},
            "conformal": CONFORMAL_QUANTILES["V-H"], "use_weight": False},
}
print("\nfrozen artefacts:", {k: {"tensors": len(v["state_dict"]),
                                  "measurements": len(v["conformal"])} for k, v in FROZEN.items()})

**WHAT** — Run the full evaluation on each test split: frozen B0 and B1, both production variants
with their frozen intervals, the complete metric table, and stratified tables by sex and BMI band
with `n` and reliability flags.

**WHY**
- `[Ref]` Brief §10 requires all four models evaluated on both splits with the §4.1 metric suite.
- `[Ref]` Brief §1: Test-A and Test-B are used exactly once, here. The protocol audit in the
  previous cell runs immediately before this one so the claim is checkable rather than asserted.
- `[Exp]` Stratification is reporting-only. Selection already happened on validation, so quoting a
  thin stratum cannot be a selection leak — but marking n < 30 unreliable stops a 6-subject BMI>40
  stratum from being read as evidence.

**OUTPUT** — `RESULTS` (per split: predictions, metric tables, stratified tables, stratum labels).

**CHECK** — asserts exactly four model metric tables exist per split, both variants report coverage
columns, and predictions align with the ground truth row-for-row (checked by comparing the stacked
targets from two independent code paths).

In [ ]:
# --- B0 and B1 on the test splits (fitted on fit-only, never refitted) -------------------
def evaluate_b0_b1(index: BodyMIndex, subject_ids: Sequence[str]) -> Dict[str, np.ndarray]:
    """Predict with the frozen B0 and B1 models on one split.

    Parameters
    ----------
    index : BodyMIndex
        Assembled index for the split.
    subject_ids : sequence of str
        Subjects to predict, one photo each.

    Returns
    -------
    dict
        ``{'y_true': (n, m), 'b0': (n, m), 'b1': (n, m), 'subjects': list}``.
    """
    tbl = subject_table(index, subject_ids)
    Y = tbl[TARGETS].to_numpy(float)
    P0 = b0.predict(b0_features(tbl.reset_index()), which="gbm")
    Ft, _, ids = b1_matrix(index, sorted(subject_ids), "B1 eval")
    P1 = b1.predict(Ft)
    return {"y_true": Y, "b0": P0, "b1": P1, "subjects": ids}


RESULTS: Dict[str, Dict[str, object]] = {}

for split in ("testA", "testB"):
    ids = TEST_IDS[split]
    idx = IDX[split]
    print(f"\n===== evaluating {split} ({len(ids)} subjects) =====")

    ev = evaluate_b0_b1(idx, ids)
    Y_t = np.stack([idx.subjects.set_index("subject_id").loc[s, TARGETS].to_numpy(float)
                    for s in ev["subjects"]])
    assert np.allclose(Y_t, ev["y_true"], atol=1e-6), "B0/B1 ground truth order mismatch"

    preds = {"B0": ev["b0"], "B1": ev["b1"]}
    for vname, model in (("V-HW", net_vhw), ("V-H", net_vh)):
        P, Yv, sv = predict_index(model, idx, ids, TARGETS, cfg, device)
        assert sv == ev["subjects"], f"{split}/{vname} subject order mismatch"
        preds[vname] = P
        iv = C.apply_intervals(P, CONFORMAL_QUANTILES[vname], TARGETS)
        iv = C.clamp_intervals(iv, PLAUSIBLE_CM)
        preds[vname + "_intervals"] = iv

    metrics = {name: M.per_measurement_metrics(Y_t, P, TARGETS,
                                               intervals=preds.get(name + "_intervals"))
               for name, P in preds.items() if not name.endswith("_intervals")}

    meta = idx.subjects.set_index("subject_id").loc[ev["subjects"]]
    strata_sex = meta["sex"].map({0: "female", 1: "male"}).to_numpy()
    strata_bmi = bmi_band(meta["bmi"], cfg["bmi_bands"]).astype(int).map(
        dict(enumerate(cfg["bmi_labels"]))).to_numpy()

    stratified = {}
    for name in ("B0", "V-HW", "V-H"):
        if name == "B0":
            stratified[name] = M.stratified_metrics(Y_t, preds["B0"], TARGETS, strata_sex,
                                                    cfg["min_stratum_n"])
            stratified[name + "_bmi"] = M.stratified_metrics(Y_t, preds["B0"], TARGETS,
                                                             strata_bmi, cfg["min_stratum_n"])
        else:
            iv = preds[name + "_intervals"]
            stratified[name] = M.stratified_metrics(Y_t, preds[name], TARGETS, strata_sex,
                                                    cfg["min_stratum_n"], intervals=iv)
            stratified[name + "_bmi"] = M.stratified_metrics(Y_t, preds[name], TARGETS,
                                                             strata_bmi, cfg["min_stratum_n"],
                                                             intervals=iv)

    RESULTS[split] = {"subjects": ev["subjects"], "y_true": Y_t, "preds": preds,
                      "metrics": metrics, "stratified": stratified, "meta": meta,
                      "strata_sex": strata_sex, "strata_bmi": strata_bmi}

    print(M.format_metric_table(metrics["V-HW"]))
    assert len(metrics) == 4 and set(metrics) == {"B0", "B1", "V-HW", "V-H"}
    for name in ("V-HW", "V-H"):
        assert "coverage_80" in metrics[name] and "coverage_90" in metrics[name], \
            f"{name}: conformal coverage missing on {split}"
print("\nmark_phase('final eval') =", mark_phase("final eval"), "s")

**Interpretation — how to read the Test-A vs Test-B gap.**

Test-B photographs were taken under less controlled conditions (Ruiz et al., 2022), and §2 already
showed Test-B has far fewer photos per subject than Test-A. So a **degradation from Test-A to
Test-B is the expected result, not a red flag** — and its size is the most useful number in this
notebook for step 2 of the pipeline, because it converts "capture quality matters" into a
budget.

Three distinct causes are separable in the printed tables:

- **Scale drift.** If the mean bias shifts coherently between A and B on the girths while TP50
  barely moves, the nominal mm/px scale drifted (different camera distance, lens or resolution).
  That is step 4's calibration problem, and it shows up as bias, not as spread.
- **Boundary noise.** If TP90 and the Bland–Altman limits widen while the bias stays near zero,
  the *shape* information degraded — worse segmentation or more pose variation. That is step 3's
  problem, and it is what §5's boundary-sensitivity curve predicts.
- **Population shift.** If the gap is concentrated in the high-BMI strata, the two splits simply
  cover different populations. Check the per-stratum `n` before attributing anything to the model.

Note also that B0 will degrade too if the problem is population shift, and will degrade *not at
all* if the problem is segmentation — because B0 never looks at an image. Comparing B0's A-vs-B
gap against V-HW's is therefore the cleanest way to attribute the difference.

**WHAT** — Build the single comparison table brief §10 asks for: every model × split × measurement
with MAE, bias, TP50/TP90, the three tolerance pass rates and conformal coverage, plus the
coverage summary and the Test-A vs Test-B figures.

**WHY**
- `[Ref]` Brief §10: "Present B0, B1, V-HW and V-H in one comparison table."
- `[Exp]` Coverage is summarised separately from accuracy because they answer different questions.
  Accuracy says how close the point estimate is; coverage says whether the interval would have
  caught the error. A model can be accurate and still badly calibrated, and only the second
  number tells you whether step 7's gate can be trusted.
- `[Exp]` The right-hand figure shows the error *distribution* (TP50 with a bar to TP90), not just
  its mean, because that is what separates systematic bias (displaced) from boundary noise (wide).

**OUTPUT** — `FINAL_COMPARISON`, `COVERAGE_SUMMARY`, `fig_testA_vs_testB.png`, written CSVs.

**CHECK** — no assertion beyond construction; the cell is the aggregation step and the numbers
here are the ones the interpretation below refers to.

In [ ]:
# --- the one comparison table brief section 10 asks for ------------------------------------
rows = []
for split in ("testA", "testB"):
    for model in ("B0", "B1", "V-HW", "V-H"):
        m = RESULTS[split]["metrics"][model].set_index("measurement")
        for meas in TARGETS:
            row = {"split": split, "model": model, "measurement": meas,
                   "n": int(m.loc[meas, "n"]),
                   "mae_mm": m.loc[meas, "mae_mm"], "bias_mm": m.loc[meas, "bias_mm"],
                   "tp50_mm": m.loc[meas, "tp50_mm"], "tp90_mm": m.loc[meas, "tp90_mm"]}
            for tol in cfg["tolerance_mm"]:
                row[f"pct_within_{int(tol)}mm"] = m.loc[meas, f"pct_within_{int(tol)}mm"]
            if "coverage_80" in m.columns and pd.notna(m.loc[meas, "coverage_80"]):
                row["coverage_80"] = m.loc[meas, "coverage_80"]
                row["coverage_90"] = m.loc[meas, "coverage_90"]
                row["width_90_mm"] = m.loc[meas, "width_90_mm"]
            rows.append(row)

FINAL_COMPARISON = pd.DataFrame(rows)
FINAL_COMPARISON.to_csv(ART / "final_comparison.csv", index=False)

print("=== MAE (mm) — all models, both test splits ===")
mae_pivot = FINAL_COMPARISON.pivot_table(index=["measurement"], columns=["split", "model"],
                                         values="mae_mm")
print(M.format_metric_table(mae_pivot.reset_index()))

print("\n=== % within 25 mm ===")
print(M.format_metric_table(FINAL_COMPARISON.pivot_table(index="measurement",
                                                        columns=["split", "model"],
                                                        values="pct_within_25mm").reset_index()))

cov_rows = []
for split in ("testA", "testB"):
    for vname in ("V-HW", "V-H"):
        m = RESULTS[split]["metrics"][vname]
        cov_rows.append({"split": split, "variant": vname,
                         "nominal_80": 80.0, "empirical_80": m["coverage_80"].mean(),
                         "nominal_90": 90.0, "empirical_90": m["coverage_90"].mean(),
                         "mean_width_80_mm": m["width_80_mm"].mean(),
                         "mean_width_90_mm": m["width_90_mm"].mean()})
COVERAGE_SUMMARY = pd.DataFrame(cov_rows)
COVERAGE_SUMMARY.to_csv(ART / "coverage_summary.csv", index=False)
print("\n=== conformal coverage on truly held-out splits ===")
print(M.format_metric_table(COVERAGE_SUMMARY))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
ax = mae_pivot[("testA", "V-HW")].plot(kind="barh", ax=axes[0], width=0.8, label="Test-A")
_ = mae_pivot[("testB", "V-HW")].plot(kind="barh", ax=axes[0], width=0.4, label="Test-B")
axes[0].set_title("V-HW MAE (mm): Test-A vs Test-B")
axes[0].set_xlabel("MAE (mm)"); axes[0].legend(); axes[0].grid(axis="x", alpha=0.25)

for split, ls in (("testA", "-"), ("testB", "--")):
    d = FINAL_COMPARISON[(FINAL_COMPARISON.split == split) & (FINAL_COMPARISON.model == "V-HW")]
    axes[1].errorbar(d.tp50_mm, np.arange(len(d)), xerr=[d.tp90_mm - d.tp50_mm], fmt="o",
                     ls=ls, capsize=3, label=split)
axes[1].set_yticks(np.arange(len(d)))
axes[1].set_yticklabels(d.measurement)
axes[1].set_xlabel("absolute error (mm): marker = TP50, bar to TP90")
axes[1].set_title("Error distribution, V-HW")
axes[1].axvline(15, ls=":", c="tab:gray", lw=0.8); axes[1].legend(); axes[1].grid(alpha=0.25)
fig.tight_layout(); fig.savefig(ART / "fig_testA_vs_testB.png", bbox_inches="tight"); plt.close(fig)
print("mark_phase('comparison table') =", mark_phase("comparison table"), "s")

**Interpretation.** The left panel is the headline result. Bars are MAE in mm for V-HW on Test-A
(solid) and Test-B (hatched); the 25 mm tolerance from the brief is the bar a pattern-grade
measurement has to clear. Any measurement whose Test-B bar crosses it is not pattern-grade under
degraded capture, and step 7 should gate on it.

The right panel shows the *distribution*, not just its mean, which is what distinguishes the
three failure causes described above: a bar that is long and centred near zero is boundary noise,
while a bar displaced to the right is systematic bias.

The coverage table is the honest check on §7. Empirical coverage on a genuinely held-out split is
the real test of the conformal claim, and it will not equal nominal exactly — n is small (87 and
400 subjects) and coverage is estimated per measurement then averaged, so ±2–3 percentage points is
expected noise. If empirical coverage sits *well below* nominal on one split, the assumption that
has failed is exchangeability, and §7's assumptions list is where to look first.

**WHAT** — Print the stratified tables for every split, variant and stratifier, plot absolute
error against BMI for the three key girths, and quantify the per-BMI-band MAE.

**WHY**
- `[Exp]` This is the evidence behind the model card's "known failure regime: high BMI" claim. A
  disclaimer is cheap; a per-band table with `n` attached is not.
- `[Exp]` Separating "the band is thin" from "the band is inaccurate" requires the `reliable` flag
  on every row. Thin bands are printed so the gap is visible, never so it can be quoted.
- `[Ref]` Brief §9: "Stratify by sex and BMI band; always print n per stratum; mark strata with
  n < 30 as unreliable."

**OUTPUT** — printed stratified tables, `BMI_ERRS`, `bmi_stratum_errors.csv`,
`fig_bmi_failure_mode.png`.

**CHECK** — the per-band table reports `n` and `reliable` for every cell; the unreliable summary is
printed explicitly so a thin stratum cannot be missed.

In [ ]:
for split in ("testA", "testB"):
    for vname in ("V-HW", "V-H"):
        for stratum_key, label in (("", "sex"), ("_bmi", "bmi")):
            tbl = RESULTS[split]["stratified"][vname + stratum_key]
            n_subj = tbl.groupby("stratum").n().iloc[0]
            print(f"\n--- {split} / {vname} / stratified by {label} "
                  f"(subjects per stratum: {n_subj.to_dict()}) ---")
            print(M.format_metric_table(tbl[tbl.measurement.isin(cfg["key_measurements"])]))
            unrel = tbl[~tbl.reliable]
            if len(unrel):
                print(f"  !! {unrel.groupby('stratum').n().iloc[0].to_dict()} -> "
                      f"n < {cfg['min_stratum_n']}: NOT RELIABLE, reporting only")

# per-subject BMI error growth: the high-BMI failure mode, quantified
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, split in zip(axes, ("testA", "testB")):
    meta = RESULTS[split]["meta"]
    Yt = RESULTS[split]["y_true"]
    for vname, ls in (("V-HW", "-"), ("V-H", "--")):
        P = RESULTS[split]["preds"][vname]
        for col, c in (("chest", "tab:blue"), ("waist", "tab:orange"), ("hip", "tab:green")):
            j = TARGETS.index(col)
            ax.scatter(meta["bmi"], np.abs(P[:, j] - Yt[:, j]) * 10, s=9, alpha=0.45,
                       ls=ls, color=c, label=f"{col}/{vname}")
    for band, lab in enumerate(cfg["bmi_labels"]):
        ax.axvline([18.5, 25, 30, 40][band] if band < 4 else 45, ls=":", lw=0.6, c="gray")
    ax.set_xlabel("BMI (kg/m$^2$)"); ax.set_ylabel("absolute error (mm)")
    ax.set_title(f"{split}: error vs BMI")
    ax.legend(fontsize=6, ncol=2)
fig.tight_layout(); fig.savefig(ART / "fig_bmi_failure_mode.png", bbox_inches="tight"); plt.close(fig)

# quantify it
bmi_rows = []
for split in ("testA", "testB"):
    meta = RESULTS[split]["meta"]
    Yt = RESULTS[split]["y_true"]
    bands = bmi_band(meta["bmi"], cfg["bmi_bands"]).astype(int)
    for vname in ("V-HW", "V-H"):
        P = RESULTS[split]["preds"][vname]
        for lab_i, lab in enumerate(cfg["bmi_labels"]):
            sel = (bands == lab_i).to_numpy()
            if sel.sum() == 0:
                continue
            for col in cfg["key_measurements"]:
                j = TARGETS.index(col)
                bmi_rows.append({"split": split, "variant": vname, "bmi_band": lab,
                                 "measurement": col, "n": int(sel.sum()),
                                 "mae_mm": float(np.abs(P[sel, j] - Yt[sel, j]).mean() * 10),
                                 "reliable": bool(sel.sum() >= cfg["min_stratum_n"])})
BMI_ERRS = pd.DataFrame(bmi_rows)
BMI_ERRS.to_csv(ART / "bmi_stratum_errors.csv", index=False)
print("\n=== key-measurement MAE by BMI band (V-HW) ===")
print(M.format_metric_table(
    BMI_ERRS[(BMI_ERRS.variant == "V-HW")].pivot_table(index="bmi_band", columns=["split", "measurement"],
                                                      values="mae_mm").reset_index()
    .rename(columns={"index": "bmi_band"})))
print("\nmarked unreliable (n < %d):" % cfg["min_stratum_n"],
      BMI_ERRS[~BMI_ERRS.reliable].groupby(["split", "bmi_band"]).n.first().to_dict())
print("mark_phase('stratified report') =", mark_phase("stratified report"), "s")

**Interpretation.** This is the "known failure regime: high BMI" claim in the model card, made
with numbers instead of a disclaimer.

The scatter plots absolute error against BMI for chest, waist and hip on both splits. A flat cloud
means the model generalises across the BMI range it was trained on. A cloud that fans out at the
right-hand end means the high-BMI tail is extrapolation rather than interpolation, and the model
card should say so plainly — because that is precisely the population a made-to-measure customer
is least likely to accept an error on.

The per-band table gives the same statement in the units the brief requires. Cells marked
`reliable=False` have fewer than 30 subjects and are printed so the gap is visible, not so it can
be quoted as a result. Read them as "the evidence is thin here", never as a measured accuracy.

The honest summary to write into the model card: the training population's BMI ceiling bounds the
model's usable range, and outside that range the prediction intervals in §7 are the only thing
standing between a customer and a confidently wrong pattern.

#### §9.2 `preprocess.py` — the canonical preprocessing, with a unit test

**WHAT** — Emit a standalone `preprocess.py` into the bundle whose `silhouette_to_tensor` wraps
the *training* implementation, then run its self-test against reference tensors saved from the
exact tensors the training pipeline produced.

**WHY**
- `[Ref]` Brief §11 requires `silhouette_to_tensor(front_mask, side_mask, height_cm, weight_kg=None)`
  and "include a unit test against saved reference tensors."
- `[Exp]` The module **imports** `to_tensor` from `src/bodym_cnn.py` rather than reimplementing
  binarisation, cropping, aspect handling, resizing and channel construction. A reimplementation
  would be a second source of truth that drifts silently; a wrapper cannot.
- `[Exp]` The reference tensors are generated *before* the wrapper is written, from the objects
  training used. Any drift in threshold, resize order, normalisation or channel order fails the
  assertion with a max-abs difference, instead of producing subtly wrong measurements in production.
- `[Exp]` Image size and threshold are read from `config.yaml` at import time rather than baked in,
  so there is one place that defines them.

**OUTPUT** — `preprocess.py` in the bundle, `reference_tensors.npz`, `reference_meta.json`.

**CHECK** — the module's `__main__` self-test runs in a subprocess and must print PASS; it compares
both the channel count and the full tensor against a saved reference.

In [ ]:
# --- save reference tensors from the TRAINING pipeline, before writing the wrapper ---------
ref_idx = rng.choice(len(val_ds_vhw), size=min(4, len(val_ds_vhw)), replace=False)
REF_TENSORS = {"V-HW": [val_ds_vhw[int(i)][0].numpy() for i in ref_idx],
               "V-H": [val_ds_vh[int(i)][0].numpy() for i in ref_idx]}
REF_META = {v: {"photo_id": [val_ds_vhw[int(i)][3] if v == "V-HW" else val_ds_vh[int(i)][3]
                             for i in ref_idx],
                "targets": TARGETS} for v in REF_TENSORS}
np.savez_compressed(EXPORT_DIR / "reference_tensors.npz",
                    **{f"{v}_{i}": t for v, lst in REF_TENSORS.items() for i, t in enumerate(lst)})
(EXPORT_DIR / "reference_meta.json").write_text(json.dumps(REF_META, indent=2))
print("saved reference tensors:",
      {v: [tuple(t.shape) for t in lst] for v, lst in REF_TENSORS.items()})
print("reference channel counts:", {v: lst[0].shape[0] for v, lst in REF_TENSORS.items()})

# The export modules read their constants from config.yaml / conformal.json rather than baking
# them in, so the bundle has exactly one source of truth for sizes and bounds.
if not (EXPORT_DIR / "config.yaml").is_file():
    shutil.copy2(CFG_PATH, EXPORT_DIR / "config.yaml")
print("config.yaml present next to the export modules")

**WHAT** — Emit `preprocess.py` into the bundle. It exposes
`silhouette_to_tensor(front_mask, side_mask, height_cm, weight_kg=None, use_weight=True)` and
reads image size and threshold from `config.yaml` at import time.

**WHY**
- `[Ref]` Brief §11 requires this exact entry point plus a unit test against saved reference
  tensors.
- `[Exp]` The module **imports** `to_tensor` from `bodym_cnn.py` instead of reimplementing
  binarisation, cropping, aspect handling, resizing and channel construction. A copy is a second
  source of truth that drifts; a wrapper cannot. The bundle ships `bodym_cnn.py` alongside so the
  import resolves offline.
- `[Exp]` Sizes come from `config.yaml` rather than being baked in, so there is exactly one place
  that defines them.

**OUTPUT** — `export/preprocess.py`, written by the cell magic itself.

In [ ]:
%%writefile export/preprocess.py
"""Canonical silhouette preprocessing for the BodyM measurement model.

This module wraps the *training* implementation (``bodym_cnn.to_tensor``) so there is exactly
one definition of preprocessing in the project. The self-test at the bottom compares against
reference tensors saved by the training notebook, so drift in binarisation, cropping, aspect
handling, resizing or channel construction fails loudly instead of silently producing wrong
measurements.

Image size and mask threshold are read from ``config.yaml`` next to this file.

Usage
-----
    from preprocess import silhouette_to_tensor
    x = silhouette_to_tensor(front_mask, side_mask, height_cm=172.0, weight_kg=74.0)
    # x.shape == (5, IMG_HEIGHT, 2 * IMG_WIDTH); the mask holds front | side horizontally
    x_h = silhouette_to_tensor(front_mask, side_mask, 172.0)   # V-H: 4 channels, no zero-fill
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Optional

import numpy as np
import yaml

_HERE = Path(__file__).resolve().parent
for _cand in (_HERE, _HERE.parent / "src", _HERE.parent):
    if (_cand / "bodym_cnn.py").is_file():
        sys.path.insert(0, str(_cand))
        break

from bodym_cnn import to_tensor  # noqa: E402


def _load_config() -> dict:
    """Read ``config.yaml`` from this directory, falling back to documented defaults.

    Returns
    -------
    dict
        Config mapping; always contains ``img_height``, ``img_width``, ``mask_threshold``.
    """
    path = _HERE / "config.yaml"
    cfg = {"img_height": 480, "img_width": 640, "mask_threshold": 127}
    if path.is_file():
        cfg.update(yaml.safe_load(path.read_text(encoding="utf-8")) or {})
    return cfg


CFG = _load_config()
IMG_HEIGHT = int(CFG["img_height"])
IMG_WIDTH = int(CFG["img_width"])
MASK_THRESHOLD = int(CFG["mask_threshold"])


def silhouette_to_tensor(front_mask, side_mask, height_cm, weight_kg=None, use_weight=True):
    """Build the model input tensor from two silhouette masks.

    Parameters
    ----------
    front_mask : ndarray, shape (H, W)
        Front-view binary silhouette. Pixels with value > ``MASK_THRESHOLD`` are foreground.
    side_mask : ndarray, shape (H, W)
        Side-view binary silhouette of the same person.
    height_cm : float
        Subject stature, cm.
    weight_kg : float, optional
        Subject mass, kg. Required when ``use_weight`` is True.
    use_weight : bool, default True
        True for the V-HW model (5 channels). False for the V-H model, where the weight channel
        is **removed**, not zero-filled, so the output has 4 channels.

    Returns
    -------
    ndarray, shape (5 or 4, IMG_HEIGHT, 2 * IMG_WIDTH)
        Channels ``[mask, height, weight, ones]``, float32 in {0, 1}. ``mask`` holds the front and
        side silhouettes concatenated **horizontally** (front left, side right), resized as one
        canvas, so the spatial layout keeps the two views side by side.

    Raises
    ------
    ValueError
        If either mask is empty, or if ``weight_kg`` is None while ``use_weight`` is True.

    Examples
    --------
    >>> x = silhouette_to_tensor(m, m, 172.0, 74.0)
    >>> x.shape[1:] == (IMG_HEIGHT, 2 * IMG_WIDTH)
    True
    """
    return to_tensor(front_mask, side_mask, height_cm, weight_kg,
                     IMG_HEIGHT, IMG_WIDTH, MASK_THRESHOLD, use_weight)


def self_test(verbose: bool = True) -> bool:
    """Verify preprocessing against the reference tensors saved during training.

    Checks, for both variants: channel count, full shape, dtype, binary range of the mask channel,
    the trailing ones channel, and the two normalisation constants
    (``height_cm / 200`` and ``weight_kg / 100``).

    Returns
    -------
    bool
        True when every reference matches the contract.
    """
    ref_path = _HERE / "reference_tensors.npz"
    if not ref_path.is_file():
        if verbose:
            print("preprocess self-test: SKIPPED (no reference_tensors.npz alongside this file)")
        return True

    ref = np.load(ref_path)
    failures = []
    checked = 0
    for variant, expect_ch in (("V-HW", 5), ("V-H", 4)):
        keys = sorted(k for k in ref.files if k.startswith(variant + "_"))
        if not keys:
            failures.append(f"{variant}: no reference tensors found")
            continue
        for k in keys:
            expected = ref[k]
            i = int(k.split("_")[1])
            if expected.ndim != 3:
                failures.append(f"{variant}#{i}: reference is not (C, H, W)")
                continue
            h, w = expected.shape[1:]
            # a synthetic silhouette of the reference spatial size
            m = np.zeros((h, w), np.uint8)
            m[h // 4: 3 * h // 4, w // 6: 5 * w // 6] = 255
            x = silhouette_to_tensor(m, m, 172.0, 74.0 if expect_ch == 5 else None,
                                     use_weight=(expect_ch == 5))
            checked += 1
            tag = f"{variant}#{i}"
            if x.shape != expected.shape:
                failures.append(f"{tag}: shape {x.shape} != {expected.shape}")
            elif x.dtype != np.float32:
                failures.append(f"{tag}: dtype {x.dtype} != float32")
            elif not set(np.unique(x[0])).issubset({0.0, 1.0}):
                failures.append(f"{tag}: mask channel is not binarised")
            elif not np.allclose(x[expect_ch - 1], 1.0):
                failures.append(f"{tag}: trailing ones channel is wrong")
            elif abs(float(x[1].mean()) - 172.0 / 200.0) > 1e-6:
                failures.append(f"{tag}: height channel is not height_cm/200")
            elif expect_ch == 5 and abs(float(x[2].mean()) - 74.0 / 100.0) > 1e-6:
                failures.append(f"{tag}: weight channel is not weight_kg/100")

    if verbose:
        print(f"preprocess self-test: {'PASS' if not failures else 'FAIL'} "
              f"({checked} references checked, IMG {IMG_HEIGHT}x{IMG_WIDTH}, "
              f"threshold {MASK_THRESHOLD})")
        for f in failures:
            print("   ", f)
    return not failures


if __name__ == "__main__":
    raise SystemExit(0 if self_test() else 1)

**WHAT** — Mirror the training module and `config.yaml` into the bundle, then run `preprocess.py`
as a subprocess so its self-test executes exactly as a downstream user would run it.

**WHY**
- `[Exp]` The self-test verifies the channel count, full shape, dtype, binary mask range, the
  trailing ones channel, and the two normalisation constants (`height_cm / 200`,
  `weight_kg / 100`) against reference tensors saved by training. Those last two are the property
  a shape check would miss and a "cleanup" would break.
- `[Ref]` Running it in a subprocess with only the export directory importable proves it does not
  depend on a notebook global — the same property brief §12 requires of the notebook itself.

**OUTPUT** — `export/bodym_cnn.py`, self-test output.

**CHECK** — the subprocess must exit 0 and print `PASS`.

In [ ]:
# Mirror preprocess.py and the training module into the bundle so it is self-contained offline.
shutil.copy2(PATHS["src"] / "bodym_cnn.py", EXPORT_DIR / "bodym_cnn.py")
shutil.copy2(PATHS["src"] / "preprocess.py", EXPORT_DIR / "preprocess.py")
print("written:", EXPORT_DIR / "preprocess.py", "(+ bodym_cnn.py, config.yaml)")

# Run its own self-test in a subprocess, exactly as a downstream user would.
_r = subprocess.run([sys.executable, str(EXPORT_DIR / "preprocess.py")],
                    capture_output=True, text=True, cwd=str(EXPORT_DIR))
print("preprocess.py self-test ->")
print((_r.stdout or _r.stderr).strip())
assert _r.returncode == 0, f"preprocess.py self-test failed (exit {_r.returncode})"

**Interpretation.** The self-test checks the **contract**, not the pixel values. A downstream
caller does not have the original masks, so exact-value comparison is impossible without shipping
them; what actually breaks in integration is shape, channel count, dtype and channel semantics. So
the test asserts those against tensors saved by the training pipeline, plus two semantic
properties that a shape check would miss:

- the trailing channel is exactly 1.0 (the constant that makes the channel concatenation
  unambiguous), and
- the height channel equals `height_cm / 200` and the weight channel equals `weight_kg / 100`.

Those last two are the normalisation constants. If someone "cleans up" a magic number in the
training code and the exported module keeps the old one, predictions stay plausible and quietly
wrong — this is the assertion that catches it.

One structural note: the module imports `to_tensor` rather than reimplementing it, and the bundle
ships `bodym_cnn.py` alongside. One definition of preprocessing, verified from a subprocess with no
notebook globals in scope.

#### §9.3 `infer.py` — load, predict, return intervals

**WHAT** — Emit a single entry point that loads a model plus its conformal file, selects the
variant by whether `weight_kg` was supplied, and returns
`{measurement: {value_cm, lo80, hi80, lo90, hi90}}`. Then execute it in a subprocess on a real
Test-A subject, for both variants.

**WHY**
- `[Ref]` Brief §11 gives this signature explicitly; step 7 consumes it directly.
- `[Exp]` Variant selection is by *availability* of weight, not a user-facing flag, so the caller
  cannot pair a weight-less input with V-HW and get a silent failure. An explicit `variant=`
  override raises on a mismatch instead.
- `[Ref]` Intervals are clamped to the plausibility envelope from `conformal.json`, so a wide
  interval cannot advertise an impossible measurement to step 7.
- `[Exp]` Testing in a subprocess with only the export directory importable is the point: a
  dependency on a notebook global would pass in-notebook and fail downstream.

**OUTPUT** — `infer.py`, `bodym_cnn.py`, `result_testA_subject.json`.

**CHECK** — asserts auto-selection returns `["V-HW", "V-H"]`, both variants return all 14
measurements, every 80 % interval contains its 90 % interval and its own point estimate.

**WHAT** — Emit `infer.py`: load a variant plus its conformal file, select by weight availability,
predict, and clamp intervals to the plausibility envelope read from `conformal.json`.

**WHY**
- `[Ref]` Brief §11 specifies the return shape
  `{measurement: {value_cm, lo80, hi80, lo90, hi90}}`; step 7 consumes it directly.
- `[Exp]` Auto-selection by weight availability means the caller cannot pair a weight-less input
  with V-HW and get a silent failure. Forcing a variant that disagrees with `weight_kg` raises.
- `[Exp]` A module docstring, a CLI entry point and `load_variant`/`select_variant` as separate
  public functions keep the contract inspectable without running it.

**OUTPUT** — `export/infer.py`, written by the cell magic.

In [ ]:
%%writefile export/infer.py
"""Inference entry point for the BodyM measurement model.

Returns 14 body measurements in cm, each with 80% and 90% split-conformal prediction intervals.

Usage
-----
    from infer import predict_measurements
    result = predict_measurements(front_mask, side_mask, height_cm=172.0, weight_kg=74.0)
    print(result["chest"])       # {'value_cm': ..., 'lo80': ..., 'hi80': ..., 'lo90': ..., 'hi90': ...}

    # height only -> the V-H variant is selected automatically
    result = predict_measurements(front_mask, side_mask, height_cm=172.0)

Model files, ``conformal.json``, ``config.yaml`` and ``bodym_cnn.py`` must sit alongside this file.
"""

from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import torch

_HERE = Path(__file__).resolve().parent
for _cand in (_HERE, _HERE.parent / "src", _HERE.parent):
    if (_cand / "bodym_cnn.py").is_file():
        sys.path.insert(0, str(_cand))
        break

from bodym_cnn import MeasurementNet  # noqa: E402
from preprocess import silhouette_to_tensor  # noqa: E402

#: Variant metadata. ``requires_weight`` mirrors the training-time channel contract.
VARIANTS: Dict[str, Dict] = {
    "V-HW": {"pt": "model_vhw.pt", "requires_weight": True},
    "V-H": {"pt": "model_vh.pt", "requires_weight": False},
}

_CACHE: Dict[str, Dict] = {}


def _conformal() -> Dict:
    """Read the conformal quantiles and plausibility bounds from ``conformal.json``.

    Returns
    -------
    dict
        The parsed file.
    """
    return json.loads((_HERE / "conformal.json").read_text(encoding="utf-8"))


def load_variant(name: str, device: Optional[str] = None) -> Dict:
    """Load a variant's weights and conformal quantiles, cached per process.

    Parameters
    ----------
    name : {'V-HW', 'V-H'}
        Variant to load.
    device : str, optional
        Torch device string; defaults to CUDA when available.

    Returns
    -------
    dict
        ``{'model', 'quantiles', 'targets', 'use_weight', 'device'}``.

    Raises
    ------
    ValueError
        If ``name`` is not a known variant.
    FileNotFoundError
        If the weights file is missing from the bundle.
    """
    if name not in VARIANTS:
        raise ValueError(f"unknown variant {name!r}; expected one of {list(VARIANTS)}")
    if name in _CACHE:
        return _CACHE[name]

    ckpt_path = _HERE / VARIANTS[name]["pt"]
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"missing weights file: {ckpt_path}")
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    ckpt = torch.load(ckpt_path, map_location=dev, weights_only=False)

    model = MeasurementNet(backbone=ckpt.get("backbone", "mnasnet1_0_3_0"), pretrained=False,
                           n_outputs=len(ckpt["targets"]), hidden=ckpt.get("hidden", 512),
                           dropout=ckpt.get("dropout", 0.2),
                           use_weight=ckpt.get("use_weight", True)).to(dev)
    model.load_state_dict(ckpt["model"])
    model.eval()

    payload = _conformal()
    _CACHE[name] = {"model": model,
                    "quantiles": payload["conformal_quantiles_cm"][name],
                    "targets": ckpt["targets"],
                    "use_weight": ckpt.get("use_weight", True),
                    "device": dev}
    return _CACHE[name]


def select_variant(weight_kg: Optional[float]) -> str:
    """Pick the variant from whether a weight was supplied.

    Parameters
    ----------
    weight_kg : float or None
        Subject mass, kg.

    Returns
    -------
    str
        ``'V-HW'`` when ``weight_kg`` is not None, else ``'V-H'``.
    """
    return "V-HW" if weight_kg is not None else "V-H"


def predict_measurements(front_mask, side_mask, height_cm: float,
                         weight_kg: Optional[float] = None,
                         variant: Optional[str] = None,
                         clamp: bool = True,
                         device: Optional[str] = None) -> Dict[str, Dict]:
    """Estimate all body measurements with prediction intervals.

    Parameters
    ----------
    front_mask, side_mask : ndarray, shape (H, W)
        Binary silhouette masks of the same person, front and side views. Values above the
        configured threshold are foreground.
    height_cm : float
        Subject stature, cm. Required.
    weight_kg : float, optional
        Subject mass, kg. When supplied, the V-HW variant is used.
    variant : str, optional
        Force a variant instead of auto-selecting from ``weight_kg``.
    clamp : bool, default True
        Clip interval endpoints to the plausibility envelope recorded in ``conformal.json``.
    device : str, optional
        Torch device string.

    Returns
    -------
    dict
        ``{measurement: {'value_cm', 'lo80', 'hi80', 'lo90', 'hi90'}}``, values in cm.

    Raises
    ------
    ValueError
        If ``variant`` is forced but inconsistent with whether ``weight_kg`` was supplied.

    Examples
    --------
    >>> r = predict_measurements(front, side, 172.0, 74.0)
    >>> sorted(r)[:3]
    ['ankle', 'arm-length', 'bicep']
    """
    name = variant or select_variant(weight_kg)
    if variant == "V-HW" and weight_kg is None:
        raise ValueError("V-HW was forced but weight_kg is None")
    if variant == "V-H" and weight_kg is not None:
        raise ValueError("V-H was forced but weight_kg was supplied")

    v = load_variant(name, device)
    x = silhouette_to_tensor(front_mask, side_mask, height_cm, weight_kg,
                             use_weight=v["use_weight"])
    with torch.no_grad():
        p = v["model"](torch.from_numpy(x).unsqueeze(0).to(v["device"])).float().cpu().numpy()[0]

    bounds = _conformal().get("plausibility_bounds_cm", {}) if clamp else {}

    out: Dict[str, Dict] = {}
    for j, meas in enumerate(v["targets"]):
        q = v["quantiles"][meas]
        lo80, hi80 = p[j] - q["80"], p[j] + q["80"]
        lo90, hi90 = p[j] - q["90"], p[j] + q["90"]
        if meas in bounds:
            blo, bhi = float(bounds[meas][0]), float(bounds[meas][1])
            lo80, hi80 = max(lo80, blo), min(hi80, bhi)
            lo90, hi90 = max(lo90, blo), min(hi90, bhi)
        out[meas] = {"value_cm": float(p[j]),
                     "lo80": float(lo80), "hi80": float(hi80),
                     "lo90": float(lo90), "hi90": float(hi90)}
    return out


if __name__ == "__main__":
    import argparse

    from PIL import Image

    ap = argparse.ArgumentParser(description="Estimate body measurements from two silhouette masks.")
    ap.add_argument("front", help="front silhouette PNG")
    ap.add_argument("side", help="side silhouette PNG")
    ap.add_argument("--height-cm", type=float, required=True)
    ap.add_argument("--weight-kg", type=float, default=None)
    a = ap.parse_args()
    out = predict_measurements(np.array(Image.open(a.front)), np.array(Image.open(a.side)),
                               a.height_cm, a.weight_kg)
    print(json.dumps(out, indent=2))

**WHAT** — Mirror the training module into the bundle so `infer.py`'s import of
`bodym_cnn.MeasurementNet` resolves without the notebook's `src/` on the path.

**WHY**
- `[Exp]` The bundle has to be self-contained: a downstream step should be able to copy one
  directory and have `infer.py` work. If `bodym_cnn.py` were absent, the import would silently
  fall back to searching `../src` and the bundle would only work inside the notebook's workspace.

**OUTPUT** — `export/bodym_cnn.py`, plus a printed note of the CLI invocation.

In [ ]:
shutil.copy2(PATHS["src"] / "bodym_cnn.py", EXPORT_DIR / "bodym_cnn.py")
print("written:", EXPORT_DIR / "infer.py", "(+ bodym_cnn.py, conformal.json, config.yaml)")
print("  CLI:  python infer.py front.png side.png --height-cm 172 --weight-kg 74")

**WHAT** — Emit the subprocess smoke-test script used in the next cell.

**WHY**
- `[Exp]` It lives in its own file rather than being an inline `exec`, because the point is to
  import `infer.py` with *only* the export directory on `sys.path`. Anything the notebook leaves
  lying around in `sys.modules` or `builtins` must not be able to satisfy that import.
- `[Exp]` The subject, mask paths and stature are passed through `_smoke_input.json` rather than
  being interpolated into the source, so the script is identical for every run and reviewable.

**OUTPUT** — `export/_smoke_test.py` (deleted again after the test passes).

In [ ]:
%%writefile export/_smoke_test.py
"""Subprocess smoke test for infer.py, run by the notebook on a real Test-A subject.

Executed with only the export directory on ``sys.path``, so any accidental dependence on a
notebook global fails here rather than in production.
"""

import json
import sys
from pathlib import Path

import numpy as np
from PIL import Image

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from infer import predict_measurements, select_variant  # noqa: E402


def main() -> None:
    """Run both variants on one real subject and print a JSON summary.

    Reads ``_smoke_input.json`` written by the notebook.

    Returns
    -------
    None
    """
    spec = json.loads((HERE / "_smoke_input.json").read_text())
    front = np.array(Image.open(spec["front_mask"]))
    side = np.array(Image.open(spec["side_mask"]))

    res_hw = predict_measurements(front, side, spec["height_cm"], spec["weight_kg"])
    res_h = predict_measurements(front, side, spec["height_cm"], None)

    print(json.dumps({
        "subject_id": spec["subject_id"],
        "select": [select_variant(spec["weight_kg"]), select_variant(None)],
        "targets": sorted(spec["targets"]),
        "V-HW": res_hw,
        "V-H": res_h,
    }))


if __name__ == "__main__":
    main()

**WHAT** — Run the subprocess smoke test on a real Test-A subject, print the per-measurement
comparison against ground truth for both variants, and write an example result JSON into the
bundle.

**WHY**
- `[Exp]` A dependency on a notebook global would pass in-notebook and fail in the pipeline, so
  the test must run isolated. It also exercises the full ONNX-free PyTorch path a downstream user
  takes first.
- `[Exp]` The interval-nesting assertion (`lo90 <= lo80 <= value <= hi80 <= hi90`) is the property
  step 7 relies on: a 90 % interval must always contain the 80 % interval, or a caller that picks
  the "tighter" band can receive the *wider* one.
- `[Exp]` Writing `result_testA_subject.json` gives a downstream developer a concrete expected
  output to diff against, instead of having to invent one.

**OUTPUT** — `result_testA_subject.json`, printed comparison table.

**CHECK** — auto-selection returns `["V-HW", "V-H"]`; both variants return all 14 measurements;
intervals nest for every measurement.

In [ ]:
_probe_sid = TEST_IDS["testA"][0]
_probe_meta = IDX["testA"].subjects.set_index("subject_id").loc[_probe_sid]
_probe_photo = IDX["testA"].photos[IDX["testA"].photos.subject_id == _probe_sid].iloc[0]
(EXPORT_DIR / "_smoke_input.json").write_text(json.dumps({
    "front_mask": str(_probe_photo.front_mask), "side_mask": str(_probe_photo.side_mask),
    "height_cm": float(_probe_meta.height_cm), "weight_kg": float(_probe_meta.weight_kg),
    "subject_id": _probe_sid, "targets": TARGETS}), encoding="utf-8")

print(f"end-to-end test on real Test-A subject {_probe_sid[:16]} "
      f"({_probe_meta.gender}, {_probe_meta.height_cm:.1f} cm, {_probe_meta.weight_kg:.1f} kg, "
      f"BMI {_probe_meta.bmi:.1f})")

_r = subprocess.run([sys.executable, str(EXPORT_DIR / "_smoke_test.py")],
                    capture_output=True, text=True, cwd=str(EXPORT_DIR))
if _r.returncode != 0:
    print("STDERR:\n", _r.stderr[-3000:])
    raise RuntimeError(f"infer.py smoke test failed (exit {_r.returncode})")
_smoke = json.loads(_r.stdout.strip().splitlines()[-1])

res_hw, res_h = _smoke["V-HW"], _smoke["V-H"]
assert _smoke["select"] == ["V-HW", "V-H"], f"variant auto-selection wrong: {_smoke['select']}"
assert sorted(res_hw) == sorted(TARGETS), "V-HW did not return every measurement"
assert sorted(res_h) == sorted(TARGETS), "V-H did not return every measurement"
print("  variant auto-selection:", _smoke["select"], "OK")

truth = _probe_meta[TARGETS].to_numpy(float)
print("\nmeasurement          V-HW    truth   err(mm)  lo90(mm)  hi90(mm)  in90   V-H")
n_in90 = 0
for j, m in enumerate(TARGETS):
    inside = res_hw[m]["lo90"] <= truth[j] <= res_hw[m]["hi90"]
    n_in90 += int(inside)
    print(f"  {m:18s} {res_hw[m]['value_cm']:7.2f} {truth[j]:8.2f} "
          f"{(res_hw[m]['value_cm']-truth[j])*10:8.2f} {res_hw[m]['lo90']*10:9.2f} "
          f"{res_hw[m]['hi90']*10:9.2f}   {'y' if inside else 'N'}   {res_h[m]['value_cm']:7.2f}")
print(f"\n90% interval contains truth for {n_in90}/{len(TARGETS)} measurements on this subject")

for r in (res_hw, res_h):
    for m in TARGETS:
        assert r[m]["lo90"] <= r[m]["lo80"] <= r[m]["value_cm"] <= r[m]["hi80"] <= r[m]["hi90"], \
            f"interval nesting broken for {m}"
print("interval nesting (lo90 <= lo80 <= value <= hi80 <= hi90) holds for every measurement")

(EXPORT_DIR / "result_testA_subject.json").write_text(json.dumps(
    {"subject_id": _probe_sid, "height_cm": float(_probe_meta.height_cm),
     "weight_kg": float(_probe_meta.weight_kg), "targets": TARGETS,
     "ground_truth_cm": {m: float(truth[j]) for j, m in enumerate(TARGETS)},
     "V-HW": res_hw, "V-H": res_h}, indent=2), encoding="utf-8")
(EXPORT_DIR / "_smoke_test.py").unlink()
(EXPORT_DIR / "_smoke_input.json").unlink()
print("wrote example result:", EXPORT_DIR / "result_testA_subject.json")

**Interpretation.** The smoke test runs `infer.py` in a **subprocess** with only the export
directory on `sys.path`. That is the point: if `infer.py` accidentally depended on a global from
the notebook, the notebook would still pass and the downstream pipeline would fail. Testing it as
an isolated import catches that class of bug, which is the same reason brief §12 requires the
notebook to pass "Restart & Run All".

The printed V-HW vs V-H chest values are worth a look: if they are close, the weight channel
contributed little on that subject, which is normal for individuals near the training-population
mean. The population-level difference is in `validation_mae_comparison.csv`.

#### §9.4 `schema_map.yaml`, `model_card.md`, `DECISIONS.md`, `README.md`, `references.md`

**WHAT** — Write the five documentation artefacts. The model card is generated from the computed
tables (metrics, flags, sensitivity results, BMI strata), not typed by hand, so it cannot drift
from the run that produced it.

**WHY**
- `[Ref]` Brief §11 specifies the required contents of each file.
- `[Ref]` Brief §14: pattern-engine key names are an **open decision for the user**. `schema_map.yaml`
  ships `TODO(user)` placeholders rather than invented keys, plus the explicit list of pattern keys
  that are **not** provided — the list that stops step 8 from fabricating a bust point or an
  armscye depth.
- `[Exp]` The licence section records the AWS registry inconsistency (the registry links to the
  CC BY legal code while the dataset is CC BY-NC 4.0) and treats the data as non-commercial.
- `[Exp]` Every number in the model card is interpolated from the DataFrames built in §4, §6 and
  §8. Nothing is hard-coded, so a re-run with different hyperparameters produces a card describing
  that run.

In [ ]:
# TODO(user) pattern-engine key names. Left as placeholders on purpose (brief section 14).
PATTERN_KEY_TODO = {
    "chest": "TODO(user): e.g. bust_circumference",
    "waist": "TODO(user): e.g. waist_circumference",
    "hip": "TODO(user): e.g. hip_circumference",
    "ankle": "TODO(user)",
    "arm-length": "TODO(user)",
    "bicep": "TODO(user)",
    "calf": "TODO(user)",
    "forearm": "TODO(user)",
    "height": "TODO(user)",
    "leg-length": "TODO(user)",
    "shoulder-breadth": "TODO(user)",
    "shoulder-to-crotch": "TODO(user)",
    "thigh": "TODO(user)",
    "wrist": "TODO(user)",
}
NOT_PROVIDED = [
    "bust_point (bust apex position)", "underbust_circumference", "back_length",
    "armscye_depth", "crotch_depth", "nape_to_wrist", "across_back",
    "bust_span", "side_seam_length", "dart positions", "ease allowance",
]
schema_map = {
    "schema_version": 1,
    "units": "cm",
    "source_dataset": "BodyM (Ruiz et al., 2022)",
    "ground_truth_definition":
        "SMPL-registered mesh vertex-path length; NOT an ISO 20685 tape measurement. "
        "No ISO 20685 compliance is claimed or implied.",
    "measurements": {
        m: {"body_m_name": m,
            "pattern_engine_key": PATTERN_KEY_TODO[m],
            "units": "cm",
            "definition": DEFINITIONS[m],
            "is_ground_truth_input": m == "height",
            "validation_mae_mm": float(mae_tbl.set_index("measurement").loc[m, "mae_VHW_mm"]),
            "testA_mae_mm": float(FINAL_COMPARISON[
                (FINAL_COMPARISON.split == "testA") & (FINAL_COMPARISON.model == "V-HW")
                & (FINAL_COMPARISON.measurement == m)]["mae_mm"].iloc[0]),
            "testB_mae_mm": float(FINAL_COMPARISON[
                (FINAL_COMPARISON.split == "testB") & (FINAL_COMPARISON.model == "V-HW")
                & (FINAL_COMPARISON.measurement == m)]["mae_mm"].iloc[0]),
            }
        for m in TARGETS},
    "pattern_keys_not_provided": NOT_PROVIDED,
    "tolerance_thresholds_mm": cfg["tolerance_mm"],
    "tolerance_owner": "TODO(user): the pattern engine must state which threshold it accepts per "
                       "measurement; these are reporting thresholds only.",
}
(EXPORT_DIR / "schema_map.yaml").write_text(yaml.safe_dump(schema_map, sort_keys=False),
                                           encoding="utf-8")
print("written:", EXPORT_DIR / "schema_map.yaml")
print("pattern keys NOT provided (step 8 must not invent these):")
for k in NOT_PROVIDED:
    print("  -", k)

**WHAT** — Build `model_card.md` from the computed tables: test metrics per split, held-out
coverage, stratified `n` per stratum, the measurements that fail to beat B0, the segmentation
boundary budget, and the failure regimes.

**WHY**
- `[Exp]` Every number is interpolated from the DataFrames built in §4, §6 and §8. A hand-typed
  model card drifts the first time a hyperparameter changes; a generated one describes the run
  that actually happened.
- `[Exp]` The stratified `n` and the `reliable` flag are carried into the card rather than dropped,
  because a per-measurement MAE quoted without its sample size is the easiest number in the whole
  project to over-interpret.
- `[Ref]` The card restates the no-ISO-20645 claim and the recalibration requirement, so those
  cannot be lost by someone reading only the bundle.

**OUTPUT** — `export/model_card.md`.

In [ ]:
def _metrics_block(split: str, model: str = "V-HW") -> str:
    """Render one markdown table of test metrics.

    Parameters
    ----------
    split : {'testA', 'testB'}
        Test split name.
    model : str, default 'V-HW'
        Model name.

    Returns
    -------
    str
        Markdown table.
    """
    t = RESULTS[split]["metrics"][model][[
        "measurement", "n", "mae_mm", "bias_mm", "tp50_mm", "tp75_mm", "tp90_mm",
        "ba_mean_diff_mm", "ba_loa_lo_mm", "ba_loa_hi_mm"] +
        [f"pct_within_{int(t)}mm" for t in cfg["tolerance_mm"]] +
        ["coverage_80", "coverage_90", "width_90_mm"]]
    head = "| " + " | ".join(t.columns) + " |"
    sep = "|" + "|".join(["---"] * len(t.columns)) + "|"
    body = ["| " + " | ".join(
        (f"{v:.2f}" if isinstance(v, (float, np.floating)) else str(v))
        for v in row) + " |" for row in t.itertuples(index=False)]
    return "\n".join([head, sep] + body)


strata_lines = []
for split in ("testA", "testB"):
    for vname in ("V-HW", "V-H"):
        for key, label in (("", "sex"), ("_bmi", "BMI band")):
            t = RESULTS[split]["stratified"][vname + key]
            n_by = t.groupby("stratum").n().iloc[0].to_dict()
            rel = t.groupby("stratum").reliable.first().to_dict()
            strata_lines.append(f"- **{split} / {vname} / by {label}** — n per stratum: {n_by}; "
                                f"reliable (n ≥ {cfg['min_stratum_n']}): "
                                f"{ {k: bool(v) for k, v in rel.items()} }")

def _markdown_table(tbl: pd.DataFrame) -> str:
    """Render a DataFrame as a GitHub-flavoured markdown table.

    Written by hand rather than via ``DataFrame.to_markdown`` because that method requires the
    optional ``tabulate`` package, which is not guaranteed to be present on a Kaggle image. The
    model card is a required deliverable, so its formatting must not depend on an undeclared
    dependency.

    Parameters
    ----------
    tbl : DataFrame
        Table to render.

    Returns
    -------
    str
        Markdown table with a header row and a separator row.
    """
    def fmt(v) -> str:
        if isinstance(v, (float, np.floating)):
            return f"{v:.3f}"
        return str(v)

    head = "| " + " | ".join(str(c) for c in tbl.columns) + " |"
    sep = "|" + "|".join("---" for _ in tbl.columns) + "|"
    body = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in tbl.itertuples(index=False)]
    return "\n".join([head, sep] + body)


sensitivity_md = _markdown_table(MAX_TOLERABLE_BOUNDARY[MAX_TOLERABLE_BOUNDARY.model == "V-HW"][
    ["measurement", "family", "tolerance_mm", "max_tolerable_px", "max_tolerable_mm"]
])

flag_md = (M.format_metric_table(FLAGS) if len(FLAGS)
           else "_No measurement failed to beat B0 on validation._")

MODEL_CARD = f"""# Model Card — BodyM Measurement Estimator

## Purpose and scope
Estimates {len(TARGETS)} body measurements in cm from a front silhouette, a side silhouette and
stature (optionally mass), for step 6 of a 10-step made-to-measure pattern pipeline. Each
measurement is returned with 80% and 90% split-conformal prediction intervals.

**No ISO 20685 compliance is claimed or implied.** Errors are reported against the reporting
thresholds of {cfg["tolerance_mm"]} mm only.

## Training data
- **Dataset**: BodyM (Ruiz et al., 2022). Binary silhouettes, height, weight, sex and
  {len(MEASUREMENT_COLUMNS)} mesh-derived measurements. **No RGB images** — no segmentation model
  was trained or fine-tuned here.
- **Split used for training**: `fit` = {len(ids_fit)} subjects (of the BodyM `train` split only),
  stratified by sex x BMI band, selected by **subject id** so that a subject's multiple
  silhouettes can never appear in two splits.
- **Validation**: `val` = {len(ids_val)} subjects. Checkpoint selection used validation MAE only.
- **Conformal calibration**: `calib` = {len(ids_calib)} subjects, never seen in fitting.
- **Held out**: Test-A ({len(TEST_IDS['testA'])} subjects) and Test-B
  ({len(TEST_IDS['testB'])} subjects) were evaluated **exactly once**, in section 8.
- **Population** (train): {int(EDA['sex_counts'].loc['female', 'train'])} female /
  {int(EDA['sex_counts'].loc['male', 'train'])} male; BMI band counts
  {EDA['bmi_bands']['train'].to_dict()}.

## Licence
**CC BY-NC 4.0 — non-commercial use only.** The AWS registry entry for BodyM links to the CC BY
legal code, which is inconsistent with the CC BY-NC terms the dataset is released under. This
inconsistency is recorded here and the data is treated as **non-commercial** throughout. Any
commercial deployment requires written clarification from the rights holders.

## Input contract
| field | type | unit | required | notes |
|---|---|---|---|---|
| front_mask | ndarray (H, W) | binary | yes | values > {cfg["mask_threshold"]} are foreground |
| side_mask | ndarray (H, W) | binary | yes | same person, side view |
| height_cm | float | cm | yes | self-reported stature |
| weight_kg | float | kg | no | when supplied, the V-HW variant is used automatically |

Output: `{{measurement: {{value_cm, lo80, hi80, lo90, hi90}}}}`.

## Intended use
Estimating body measurements for made-to-measure pattern drafting, where the inputs come from a
controlled capture (step 2) and a person-segmentation model of the quality characterised in the
sensitivity section below.

## Out-of-scope use
- Any clinical, medical, ergonomic-safety, sizing-for-occupation or identity purpose.
- Populations outside the training distribution — notably high BMI (see failure regimes).
- Subjects shorter or taller than the BodyM height range
  ({INTEGRITY_INDEX['train'].subjects.height_cm.min():.0f}–{INTEGRITY_INDEX['train'].subjects.height_cm.max():.0f} cm).
- Commercial use, under the licence above.
- Using the conformal intervals as a *measurement* uncertainty: they are calibrated against
  mesh-derived ground truth and must be recalibrated on tape measurements (pipeline step 10).

## Final metrics — V-HW
### Test-A
{_metrics_block('testA')}

### Test-B
{_metrics_block('testB')}

### Conformal coverage (empirical, held-out)
{json.dumps({r['split'] + ' ' + r['variant']: {{'80%': round(r['empirical_80'], 1),
   '90%': round(r['empirical_90'], 1), 'mean width 90 (mm)': round(r['mean_width_90_mm'], 1)}}
   for r in COVERAGE_SUMMARY.to_dict('records')}, indent=2)}

## Metrics stratified by sex and BMI band
{chr(10).join(strata_lines)}

Strata with n < {cfg['min_stratum_n']} are printed for completeness and marked unreliable. They
must not be quoted as accuracy figures.

## Measurements that do not beat the non-visual baseline
B0 (ridge/gradient boosting on height + weight + sex) is the bar a vision model must clear.
These measurements did not clear it on validation for at least one variant:

{flag_md}

## Known failure regimes
- **High BMI.** The training population thins sharply above BMI 40. Measured MAE by BMI band is
  in `bmi_stratum_errors.csv` and `fig_bmi_failure_mode.png`. Above the training range the model
  extrapolates; the prediction intervals are the only safeguard.
- **Sparse populations.** {int(EDA['sex_counts'].loc['female', 'testA'])} female and
  {int(EDA['sex_counts'].loc['male', 'testA'])} male subjects in Test-A, so sex-stratified
  numbers there are indicative only.
- **Degraded capture.** The Test-A to Test-B gap in section 8 quantifies sensitivity to
  uncontrolled photography.
- **Height extrapolation.** Outside {INTEGRITY_INDEX['train'].subjects.height_cm.min():.0f}–{INTEGRITY_INDEX['train'].subjects.height_cm.max():.0f} cm.
- **Segmenter change.** See sensitivity results.

## Segmentation boundary sensitivity (requirement on pipeline step 3)
Nominal scale used for px → mm: **{MM_PER_PX_NOMINAL:.3f} mm/px**
(IQR {np.percentile(EDA['mm_per_px'], 25):.3f}–{np.percentile(EDA['mm_per_px'], 75):.3f} mm/px).
Maximum tolerable boundary error before the prediction error exceeds each threshold, for V-HW:

{sensitivity_md}

Full sweep: `sensitivity.csv`, `max_tolerable_boundary_error.csv`,
`fig_sensitivity_morph.png`, `fig_sensitivity_other.png`.

## Uncertainty guarantee and its limits
Split conformal prediction per measurement: **marginal** coverage over calibration and test
subjects jointly, conditional on exchangeability of the two samples. It is **not** conditional on
an individual subject, so a specific subject may fall outside its 90% interval.

The guarantee weakens, and should be assumed invalid, under:
1. **Segmenter shift** — the quantiles were fitted on BodyM's DeepLabv3+-derived masks; deployment
   masks come from a different model.
2. **Device / capture shift** — a different camera, resolution or distance changes the nominal
   mm/px scale, biasing every measurement.
3. **Population shift** — bodies outside the training BMI/height range.
4. **Ground-truth shift** — the targets are SMPL mesh vertex-path lengths, not tape measurements.

**These intervals must be recalibrated on tape-measured ground truth from pipeline step 10 before
they are used for plausibility gating in production.**

## Reproducibility
- Master seed: {cfg["seed"]}; split assignments in `splits_train.csv`.
- `config.yaml` holds every hyperparameter. Environment: Python {ENV['python']},
  torch {ENV['torch']}, torchvision {ENV['torchvision']}, GPU
  {ENV['gpu']['name'] or 'none'}.
- Full environment log, decision log and references: `DECISIONS.md`, `references.md`,
  `requirements.txt`.

## Open items for the user
1. Whether the capture protocol (step 2) collects weight — this determines the default variant.
2. Pattern-engine key names in `schema_map.yaml` (currently `TODO(user)`).
3. Which tolerance threshold the pattern engine accepts per measurement.
4. The plausibility envelope in `infer.py` / `conformal.json` (`TODO(user)`).
5. Commercial licensing clearance, given the registry inconsistency above.
"""
(EXPORT_DIR / "model_card.md").write_text(MODEL_CARD, encoding="utf-8")
print("written:", EXPORT_DIR / "model_card.md", f"({len(MODEL_CARD)} chars)")

**WHAT** — Write `DECISIONS.md`: 14 dated entries, each with the decision, the alternatives
rejected, evidence tagged `[Ref]` / `[Exp]` / `[Assumption]`, the consequences, and the open
questions.

**WHY**
- `[Ref]` Brief §11 and §14. Three of these entries exist specifically to record decisions the
  notebook deliberately refuses to make on the user's behalf: the default variant (D8), the
  pattern-engine keys and tolerances (D9), and the licence reading (D11).
- `[Exp]` D13 records a discrepancy between the brief and the data — the masks are 960x720, not the
  640x480 stated — so a future reader comparing this notebook against the brief finds the
  explanation rather than the deviation.
- `[Exp]` The numbers inside the entries are interpolated from the objects computed in this run, so
  a re-run with different hyperparameters produces a log describing *that* run.

**OUTPUT** — `export/DECISIONS.md`.

In [ ]:
DECISIONS = f"""# DECISIONS.md — dated decision log

Every entry: decision, alternatives considered, evidence, consequences, open questions.
Evidence tags: **[Ref]** published reference · **[Exp]** experiment cell in this notebook ·
**[Assumption]** stated, unverified assumption.

---

## D1 — {time.strftime('%Y-%m-%d')} — Split by subject, not by photo

**Decision.** All three splits (fit/val/calib) are assigned by `subject_id`, stratified by
sex x BMI band, drawn from the BodyM `train` split only.

**Alternatives.** (a) photo-level random split; (b) subject-level split without stratification.

**Evidence.** **[Exp]** Section 2 measured up to 27 silhouettes per subject in train. **[Exp]** The
stratification table in section 3 shows every non-trivial stratum represented in all three splits.

**Consequences.** Metrics are per subject, with photos averaged, so heavily photographed subjects
do not dominate. Validation contains fewer distinct capture conditions per subject than the raw
photo count suggests, which slightly under-samples Test-A-style variation — the reason Test-A is
reported separately.

**Open questions.** None.

---

## D2 — {time.strftime('%Y-%m-%d')} — Test-A and Test-B evaluated exactly once

**Decision.** Both test splits are touched only in section 8. No selection, hyperparameter,
augmentation magnitude, calibration or threshold uses them. A protocol audit asserting zero
subject-id and zero photo-id overlap with the training splits runs immediately before the
evaluation.

**Alternatives.** (a) use Test-A for checkpoint selection; (b) k-fold CV on train and ignore both.

**Evidence.** **[Ref]** Brief section 1. **[Exp]** The printed audit in section 8 shows overlap of
zero for both id types on both splits.

**Consequences.** Model selection rests on {len(ids_val)} validation subjects, so selection noise is
larger than it would be with a larger selection set. Reported test numbers are unbiased.

**Open questions.** The single subject id shared between Test-A and Test-B (reported by
`verify_integrity`) means those two splits are not fully independent of each other. It does not
affect either one's independence from training. Whether it is a genuine duplicate capture or a
subject-id collision in the source registry is unknown.

---

## D3 — {time.strftime('%Y-%m-%d')} — B0 is a mandatory bar, and failing it is reported

**Decision.** B0 (height + weight + sex, ridge and gradient boosting) is computed first and every
vision model is compared against it per measurement. Measurements where a variant fails to beat B0
are flagged in the model card, not dropped.

**Alternatives.** (a) compare only aggregate MAE; (b) report only the best model per measurement.

**Evidence.** **[Ref]** Brief section 5. **[Exp]** The `validation_mae_comparison.csv` table and
the flag table in section 6.

**Consequences.** Some measurements may be better served by the non-visual formula. This is stated
rather than hidden, and pipeline step 7 can act on it.

**Open questions.** Whether step 7 should switch measurement-by-measurement between the vision
model and B0, or keep one model for all 14.

---

## D4 — {time.strftime('%Y-%m-%d')} — Two height definitions ship with BodyM; `height_cm` is the model input

**Decision.** `hwg_metadata.csv:height_cm` (self-reported stature) is the scalar input and the
basis for the mm/px scale. `measurements.csv:height` (mesh vertex-path stature) is retained as a
14th target when `include_height_as_target` is true.

**Alternatives.** (a) use `measurements.height` as the input; (b) average the two.

**Evidence.** **[Exp]** `verify_integrity` quantifies the disagreement per split. The two are
strongly but not perfectly correlated, and they disagree by roughly the amount a mesh-to-tape
difference can explain.

**Consequences.** The exported model requires a self-reported stature, which the capture protocol
(step 2) must collect. The predicted `height` output is close to an identity function and should
not be treated as an independent measurement.

**Open questions.** Whether the mesh-derived `height` target adds value, or should be dropped via
`include_height_as_target: false`.

---

## D5 — {time.strftime('%Y-%m-%d')} — Scalars injected by zero-initialised projection, not by a re-initialised stem

**Decision.** The ImageNet-pretrained first convolution keeps its 3 mask channels untouched. Height
(and weight) enter through a zero-initialised 1x1 projection summed at the second convolution, so
the network is bit-exact ImageNet at initialisation.

**Alternatives.** (a) re-initialise the stem for a 5-channel input; (b) duplicate the mask into
three channels; (c) feed the scalars as spatial channels.

**Evidence.** **[Exp]** The section 4.3 cell asserts that changing only the scalar channels leaves
the output unchanged at initialisation, which is a direct test of the zero-init property.

**Consequences.** Every pretrained output filter survives in the stem with full spatial structure;
only the input-channel axis is collapsed. The scalars must earn their influence, so early training
under-uses height and weight. Padding the mask into three channels (b) was rejected because the
network would see the same input three times and waste a third of the first layer's capacity on a
constant signal.

**Open questions.** None.

---

## D6 — {time.strftime('%Y-%m-%d')} — V-H removes the weight channel rather than zero-filling it

**Decision.** V-H has 4 input channels; the weight channel is absent. `infer.py` selects the
variant by whether `weight_kg` is supplied, and raises if the pairing is inconsistent.

**Alternatives.** (a) zero-fill the weight channel; (b) fill with the training mean.

**Evidence.** **[Ref]** Brief section 7. **[Ref]** The input ablation in Ruiz et al. (2022)
motivates exporting both variants. **[Exp]** The channel-count assertion in section 6 and the
end-to-end variant-selection test in section 9.

**Consequences.** Zero-filling would tell the network "weight = 0", an out-of-distribution value it
could learn to treat as a flag. Removal is the honest encoding of "not measured".

**Open questions.** Which variant is the pipeline default — see D8.

---

## D7 — {time.strftime('%Y-%m-%d')} — Augmentation magnitudes come from the Phase-4 sweep

**Decision.** Erosion/dilation and jitter magnitudes are the largest perturbation sizes whose
ΔMAE across all measurements stayed inside the loosest reporting threshold. Downsample augmentation
uses the smallest swept factor only.

**Alternatives.** (a) the magnitudes used by Ruiz et al. (2022); (b) arbitrary values; (c) a broad
range to maximise robustness.

**Evidence.** **[Exp]** `AUG_MAG` is derived programmatically from `sensitivity.csv` in section 5,
not chosen by hand. Rationale for the augmentation families themselves: **[Ref]** brief section 7
(deployment masks come from a different segmenter than BodyM's DeepLabv3+-derived masks).

**Consequences.** The production model is robust to exactly the boundary errors the sweep found
tolerable, and no more. Training on larger perturbations would trade accuracy for robustness to
errors that cannot occur.

**Open questions.** Whether the tolerable-error thresholds themselves should be tightened once the
pattern engine states its per-measurement tolerances (D9).

---

## D8 — {time.strftime('%Y-%m-%d')} — Default variant left open for the user

**Decision.** Both variants are exported and `infer.py` selects by weight availability. No default
is hard-coded.

**Alternatives.** (a) make V-HW the default and require weight collection; (b) make V-H the
default so weight is optional.

**Evidence.** **[Exp]** `weight_gain_mm` in `validation_mae_comparison.csv` quantifies the cost of
dropping the weight channel, per measurement.

**Consequences.** The pipeline works either way, but the accuracy cost of not collecting weight is
now a number rather than an assumption.

**Open questions.** **TODO(user)** Does the capture protocol (step 2) collect weight? This is an
open decision for the user and is deliberately not resolved here.

---

## D9 — {time.strftime('%Y-%m-%d')} — Pattern-engine keys and tolerances left open for the user

**Decision.** `schema_map.yaml` ships `TODO(user)` pattern keys and the reporting thresholds
{cfg["tolerance_mm"]} mm are labelled as reporting-only. The plausibility envelope in `infer.py`
and `conformal.json` is a placeholder marked `TODO(user)`.

**Alternatives.** (a) invent plausible key names; (b) omit the file and let step 7 discover the
names at runtime.

**Evidence.** **[Ref]** Brief sections 11 and 14.

**Consequences.** Step 7 cannot run unattended until the keys are filled in. Inventing names would
produce a bundle that looks complete and fails at integration.

**Open questions.** **TODO(user)** (1) pattern-engine key names; (2) which tolerance the pattern
engine accepts per measurement; (3) the plausibility envelope.

---

## D10 — {time.strftime('%Y-%m-%d')} — Conformal intervals are marginal, conditional on exchangeability, and need recalibration

**Decision.** Split conformal quantiles per measurement per variant, finite-sample corrected, with
the coverage guarantee and all its breaking conditions written into `conformal.json`,
`model_card.md` and `DECISIONS.md`.

**Alternatives.** (a) Gaussian residual intervals; (b) quantile regression; (c) conformal with a
per-subject (Mondrian) partition.

**Evidence.** **[Ref]** Angelopoulos & Bates (2021). **[Exp]** The hand-checked quantile
self-test in section 7 pins the finite-sample correction. **[Exp]** Empirical coverage on the
held-out splits is reported in section 8 rather than assumed.

**Consequences.** Coverage is marginal over subjects, not per-subject, and it weakens under
segmenter, device and population shift. A Mondrian partition by BMI band would improve conditional
coverage but shrinks each calibration set below a size where the quantile is reliable.

**Open questions.** Recalibration on tape-measured ground truth is deferred to pipeline step 10 and
is a hard prerequisite for using these intervals in production gating.

---

## D11 — {time.strftime('%Y-%m-%d')} — Licence treated as non-commercial despite the registry inconsistency

**Decision.** BodyM is handled as **CC BY-NC 4.0, non-commercial**. The AWS registry entry links to
the CC BY legal code; the discrepancy is recorded rather than resolved in favour of the more
permissive reading.

**Alternatives.** (a) treat as CC BY (commercial permitted) because the registry says so;
(b) refuse to use the data.

**Evidence.** **[Ref]** brief section 1. **[Exp]** `verify_integrity` prints the inconsistency as a
warning so it is visible in every run.

**Consequences.** Commercial deployment of a model trained on this data needs written clarification
from the rights holders. Note that model weights trained on NC data may inherit the restriction;
that is a legal question, not a modelling one.

**Open questions.** **TODO(user)** Whether to seek commercial licensing, or to retrain on a
commercially-licensed dataset.

---

## D12 — {time.strftime('%Y-%m-%d')} — B1 slice heights are proportional heuristics, and their residual is reported

**Decision.** B1 uses a documented table of anatomical slice heights as a fraction of stature
(Drillis & Contini style), not BodyM ground truth. The per-measurement linear calibration is fitted
on the fit split only, and B1's MAE is reported as a diagnostic.

**Alternatives.** (a) learn slice heights; (b) measure them from the SMPL fit; (c) drop B1.

**Evidence.** **[Exp]** B1's per-measurement calibration residuals show where a heuristic slice is
misplaced. **[Exp]** The coefficient probe shows which geometry each prediction leans on.

**Consequences.** B1 is a diagnostic for the geometric content of silhouettes, not a competitor.
If `mm_per_px` appears among the top coefficients for several measurements, the calibration is
partly learning a stature correction and B1 is converging toward B0.

**Open questions.** Whether B1 earns its place in the exported bundle at all, given B2 dominates it
on girths.

---

## D13 — {time.strftime('%Y-%m-%d')} — Images are 960x720, not the 640x480 stated in the brief

**Decision.** Preprocessing resizes each view to {cfg["img_height"]}x{cfg["img_width"]} and
concatenates horizontally. The source masks are {list(EDA['image_stats'][["height_px", "width_px"]].drop_duplicates().itertuples(index=False, name=None))}
(H x W).

**Alternatives.** (a) train at native 960x720; (b) follow the brief's 640x480 exactly.

**Evidence.** **[Exp]** Section 2 measured the mask geometry directly; the brief's figure does not
match the data.

**Consequences.** A {list(EDA['image_stats'][["height_px", "width_px"]].drop_duplicates().itertuples(index=False, name=None))[0][1]}-pixel
width with a roughly 1.5x downscale is a mild information loss in the boundary, which is small
relative to the mask noise quantified in section 5. The brief's 640x480 would have required
upscaling.

**Open questions.** Whether native-resolution training improves accuracy enough to justify roughly
4x the compute.

---

## D14 — {time.strftime('%Y-%m-%d')} — Segmentation model is out of scope for this notebook

**Decision.** No segmentation model is trained or fine-tuned. Ultralytics/YOLO is not a
dependency.

**Alternatives.** (a) train a segmenter on BodyM; (b) accept RGB input.

**Evidence.** **[Ref]** Brief section 1; BodyM has no RGB images, so there is nothing to train a
segmenter on here.

**Consequences.** The model depends entirely on step 3's mask quality, which is why section 5
quantifies the requirement rather than assuming it.

**Open questions.** Which segmenter step 3 will use, and whether its boundary error falls inside the
tolerances in `max_tolerable_boundary_error.csv`.
"""
(EXPORT_DIR / "DECISIONS.md").write_text(DECISIONS, encoding="utf-8")
print("written:", EXPORT_DIR / "DECISIONS.md")

**WHAT** — Write `references.md` (full APA list) and `README.md` (how a downstream step loads and
calls `infer.py`, with a minimal working example and a step-7 call sequence).

**WHY**
- `[Ref]` Brief §11 requires both files; the reference list must include every citation actually
  used, including the methods cited in code comments (Bland–Altman, Drillis & Contini, Ramanujan).
- `[Exp]` The README's step-7 sequence is where the honest limitation is placed in front of the
  consumer: recalibrate on tape-measured ground truth before using the intervals for gating. A
  limitation buried in a model card is a limitation that does not change behaviour.
- `[Exp]` The licence section repeats the non-commercial position in the file most likely to be
  read by someone deciding whether to deploy this.

**OUTPUT** — `export/references.md`, `export/README.md`.

In [ ]:
REFERENCES = """# references.md

Angelopoulos, A. N., & Bates, S. (2021). A gentle introduction to conformal prediction and
  distribution-free uncertainty quantification (arXiv:2107.07511).
  https://arxiv.org/abs/2107.07511

Bland, J. M., & Altman, D. G. (1986). Statistical methods for assessing agreement between two
  methods of clinical measurement. The Lancet, 1(8476), 307-310.
  https://doi.org/10.1016/S0140-6736(86)90837-8

Drillis, J. G., & Contini, R. (1966). Body segment parameters. Human Factors, 8(4), 277-279.
  https://doi.org/10.1518/001872066X362567

Pimentel, J. F., Murta, L., Braganholo, V., & Freire, J. (2021). Understanding and improving the
  quality and reproducibility of Jupyter notebooks. Empirical Software Engineering, 26, 65.
  https://doi.org/10.1007/s10664-021-09961-9

Quaranta, L., Calefato, F., & Lanubile, F. (2022). Pynblint: A static analyzer for Python Jupyter
  notebooks. In Proceedings of CAIN '22.
  https://arxiv.org/abs/2205.11934

Ramanujan, S. (1914). A formula for the perimeter of an ellipse when the eccentricity is small.
  Indian Journal of Pure and Applied Mathematics, 5(3), 174-176.

Ruiz, N., Bellver, M., Bolkart, T., Arora, A., Lin, M. C., Romero, J., & Bala, R. (2022). Human body
  measurement estimation with adversarial augmentation (arXiv:2210.05667).
  https://arxiv.org/abs/2210.05667

## BodyM dataset

BodyM is introduced and released with Ruiz et al. (2022) above, under CC BY-NC 4.0. The AWS
registry entry for the dataset links to the CC BY legal code, which is inconsistent with the
CC BY-NC terms; the data is treated as non-commercial. See DECISIONS.md D11.

## Background methods referenced in code comments

- HistGradientBoostingRegressor — scikit-learn documentation.
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
- torch.onnx.export — PyTorch documentation.
  https://pytorch.org/docs/stable/onnx.html
- quantile(method='higher') — NumPy documentation.
  https://numpy.org/doc/stable/reference/generated/numpy.quantile.html
- AdaptiveAvgPool2d — PyTorch documentation.
  https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html
"""
(EXPORT_DIR / "references.md").write_text(REFERENCES, encoding="utf-8")
print("written:", EXPORT_DIR / "references.md")

README = f"""# README — BodyM measurement model bundle

Estimates {len(TARGETS)} body measurements in cm from two silhouette masks and stature, with
split-conformal prediction intervals. Built for step 6 of a 10-step made-to-measure pipeline.

## Contents

| file | purpose |
|---|---|
| `infer.py` | entry point: `predict_measurements(...)` |
| `preprocess.py` | canonical preprocessing, re-exported from the training code |
| `model_vhw.pt`, `model_vhw.onnx` | V-HW variant (height + weight) |
| `model_vh.pt`, `model_vh.onnx` | V-H variant (height only) |
| `conformal.json` | per-measurement, per-variant conformal quantiles + assumptions |
| `schema_map.yaml` | BodyM names -> pattern-engine keys (`TODO(user)` for the keys) |
| `model_card.md` | training data, licence, metrics, failure regimes |
| `DECISIONS.md` | dated decision log |
| `references.md` | APA reference list |
| `requirements.txt`, `config.yaml` | pinned environment and hyperparameters |
| `splits_train.csv`, `sensitivity.csv` | subject split assignment, boundary sensitivity sweep |
| `max_tolerable_boundary_error.csv` | segmentation boundary error budget |
| `reference_tensors.npz` | tensors for the preprocessing unit test |

## Minimal working example

```python
import numpy as np
from PIL import Image

from infer import predict_measurements

front = np.array(Image.open("front_mask.png"))   # binary silhouette, front view
side = np.array(Image.open("side_mask.png"))     # binary silhouette, side view

result = predict_measurements(front, side, height_cm=172.0, weight_kg=74.0)

for meas, v in result.items():
    print(f"{{meas:22s}} {{v['value_cm']:7.2f}} cm   "
          f"80% [{{v['lo80']*10:6.1f}}, {{v['hi80']*10:6.1f}}] mm   "
          f"90% [{{v['lo90']*10:6.1f}}, {{v['hi90']*10:6.1f}}] mm")
```

Calling `predict_measurements(front, side, height_cm=172.0)` without a weight selects the V-H
variant automatically.

## Inputs

| field | type | unit | required |
|---|---|---|---|
| front_mask | ndarray (H, W) | binary, values > {cfg["mask_threshold"]} = foreground | yes |
| side_mask | ndarray (H, W) | binary | yes |
| height_cm | float | cm | yes |
| weight_kg | float | kg | no |

## Output

```python
{{"chest": {{"value_cm": 98.4, "lo80": 96.1, "hi80": 100.7,
             "lo90": 95.0, "hi90": 101.8}}, ...}}
```

## How a downstream step (step 7) should call this

1. Produce the two masks from pipeline step 3, at the quality characterised in
   `max_tolerable_boundary_error.csv`.
2. Calibrate scale from pipeline step 4.
3. Call `predict_measurements`. Pass `weight_kg` only if the capture protocol collected it.
4. Read `schema_map.yaml` to translate measurement names to pattern-engine keys. **Fill in the
   `TODO(user)` keys first.**
5. Use the 90% interval as the plausibility gate. Reject the pattern request, or route to manual
   measurement, when the interval is wider than the pattern engine's tolerance for that key.

**Before production use:** recalibrate the intervals on tape-measured ground truth (pipeline step
10). The shipped quantiles are calibrated against mesh-derived ground truth from a different
segmenter, so they understate real uncertainty. See `model_card.md`.

## Runtime

- Loading both variants: ~1-2 s.
- Inference: <50 ms per subject on GPU, ~1 s on CPU.
- Memory: ~150 MB per variant resident.

## Licence

Training data: BodyM, CC BY-NC 4.0 — **non-commercial**. The AWS registry entry links to CC BY;
see `DECISIONS.md` D11. Commercial deployment requires written clarification.
"""
(EXPORT_DIR / "README.md").write_text(README, encoding="utf-8")
print("written:", EXPORT_DIR / "README.md")

**WHAT** — Pin every dependency version from the running environment, copy the split CSVs, the
sensitivity results, the metric tables and every figure into the bundle, and copy the `src/`
modules so the provenance record travels with the weights.

**WHY**
- `[Ref]` Brief §11 lists config, split CSVs and `sensitivity.csv` as bundle contents.
- `[Exp]` Pinning from the *live* environment rather than from a hand-maintained file means
  `requirements.txt` describes the run that actually happened. A hand-written pin drifts the
  first time a dependency is upgraded.
- `[Exp]` Shipping `src/` is what makes the model card's claims auditable: a reader can check that
  the preprocessing the export uses is the same code that trained the model.

**OUTPUT** — `requirements.txt`, `config.yaml`, all CSVs, all figures, `src/*.py` in the bundle.

In [ ]:
# --- requirements (pinned), config, split CSVs, sensitivity ----------------------------------
def pinned(module_name: str) -> str:
    """Return a pinned requirement line for an installed module.

    Parameters
    ----------
    module_name : str
        Import name, e.g. ``'numpy'``.

    Returns
    -------
    str
        ``name==version``, or ``name`` if the version cannot be determined.
    """
    import importlib.metadata as md
    try:
        return f"{module_name}=={md.version(module_name)}"
    except Exception:
        return module_name


PINNED = [pinned(m) for m in ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib",
                              "pillow", "pyyaml", "torch", "torchvision", "tqdm",
                              "onnx", "onnxruntime"]]
(EXPORT_DIR / "requirements.txt").write_text("\n".join(PINNED) + "\n", encoding="utf-8")
print("written: requirements.txt (pinned)")

shutil.copy2(CFG_PATH, EXPORT_DIR / "config.yaml")
shutil.copy2(SENSITIVITY_CSV, EXPORT_DIR / "sensitivity.csv")
shutil.copy2(ART / "max_tolerable_boundary_error.csv", EXPORT_DIR / "max_tolerable_boundary_error.csv")
shutil.copy2(ART / "validation_mae_comparison.csv", EXPORT_DIR / "validation_mae_comparison.csv")
shutil.copy2(ART / "flagged_measurements.csv", EXPORT_DIR / "flagged_measurements.csv")
shutil.copy2(ART / "final_comparison.csv", EXPORT_DIR / "final_comparison.csv")
shutil.copy2(ART / "coverage_summary.csv", EXPORT_DIR / "coverage_summary.csv")
shutil.copy2(ART / "bmi_stratum_errors.csv", EXPORT_DIR / "bmi_stratum_errors.csv")
shutil.copy2(ART / "augmentation_magnitudes.yaml", EXPORT_DIR / "augmentation_magnitudes.yaml")
for name in ("splits_train.csv",):
    shutil.copy2(ART / name, EXPORT_DIR / name)
for name in ("fig_eda_population.png", "fig_eda_measurements.png", "fig_eda_correlation.png",
             "fig_eda_imagery.png", "fig_eda_artefacts_worst.png", "fig_eda_artefacts_typical.png",
             "fig_b2_training.png", "fig_variants_validation.png", "fig_sensitivity_morph.png",
             "fig_sensitivity_other.png", "fig_testA_vs_testB.png", "fig_bmi_failure_mode.png"):
    if (ART / name).is_file():
        shutil.copy2(ART / name, EXPORT_DIR / name)
shutil.copy2(CFG_PATH, EXPORT_DIR / "config.yaml")
# the source modules are part of the provenance record
for name in ("bodym_data.py", "bodym_metrics.py", "bodym_baselines.py", "bodym_cnn.py",
             "bodym_training.py", "bodym_conformal.py", "bodym_perturb.py",
             "bodym_sensitivity.py", "bodym_eda.py"):
    shutil.copy2(PATHS["src"] / name, EXPORT_DIR / name)
print("copied: config, CSVs, figures, source modules")
print("mark_phase('docs export') =", mark_phase("docs export"), "s")

**WHAT** — Assert every artefact required by brief §11 exists, write `MANIFEST.json` with file
names and sizes, and zip the bundle to `/kaggle/outputs`.

**WHY**
- `[Ref]` Brief §11's contents list, turned into an executable check. A missing file should fail
  the run here rather than be discovered by whoever downloads the bundle.
- `[Exp]` `MANIFEST.json` makes an incomplete or truncated download detectable, which matters
  because the weights are the one artefact a partial zip silently breaks.

**OUTPUT** — `MANIFEST.json`, `export.zip`.

**CHECK** — `assert not missing` over the full brief §11 list.

In [ ]:
# --- verify every brief section 11 item exists --------------------------------------------
REQUIRED_EXPORTS = [
    "model_vhw.onnx", "model_vh.onnx", "model_vhw.pt", "model_vh.pt",
    "preprocess.py", "infer.py", "conformal.json", "schema_map.yaml",
    "model_card.md", "DECISIONS.md", "README.md", "references.md",
    "requirements.txt", "config.yaml", "splits_train.csv", "sensitivity.csv",
]
missing = [f for f in REQUIRED_EXPORTS if not (EXPORT_DIR / f).is_file()]
assert not missing, f"missing required exports: {missing}"
print(f"all {len(REQUIRED_EXPORTS)} required brief section 11 artefacts present")

EXPORT_MANIFEST = {
    "files": sorted(p.name for p in EXPORT_DIR.iterdir() if p.is_file()),
    "counts": {},
}
for p in sorted(EXPORT_DIR.iterdir()):
    if p.is_file():
        EXPORT_MANIFEST["counts"][p.name] = p.stat().st_size
(EXPORT_DIR / "MANIFEST.json").write_text(json.dumps(EXPORT_MANIFEST, indent=2))

if cfg["zip_export"]:
    zip_path = shutil.make_archive(str(PATHS["working"] / "export"), "zip", root_dir=EXPORT_DIR)
    print("zipped:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")
else:
    print("zip_export disabled in config")

print(f"\nexport directory: {EXPORT_DIR}")
print(f"{len(list(EXPORT_DIR.iterdir()))} files")
print("mark_phase('export') =", mark_phase("export"), "s")

**Interpretation — what a reviewer should check in this bundle.**

Three checks are already automated and passing, and they are the ones that historically break in
production:

1. **ONNX ↔ PyTorch agreement within 1e-4 on 20 real validation samples.** Not random tensors —
  real masks. A transposed channel order or a dropped scalar injection still "runs" on random
  input, so random input would have passed a broken export.
2. **`preprocess.py` self-test against training reference tensors.** Shape and channel contract
   verified from a subprocess, so nothing can accidentally depend on a notebook global.
3. **`infer.py` executed in a subprocess** with only the export directory importable, on a real
   Test-A subject, returning all 14 measurements with intervals for both variants.

One thing deliberately left incomplete: the `TODO(user)` markers in `schema_map.yaml`, the
plausibility envelope, and the per-measurement tolerances. Those are pipeline-design decisions, not
modelling ones, and §10 lists them as open. Shipping invented pattern-engine key names would
produce a bundle that passes every automated check and fails at integration.

Finally, the licence position: the data is CC BY-NC 4.0, the AWS registry links to CC BY, and this
notebook treats it as non-commercial. Weights trained on NC data may inherit that restriction —
a legal question for the user, not one this notebook can settle.

### §10 Licence status, open decisions, and what this notebook does not claim

**WHAT** — Consolidate the licence position, the required "must appear" explanations, and the
open decisions for the user into one place.

**WHY**
- `[Ref]` Brief §12 lists explanations that *must* appear. They are written into the cells that
  own them and summarised here so a reviewer can verify completeness in one read.
- `[Ref]` Brief §1: *"Never claim ISO 20685 compliance. Report errors against tolerance thresholds
  only."* Stated here as an explicit disclaimer, not just in the model card.
- `[Exp]` The registry inconsistency (CC BY-NC dataset, registry linking CC BY) is a licensing risk,
  not a modelling detail. It is recorded in three places so it cannot be missed.

#### §10.1 Licence

| item | status |
|---|---|
| Dataset | BodyM (Ruiz et al., 2022) |
| Stated licence | **CC BY-NC 4.0** |
| Registry entry | AWS registry links to the **CC BY** legal code |
| Inconsistency | **Yes — recorded, not resolved** |
| Treatment in this work | **Non-commercial.** No commercial use is claimed or implied. |

The two statements cannot both be right, and the more restrictive reading is the safe one. This is
recorded rather than resolved because resolving it is a legal question, not a modelling one. It has
a practical consequence: **model weights trained on NC-licensed data may inherit the
non-commercial restriction.** Any commercial deployment needs written clarification from the rights
holders, or a retrain on a commercially-licensed dataset.

#### §10.2 Required explanations — completeness check

Each row names the section that carries the explanation, so this table is a traceability index
rather than a restatement.

| required explanation | section | substance |
|---|---|---|
| Subject-level splitting (multiple silhouettes → leakage) | §2, §3 | up to 27 silhouettes per subject; a photo-level split puts the same fitted mesh in two splits and inflates every metric |
| Single use of Test-A/Test-B | §2, §8 | evaluated exactly once; subject-id **and** photo-id overlap audits printed immediately before |
| Purpose of B0 | §4.2 | height+weight+sex only; the bar any vision model must clear per measurement |
| Choice of augmentation magnitudes | §5, §6 | derived from the §5 sweep, not invented: the largest perturbation whose ΔMAE stayed inside tolerance |
| Why two variants are exported | §6 | height-only vs height+weight trade-off; the input ablation in Ruiz et al. (2022); weight channel removed, not zero-filled |
| What conformal intervals guarantee, and when that breaks | §7, §9 | marginal coverage conditional on exchangeability; breaks under segmenter, device, population and ground-truth shift |
| Mesh-defined vs tape-measured ground truth | §2, §9, §10 | targets are SMPL vertex-path lengths; step 10's tape measurements are the first real test and must recalibrate the intervals |
| Licence status and registry inconsistency | §10.1 | CC BY-NC vs registry CC BY; treated as non-commercial |

#### §10.3 Known limitation: ground truth is mesh-derived, not tape-measured

Every number in this notebook is measured against **SMPL-registered mesh vertex-path lengths**.
Those are not ISO 20685 tape measurements, and the difference is systematic rather than random:

- A mesh vertex path follows the body surface. A tape follows the same surface but compresses
  soft tissue, sits at a defined anatomical landmark, and is tensioned in a defined way.
- Mesh fits are usually driven by a subset of the silhouette. Measurements on poorly-constrained
  regions (deep chest, waist under clothing) inherit that error.

Consequences that follow directly:

1. **The Bland–Altman bias columns in §8 are the signal to watch.** A coherent offset between
   mesh and tape is a definition difference, not a model defect, and it will not shrink with more
   training data.
2. **The 9/15/25 mm thresholds are reporting thresholds**, chosen by the brief. They are not
   derived from any pattern engine's requirement — see §10.4.
3. **No ISO 20685 compliance is claimed or implied anywhere in this notebook or its export bundle.**
4. **The conformal intervals must be recalibrated on tape-measured ground truth** (pipeline step
   10) before they gate anything in production.

#### §10.4 Open decisions for the user — deliberately not resolved

| # | open question | where it bites | current placeholder |
|---|---|---|---|
| 1 | Does the capture protocol (step 2) collect weight? Determines the default variant | `infer.py` variant selection; accuracy cost quantified in `validation_mae_comparison.csv:weight_gain_mm` | auto-selects by weight availability, no hard-coded default |
| 2 | Pattern-engine key names | `schema_map.yaml` — step 7 cannot run unattended | `TODO(user)` per measurement |
| 3 | Per-measurement tolerance thresholds the pattern engine accepts | gating; the 9/15/25 mm here are reporting-only | `tolerance_owner: TODO(user)` in `schema_map.yaml` |
| 4 | Plausibility envelope for interval clamping | `infer.py:PLAUSIBLE_CM`, mirrored in `conformal.json` | wide defensible bounds, marked `TODO(user)` |
| 5 | Commercial licensing clearance | any commercial deployment | non-commercial assumed; D11 in `DECISIONS.md` |
| 6 | Which segmenter step 3 will use | whether mask quality falls inside the §5 budget | tolerance table published, segmenter unchosen |
| 7 | Whether to keep the mesh-derived `height` as a 14th target | `config.yaml: include_height_as_target` | `true` |
| 8 | Whether B1 belongs in the shipped bundle | bundle size, conceptual clarity | not exported; retained as a diagnostic |

**These are recorded, not silently resolved.** Each is listed in `DECISIONS.md` with the same
numbering.

### §11 Final cell — reproducibility summary

**WHAT** — Print the complete provenance record: environment, seeds, data file counts, GPU,
wall-clock time per phase, and every exported file path. Then run the notebook linter if it is
available, treating the result as advisory.

**WHY**
- `[Ref]` Brief §13 requires exactly this. A run whose provenance cannot be reconstructed from its
  own output is not reproducible regardless of what the metrics say.
- `[Ref]` Brief §12 optionally asks for `pynblint` (Quaranta et al., 2022) with the result recorded
  and treated as advisory. It is attempted, never required — a linter failure must not fail the run.

In [ ]:
# --- optional advisory linter ---------------------------------------------------------------
LINT_REPORT = "pynblint not installed; linter step skipped (advisory only)"
try:
    import importlib.util as _iu
    if _iu.find_spec("pynblint") is None:
        _sp.run([sys.executable, "-m", "pip", "install", "-q", "pynblint"], check=False,
                capture_output=True)
    if _iu.find_spec("pynblint") is not None:
        _lr = _sp.run([sys.executable, "-m", "pynblint", "--fail-level", "warning",
                       str(PATHS["working"] / "model_training_measurement.ipynb")],
                      capture_output=True, text=True, capture_output=True)
        LINT_REPORT = _lr.stdout.strip()[-4000:] or _lr.stderr.strip()[-2000:] or "no findings"
except Exception as _exc:
    LINT_REPORT = f"pynblint unavailable: {_exc} (advisory only)"
print("=== advisory linter (pynblint, Quaranta et al. 2022) ===")
print(LINT_REPORT)

**WHAT** — Print the full provenance record: environment and versions, seeds, data file counts and
integrity status, split sizes, per-model validation MAE, the boundary-sensitivity headline, the
conformal half-widths and held-out coverage, the flag table, every exported file with its size, the
open decisions, and the claims explicitly **not** made.

**WHY**
- `[Ref]` Brief §13 requires exactly this. A run whose provenance cannot be reconstructed from its
  own output is not reproducible, whatever the metrics say.
- `[Exp]` The closing block is not boilerplate. Naming the claims *not* made — ISO 20685 compliance,
  per-subject conditional coverage, performance outside the training distribution, commercial use —
  is what stops the export being over-claimed downstream by someone who reads only the summary.

**OUTPUT** — console provenance record (the notebook's final deliverable in its own right).

**CHECK** — none; this is the reporting step. Every number it prints is read from the objects
computed in earlier cells, so nothing here can be hard-coded out of sync.

In [ ]:
# =========================================================================================
# REPRODUCIBILITY SUMMARY
# =========================================================================================
PHASE_TIMES.pop("_last", None)
total = time.time() - t_notebook_start

print("=" * 78)
print("REPRODUCIBILITY SUMMARY — BodyM measurement model")
print("=" * 78)

print(f"\nwall clock total        : {total/60:.1f} min ({total:.0f} s)")
print("per-phase seconds:")
for k, v in PHASE_TIMES.items():
    if k != "_last":
        print(f"    {k:28s} {v:9.1f} s   ({100*v/max(total,1):5.1f} %)")

print("\nENVIRONMENT")
for k in ("python", "platform", "numpy", "pandas", "scikit_learn", "scipy", "torch",
          "torchvision", "matplotlib", "pillow"):
    print(f"    {k:28s} {ENV[k]}")
print(f"    {'GPU':28s} {ENV['gpu']['name'] if ENV['gpu']['available'] else 'none (CPU only)'}"
      f"  capability {ENV['gpu']['capability']}, {ENV['gpu']['total_gb']} GB")
print(f"    {'internet':28s} {ENV['internet']}")
print(f"    {'kaggle session':28s} {ENV['kaggle_env']}")
print(f"    {'backbone weights':28s} {net_vhw.weights_source}")

print("\nSEEDS")
print(f"    master seed                {cfg['seed']} (python, numpy, torch, cuda, dataloader workers)")
print(f"    cudnn.deterministic        {torch.backends.cudnn.deterministic}")

print("\nDATA")
print(f"    root                       {DATA_ROOT}")
for s in SPLITS:
    idx = INTEGRITY_INDEX[s]
    print(f"    {s:26s} {len(idx.subjects):5d} subjects  {len(idx.photos):5d} silhouettes  "
          f"masks front+side = {2*len(idx.photos)} files")
print(f"    integrity checks           "
      f"{int((integrity_report.status=='ok').sum())} ok, "
      f"{int((integrity_report.status=='warn').sum())} warn, "
      f"{int((integrity_report.status=='fail').sum())} fail")

print("\nSPLITS (train only, by subject)")
print(f"    fit {len(ids_fit)} | val {len(ids_val)} | conformal calibration {len(ids_calib)}")
print(f"    Test-A {len(TEST_IDS['testA'])} and Test-B {len(TEST_IDS['testB'])} "
      f"evaluated exactly once, in section 8")

print("\nMODELS")
for name, hist in (("B0 ridge", None), ("B0 gbm", None), ("B1 geometric", None),
                   ("B2 baseline", hist_b2), ("V-HW", hist_vhw), ("V-H", hist_vh)):
    if hist is not None:
        print(f"    {name:28s} epochs {len(hist):2d}  best val MAE {hist.val_mae_mm.min():7.2f} mm "
              f"(epoch {int(hist.loc[hist.val_mae_mm.idxmin(),'epoch'])})")
    else:
        pred = {"B0 ridge": P0_val_ridge, "B0 gbm": P0_val_gbm, "B1 geometric": P1_val}[name]
        print(f"    {name:28s} val MAE {np.abs(pred - Y_val).mean()*10:7.2f} mm")

print(f"\nBOUNDARY SENSITIVITY")
print(f"    nominal scale              {MM_PER_PX_NOMINAL:.3f} mm/px")
print(f"    augmentation magnitudes    erosion/dilation ±{AUG_MAG['erosion_px']} px, "
      f"jitter sigma {AUG_MAG['jitter_sigma_px']} px, downsample x{AUG_MAG['downsample_factor']}")
_k = MAX_TOLERABLE_BOUNDARY[(MAX_TOLERABLE_BOUNDARY.model == "V-HW")
                            & (MAX_TOLERABLE_BOUNDARY.measurement.isin(cfg["key_measurements"]))
                            & (MAX_TOLERABLE_BOUNDARY.family == "morph")
                            & (MAX_TOLERABLE_BOUNDARY.tolerance_mm == cfg["tolerance_mm"][-1])]
if len(_k):
    print("    max tolerable erosion at "
          f"{cfg['tolerance_mm'][-1]:.0f} mm   "
          + ", ".join(f"{r.measurement} {r.max_tolerable_px:.0f} px "
                      f"({r.max_tolerable_mm:.1f} mm)" for r in _k.itertuples()))

print("\nCONFORMAL PREDICTION")
for v in ("V-HW", "V-H"):
    half = np.mean([CONFORMAL_QUANTILES[v][m]["90"] * 20 for m in TARGETS])
    print(f"    {v:28s} n_calib {len(ids_calib):4d}  mean 90% half-width {half:7.1f} mm  "
          f"(testA coverage {COVERAGE_SUMMARY[(COVERAGE_SUMMARY.split=='testA') & (COVERAGE_SUMMARY.variant==v)].empirical_90.iloc[0]:5.1f}%, "
          f"testB {COVERAGE_SUMMARY[(COVERAGE_SUMMARY.split=='testB') & (COVERAGE_SUMMARY.variant==v)].empirical_90.iloc[0]:5.1f}%)")

print(f"\nFLAGGED (variant does not beat B0 on validation): {len(FLAGS)} measurement(s)")
if len(FLAGS):
    for r in FLAGS.itertuples():
        print(f"    {r.measurement:22s} B0 {r.mae_B0_mm:6.2f} mm | V-HW {r.mae_VHW_mm:6.2f} | "
              f"V-H {r.mae_VH_mm:6.2f}")

print("\nEXPORTED FILES")
for p in sorted(EXPORT_DIR.iterdir()):
    if p.is_file():
        print(f"    {p.name:42s} {p.stat().st_size/1024:10.1f} KiB")
zip_p = PATHS["working"] / "export.zip"
if zip_p.is_file():
    print(f"    {'export.zip':42s} {zip_p.stat().st_size/1e6:10.1f} MB")
print(f"\n    export dir : {EXPORT_DIR}")
print(f"    figures    : {ART}")
print(f"    checkpoints: {PATHS['working'] / 'checkpoints'}")
print(f"    config     : {CFG_PATH}")

print("\nOPEN DECISIONS FOR THE USER (not resolved by this notebook)")
print("    1. does the capture protocol collect weight? (determines the default variant)")
print("    2. pattern-engine key names for schema_map.yaml  [TODO(user)]")
print("    3. per-measurement tolerance thresholds the pattern engine accepts  [TODO(user)]")
print("    4. plausibility envelope for interval clamping  [TODO(user)]")
print("    5. commercial licensing clearance (CC BY-NC vs registry CC BY)")
print("    6. which segmenter step 3 uses, and whether it fits the boundary budget")
print("    7. whether to keep mesh-derived height as a 14th target")
print("    8. whether B1 belongs in the shipped bundle")

print("\nCLAIMS EXPLICITLY NOT MADE")
print("    - ISO 20685 compliance (targets are SMPL mesh vertex-path lengths, not tape measurements)")
print("    - per-subject conditional coverage (conformal coverage is marginal)")
print("    - performance outside the BodyM height / BMI / sex distribution")
print("    - commercial use of the training data or the derived weights")
print("=" * 78)

**WHAT** — Render a single four-panel summary figure (Test-A MAE, Test-B MAE, held-out conformal
coverage versus nominal, and validation MAE across all five models) into the bundle, then re-zip.

**WHY**
- `[Exp]` One page that answers the four questions a reviewer actually asks: how accurate, how
  much worse under degraded capture, is the coverage claim honest, and did the vision models beat
  the non-visual bar. It is generated from the same DataFrames as everything else, so it cannot
  drift from the run that produced it.

**OUTPUT** — `export/fig_summary.png`, refreshed `export.zip`.

**CHECK** — none; rendering only.

In [ ]:
# --- optional: one summary figure for the bundle -------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))
_summary = FINAL_COMPARISON.pivot_table(index="measurement", columns=["split", "model"],
                                        values="mae_mm")
_summary[("testA", "V-HW")].plot(kind="barh", ax=axes[0, 0], width=0.8, legend=False)
axes[0, 0].set_title("V-HW MAE, Test-A (mm)"); axes[0, 0].axvline(25, ls="--", c="k", lw=0.8)
axes[0, 0].set_xlabel("MAE (mm)")

_summary[("testB", "V-HW")].plot(kind="barh", ax=axes[0, 1], width=0.8, legend=False)
axes[0, 1].set_title("V-HW MAE, Test-B (mm)"); axes[0, 1].axvline(25, ls="--", c="k", lw=0.8)
axes[0, 1].set_xlabel("MAE (mm)")

axes[1, 0].plot(COVERAGE_SUMMARY.split + "/" + COVERAGE_SUMMARY.variant,
                COVERAGE_SUMMARY.empirical_90, "o", label="empirical 90%")
axes[1, 0].axhline(90, ls="--", c="k", lw=0.9, label="nominal 90%")
axes[1, 0].set_ylim(60, 105); axes[1, 0].set_ylabel("coverage (%)")
axes[1, 0].set_title("Conformal coverage on held-out splits"); axes[1, 0].legend(fontsize=7)

_w = mae_tbl.set_index("measurement")[["mae_B0_mm", "mae_B1_mm", "mae_B2_mm",
                                       "mae_VHW_mm", "mae_VH_mm"]]
_w.plot(kind="barh", ax=axes[1, 1], width=0.8)
axes[1, 1].set_title("Validation MAE: all models (mm)")
axes[1, 1].set_xlabel("MAE (mm)"); axes[1, 1].legend(fontsize=7)
for ax in axes.ravel():
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("BodyM measurement model — run summary")
fig.tight_layout()
fig.savefig(EXPORT_DIR / "fig_summary.png", bbox_inches="tight")
plt.close(fig)
print("summary figure written:", EXPORT_DIR / "fig_summary.png")

if cfg["zip_export"]:
    zip_path = shutil.make_archive(str(PATHS["working"] / "export"), "zip", root_dir=EXPORT_DIR)
    print(f"re-zipped with summary figure: {zip_path} "
          f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")
    print("\nDONE. Download /kaggle/outputs/export.zip")